# پیش‌بینی مقدار اجاره‌بها با یادگیری ماشین

این نوت‌بوک شامل آماده‌سازی داده، تحلیل اکتشافی، مقایسه مدل‌ها، مهندسی ویژگی و ارزیابی مدل نهایی است.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor


In [2]:
df=pd.read_csv("Divar.csv")

df1=pd.read_csv("iran_city_classification.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Divar.csv'

## 1. شناخت اولیه داده‌ها

بررسی ساختار، آمار توصیفی و ستون‌های مورد استفاده.


In [4]:
df.describe()

df1.describe()

,نام شهر,دسته‌بندی
count,240,240
unique,240,2
top,karaj,شهر کوچک
freq,1,231


In [6]:
df1.columns

Index(['نام شهر', 'دسته‌بندی'], dtype='object')

In [7]:
selected_columns = [
    "has_elevator",
    "rooms_count",
    "has_warehouse",
    "has_security_guard",
    "has_sauna",
    "has_pool",
    "has_parking",
    "has_barbecue",
    "has_jacuzzi",
    "building_size",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "construction_year",
    "has_electricity",
    "has_heating_system",
    "has_cooling_system"
]

missing_values = df[selected_columns].isnull().sum()

missing_values

has_elevator          458251
rooms_count           154101
has_warehouse         271845
has_security_guard    968688
has_sauna             971521
has_pool              970610
has_parking           271844
has_barbecue          968802
has_jacuzzi           971272
building_size          19606
cat2_slug                  0
cat3_slug                  1
city_slug                  2
construction_year     184172
has_electricity       966555
has_heating_system    631031
has_cooling_system    649381
dtype: int64

In [8]:
missing_summary = pd.DataFrame({
    "missing_count": df[selected_columns].isnull().sum(),
    "missing_percent": (
        df[selected_columns].isnull().mean() * 100
    ).round(2)
})

missing_summary

,missing_count,missing_percent
has_elevator,458251,45.83
rooms_count,154101,15.41
has_warehouse,271845,27.18
has_security_guard,968688,96.87
has_sauna,971521,97.15
has_pool,970610,97.06
has_parking,271844,27.18
has_barbecue,968802,96.88
has_jacuzzi,971272,97.13
building_size,19606,1.96


In [9]:
cat2_counts = df["cat2_slug"].value_counts(dropna=False)
cat3_counts = df["cat3_slug"].value_counts(dropna=False)

cat2_counts

cat2_slug
residential-sell        558708
residential-rent        276558
commercial-rent          76567
commercial-sell          38861
temporary-rent           29903
real-estate-services     19403
Name: count, dtype: int64

In [10]:
target_columns = [
    "rent_value",
    "credit_value"
]

target_missing = df[target_columns].isnull().sum()

target_missing

rent_value      648678
credit_value    647905
dtype: int64

## 2. انتخاب داده‌های مربوط به اجاره

ساخت مجموعه داده مربوط به آگهی‌های اجاره‌ای.


In [11]:
rental_data = df[
    df["cat2_slug"].isin([
        "residential-rent",
        "commercial-rent"
    ])
].copy()

print("Rental data shape:")
print(rental_data.shape)

rental_target_missing = rental_data[
    ["rent_value", "credit_value"]
].isnull().sum()

) as file:
print("\nMissing values in rental targets:")
print(rental_target_missing)

Rental data shape:
(353125, 61)

Missing values in rental targets:
rent_value      1814
credit_value    1041
dtype: int64


In [12]:
target_missing_mask = rental_data[
    ["rent_value", "credit_value"]
].isnull()

print("Missing in both:")
print(target_missing_mask.all(axis=1).sum())

print("\nMissing in at least one:")
print(target_missing_mask.any(axis=1).sum())

print("\nComplete target records:")
print((~target_missing_mask.any(axis=1)).sum())

Missing in both:
911

Missing in at least one:
1944

Complete target records:
351181


In [13]:
target_stats = rental_data[
    ["rent_value", "credit_value"]
].describe()

) as file:
target_stats

,rent_value,credit_value
count,3.513110e+05,3.520840e+05
mean,4.102428e+10,4.872236e+10
std,3.807594e+12,4.341413e+12
min,0.000000e+00,0.000000e+00
25%,1.111110e+05,1.000000e+08
50%,5.000000e+06,2.500000e+08
75%,1.200000e+07,5.000000e+08
max,1.000000e+15,1.000000e+15


In [14]:
target_zero_counts = (
    rental_data[["rent_value", "credit_value"]] == 0
).sum()

target_zero_counts

rent_value      59240
credit_value     2863
dtype: int64

In [15]:
target_percentiles = rental_data[
    ["rent_value", "credit_value"]
].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

target_percentiles

,rent_value,credit_value
0.900,3.000000e+07,1.000000e+09
0.950,5.500000e+07,1.650000e+09
0.990,2.200000e+08,5.000000e+09
0.995,4.000000e+08,9.500000e+09
0.999,1.111111e+10,1.111111e+11


In [16]:
high_rent_counts = {
    "rent_gt_100m": (rental_data["rent_value"] > 100_000_000).sum(),
    "rent_gt_200m": (rental_data["rent_value"] > 200_000_000).sum(),
    "rent_gt_500m": (rental_data["rent_value"] > 500_000_000).sum(),
    "rent_gt_1b": (rental_data["rent_value"] > 1_000_000_000).sum(),
    "rent_gt_10b": (rental_data["rent_value"] > 10_000_000_000).sum(),
    "rent_gt_100b": (rental_data["rent_value"] > 100_000_000_000).sum(),
    
    "credit_gt_5b": (rental_data["credit_value"] > 5_000_000_000).sum(),
    "credit_gt_10b": (rental_data["credit_value"] > 10_000_000_000).sum(),
    "credit_gt_50b": (rental_data["credit_value"] > 50_000_000_000).sum(),
    "credit_gt_100b": (rental_data["credit_value"] > 100_000_000_000).sum(),
}

high_rent_counts

{'rent_gt_100m': np.int64(8886),
 'rent_gt_200m': np.int64(3656),
 'rent_gt_500m': np.int64(1438),
 'rent_gt_1b': np.int64(1039),
 'rent_gt_10b': np.int64(489),
 'rent_gt_100b': np.int64(278),
 'credit_gt_5b': np.int64(3212),
 'credit_gt_10b': np.int64(1446),
 'credit_gt_50b': np.int64(470),
 'credit_gt_100b': np.int64(393)}

In [17]:
large_values = rental_data.loc[
    (rental_data["rent_value"] > 1_000_000_000) |
    (rental_data["credit_value"] > 10_000_000_000),
    ["rent_value", "credit_value"]
].sort_values(
    ["rent_value", "credit_value"],
    ascending=False
)

large_values.head(30)

,rent_value,credit_value
235489,1.000000e+15,1.000000e+15
340850,1.000000e+15,1.000000e+15
463475,1.000000e+15,1.000000e+15
149254,8.888889e+14,3.000000e+14
554385,3.333333e+14,3.333333e+12
906064,3.333333e+14,3.333333e+12
538272,2.222222e+14,2.222222e+14
577251,1.444146e+14,1.158888e+06
881457,1.200000e+14,1.111111e+08
5131,1.111111e+14,1.111111e+14


In [18]:
extreme_target_mask = (
    (rental_data["rent_value"] >= 100_000_000_000) |
    (rental_data["credit_value"] >= 100_000_000_000)
)

print("Extreme records:", extreme_target_mask.sum())
print("Percentage:", extreme_target_mask.mean() * 100)

Extreme records: 573
Percentage: 0.1622654867256637


## 3. پاک‌سازی اولیه داده‌ها

بررسی و پاک‌سازی مقادیر غیرعادی و ساخت متغیرهای کمکی مورد استفاده در مسیر اولیه پروژه.


In [19]:
target_clean_mask = (
    rental_data["rent_value"].notna() &
    rental_data["credit_value"].notna() &
    (rental_data["rent_value"] < 100_000_000_000) &
    (rental_data["credit_value"] < 100_000_000_000)
)

rental_clean = rental_data.loc[target_clean_mask].copy()

print("Original records:", len(rental_data))
print("Clean records:", len(rental_clean))
print("Removed records:", len(rental_data) - len(rental_clean))

Original records: 353125
Clean records: 350608
Removed records: 2517


In [20]:
rental_clean["total_credit"] = (
    rental_clean["credit_value"]
    + rental_clean["rent_value"] * (100_000_000 / 3_000_000)
)

total_credit_stats = rental_clean["total_credit"].describe()

) as file:
total_credit_stats

count    3.506080e+05
mean     1.447512e+09
std      2.145485e+10
min      3.433333e+01
25%      2.733333e+08
50%      4.833333e+08
75%      9.500000e+08
max      3.167667e+12
Name: total_credit, dtype: float64

In [22]:
total_credit_percentiles = rental_clean["total_credit"].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999]
)

) as file:
total_credit_percentiles

0.900    1.966667e+09
0.950    3.300000e+09
0.990    1.053100e+10
0.995    1.733333e+10
0.999    7.001512e+10
Name: total_credit, dtype: float64

In [23]:
total_credit_extreme_counts = {
    "gt_2b": (rental_clean["total_credit"] > 2_000_000_000).sum(),
    "gt_5b": (rental_clean["total_credit"] > 5_000_000_000).sum(),
    "gt_10b": (rental_clean["total_credit"] > 10_000_000_000).sum(),
    "gt_50b": (rental_clean["total_credit"] > 50_000_000_000).sum(),
    "gt_100b": (rental_clean["total_credit"] > 100_000_000_000).sum(),
    "gt_500b": (rental_clean["total_credit"] > 500_000_000_000).sum(),
    "gt_1t": (rental_clean["total_credit"] > 1_000_000_000_000).sum(),
}

) as file:
total_credit_extreme_counts

{'gt_2b': np.int64(32790),
 'gt_5b': np.int64(9419),
 'gt_10b': np.int64(3789),
 'gt_50b': np.int64(458),
 'gt_100b': np.int64(303),
 'gt_500b': np.int64(48),
 'gt_1t': np.int64(34)}

In [24]:
high_total_credit = (
    rental_clean[
        rental_clean["total_credit"] > 50_000_000_000
    ][
        [
            "total_credit",
            "rent_value",
            "credit_value",
            "building_size",
            "city_slug",
            "cat3_slug",
            "rooms_count",
            "construction_year"
        ]
    ]
    .sort_values("total_credit", ascending=False)
    .head(30)
)

print(high_total_credit)

) as file:


        total_credit    rent_value  credit_value  building_size  city_slug  \
48074   3.167667e+12  9.500000e+10  1.000000e+09          165.0     tehran   
190342  3.002000e+12  9.000000e+10  2.000000e+09          230.0     tehran   
719631  3.001000e+12  9.000000e+10  1.000000e+09          180.0     tehran   
530137  3.001000e+12  9.000000e+10  1.000000e+09           90.0      karaj   
554559  3.000300e+12  9.000000e+10  3.000000e+08           22.0      urmia   
196055  2.834833e+12  8.500000e+10  1.500000e+09          270.0     tehran   
409902  2.833833e+12  8.500000e+10  5.000000e+08          175.0     tehran   
368791  2.644309e+12  7.932790e+10  4.527859e+07          222.0   dehloran   
146040  2.543954e+12  7.631235e+10  2.089950e+08          170.0     shiraz   
186389  2.502000e+12  7.500000e+10  2.000000e+09          220.0     tehran   
959724  2.222222e+12  6.666667e+10  0.000000e+00          120.0     dezful   
749574  2.101000e+12  6.300000e+10  1.000000e+09          131.0 

In [25]:
high_total_credit_analysis = (
    rental_clean[
        rental_clean["total_credit"] > 50_000_000_000
    ][
        [
            "total_credit",
            "rent_value",
            "credit_value",
            "building_size",
            "city_slug",
            "cat3_slug",
            "rooms_count",
            "construction_year"
        ]
    ]
    .copy()
)

high_total_credit_analysis["total_credit_per_size"] = (
    high_total_credit_analysis["total_credit"]
    / high_total_credit_analysis["building_size"]
)

high_total_credit_analysis = (
    high_total_credit_analysis
    .sort_values("total_credit_per_size", ascending=False)
    .head(30)
)

print(high_total_credit_analysis)

) as file:


        total_credit    rent_value  credit_value  building_size  \
587728  1.050000e+12  3.000000e+10  5.000000e+10            2.0   
144301  6.686667e+11  2.000000e+10  2.000000e+09            2.0   
618105  3.304527e+11  9.876543e+09  1.234568e+09            1.0   
614196  1.918485e+12  5.555556e+10  6.663334e+10           12.0   
554559  3.000300e+12  9.000000e+10  3.000000e+08           22.0   
442530  5.005000e+10  1.500000e+09  5.000000e+07            1.0   
263632  3.134943e+11  9.130903e+09  9.130903e+09            7.0   
886993  3.703836e+11  1.111114e+10  1.221111e+07           11.0   
530137  3.001000e+12  9.000000e+10  1.000000e+09           90.0   
64663   3.714815e+11  1.111111e+10  1.111111e+09           12.0   
417285  3.704815e+11  1.111111e+10  1.111111e+08           12.0   
111762  3.501000e+11  1.050000e+10  1.000000e+08           15.0   
774866  3.501000e+11  1.050000e+10  1.000000e+08           15.0   
755497  6.866667e+10  2.000000e+09  2.000000e+09            3.

In [26]:
total_credit_clean = rental_clean[
    rental_clean["total_credit"] <= 500_000_000_000
].copy()

print("Original records:", len(rental_clean))
print("Clean records:", len(total_credit_clean))
print("Removed records:", len(rental_clean) - len(total_credit_clean))

Original records: 350608
Clean records: 350560
Removed records: 48


In [27]:
low_total_credit = (
    total_credit_clean[
        total_credit_clean["total_credit"] < 10_000_000
    ][
        [
            "total_credit",
            "rent_value",
            "credit_value",
            "building_size",
            "city_slug",
            "cat3_slug",
            "rooms_count",
            "construction_year"
        ]
    ]
    .sort_values("total_credit")
    .head(30)
)

print(low_total_credit)

) as file:


          total_credit  rent_value  credit_value  building_size  \
919893       34.333333         1.0           1.0           25.0   
771800      111.000000         0.0         111.0          130.0   
298449      433.333333         1.0         400.0           76.0   
591368      470.000000         0.0         470.0           80.0   
218883      533.333333        10.0         200.0          100.0   
307446      600.000000         0.0         600.0          100.0   
410576     3633.333333       100.0         300.0          100.0   
839729    34333.333333      1000.0        1000.0       100000.0   
381408    50110.000000      1500.0         110.0           50.0   
360983   381477.666667     11111.0       11111.0           50.0   
253682  1000000.000000         0.0     1000000.0           80.0   
918663  1000000.000000         0.0     1000000.0          120.0   
21349   1000000.000000         0.0     1000000.0           90.0   
994149  1000000.000000         0.0     1000000.0           85.

In [28]:
final_target_data = total_credit_clean[
    total_credit_clean["total_credit"] >= 1_000_000
].copy()

print("Original records:", len(total_credit_clean))
print("Final records:", len(final_target_data))
print("Removed records:", len(total_credit_clean) - len(final_target_data))

) as file:
    )

Original records: 350560
Final records: 350550
Removed records: 10


## 4. مهندسی ویژگی‌های پایه

ساخت سن ساختمان، امتیاز امکانات و تبدیل تعداد اتاق‌ها به مقدار عددی.


In [29]:
final_target_data["construction_year_numeric"] = (
    final_target_data["construction_year"]
    .replace("قبل از ۱۳۷۰", "1369")
    .astype(float)
)

final_target_data["building_age"] = (
    1405 - final_target_data["construction_year_numeric"]
)

print(
    final_target_data[
        ["construction_year", "construction_year_numeric", "building_age"]
    ].head(20)
)

   construction_year  construction_year_numeric  building_age
2               ۱۴۰۱                     1401.0           4.0
3               ۱۴۰۰                     1400.0           5.0
5               ۱۳۸۹                     1389.0          16.0
6               ۱۳۹۵                     1395.0          10.0
11              ۱۳۸۵                     1385.0          20.0
15              ۱۳۹۰                     1390.0          15.0
17              ۱۳۹۸                     1398.0           7.0
18              ۱۳۹۰                     1390.0          15.0
21              ۱۴۰۲                     1402.0           3.0
23              ۱۴۰۲                     1402.0           3.0
27       قبل از ۱۳۷۰                     1369.0          36.0
36              ۱۳۸۳                     1383.0          22.0
37              ۱۳۹۸                     1398.0           7.0
38              ۱۳۹۰                     1390.0          15.0
40              ۱۳۹۸                     1398.0           7.0
44      

In [30]:
luxury_features = [
    "has_pool",
    "has_sauna",
    "has_jacuzzi",
    "has_barbecue",
    "has_security_guard"
]

final_target_data["luxury_score"] = (
    final_target_data[luxury_features]
    .eq(True)
    .sum(axis=1)
)

print(
    final_target_data["luxury_score"]
    .value_counts()
    .sort_index()
)

) as file:
    )

luxury_score
0    349927
1       367
2       182
3        34
4        28
5        12
Name: count, dtype: int64


In [37]:
rooms_value_counts = (
    final_target_data["rooms_count"]
    .value_counts(dropna=False)
)

print(rooms_value_counts)

) as file:


rooms_count
دو              154262
یک               89872
سه               50229
بدون اتاق        45167
چهار              6175
پنج یا بیشتر      4550
NaN                295
Name: count, dtype: int64


In [38]:
code = '''rooms_mapping = {
    "بدون اتاق": 0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر": 5
}

final_target_data["rooms_count_numeric"] = (
    final_target_data["rooms_count"]
    .map(rooms_mapping)
)
'''



In [40]:
print(
    final_target_data["rooms_count_numeric"]
    .value_counts(dropna=False)
    .sort_index()
)

KeyError: 'rooms_count_numeric'

In [41]:
building_size_stats = final_target_data["building_size"].describe()

print(building_size_stats)



count    3.505500e+05
mean     3.202249e+03
std      1.183584e+05
min      1.000000e+00
25%      6.800000e+01
50%      9.200000e+01
75%      1.300000e+02
max      1.000000e+07
Name: building_size, dtype: float64


In [42]:
building_size_percentiles = (
    final_target_data["building_size"]
    .quantile([0.90, 0.95, 0.99, 0.995, 0.999])
)

print(building_size_percentiles)

) as file:


0.900       185.000000
0.950       265.000000
0.990      2200.000000
0.995     32000.000000
0.999    589024.392001
Name: building_size, dtype: float64


In [44]:
largest_building_sizes = (
    final_target_data[
        [
            "building_size",
            "total_credit",
            "city_slug",
            "cat2_slug",
            "cat3_slug",
            "rooms_count_numeric",
            "construction_year"
        ]
    ]
    .sort_values("building_size", ascending=False)
    .head(50)
)

print(largest_building_sizes)

) as file:


KeyError: "['rooms_count_numeric'] not in index"

In [45]:
large_building_size_counts = (
    final_target_data["building_size"]
    .value_counts()
    .sort_index(ascending=False)
    .head(30)
)

print(large_building_size_counts)

) as file:


building_size
10000000.0    27
9999999.0      2
9000000.0      6
8000000.0      1
7777777.0      1
7774447.0      1
7000000.0      1
6000000.0      1
5555555.0      2
5255555.0      1
5000000.0     10
4600000.0      1
4500000.0      4
4458847.0      1
4444444.0      1
4000000.0      8
3690000.0      1
3600000.0      1
3500000.0      1
3000000.0     13
2855455.0      1
2800000.0      3
2531655.0      1
2500000.0      5
2400000.0      1
2222333.0      1
2222222.0      3
2200000.0      1
2000000.0     24
1900000.0      1
Name: count, dtype: int64


In [46]:
large_building_size = final_target_data[
    final_target_data["building_size"] > 10_000
]

print("Records > 10,000 m²:", len(large_building_size))

print("\nBy category:")
print(
    large_building_size["cat2_slug"]
    .value_counts()
)

print("\nBy subcategory:")
print(
    large_building_size["cat3_slug"]
    .value_counts()
)

) as file:
    )
    )
    )

Records > 10,000 m²: 2275

By category:
cat2_slug
commercial-rent     1231
residential-rent    1044
Name: count, dtype: int64

By subcategory:
cat3_slug
industry-agriculture-business-rent    957
house-villa-rent                      628
apartment-rent                        416
shop-rent                             204
office-rent                            70
Name: count, dtype: int64


In [47]:
apartment_sizes = (
    final_target_data[
        final_target_data["cat3_slug"] == "apartment-rent"
    ]["building_size"]
)

print(apartment_sizes.describe())

print("\nPercentiles:")
print(
    apartment_sizes.quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

) as file:
    )

count    2.106500e+05
mean     7.833688e+02
std      5.048484e+04
min      1.000000e+00
25%      7.500000e+01
50%      9.500000e+01
75%      1.250000e+02
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       160.0
0.950       198.0
0.990       300.0
0.995       385.0
0.999    100000.0
Name: building_size, dtype: float64


In [48]:
apartment_size_thresholds = {
    "> 300": (apartment_sizes > 300).sum(),
    "> 500": (apartment_sizes > 500).sum(),
    "> 1000": (apartment_sizes > 1000).sum(),
    "> 2000": (apartment_sizes > 2000).sum(),
    "> 5000": (apartment_sizes > 5000).sum(),
    "> 10000": (apartment_sizes > 10000).sum(),
    "> 50000": (apartment_sizes > 50000).sum(),
    "> 100000": (apartment_sizes > 100000).sum(),
}

for threshold, count in apartment_size_thresholds.items():
    print(f"{threshold}: {count}")

) as file:
    for threshold, count in apartment_size_thresholds.items():


> 300: 1956
> 500: 681
> 1000: 520
> 2000: 449
> 5000: 436
> 10000: 416
> 50000: 350
> 100000: 201


In [51]:
large_apartments = (
    final_target_data[
        (final_target_data["cat3_slug"] == "apartment-rent") &
        (final_target_data["building_size"] > 500)
    ][
        [
            "building_size",
            "total_credit",
            "city_slug",
            "rooms_count_numeric",
            "construction_year",
            "building_age",
            "luxury_score"
        ]
    ]
    .sort_values("building_size", ascending=False)
)

print(large_apartments.head(100))

) as file:


KeyError: "['rooms_count_numeric'] not in index"

In [52]:
house_villa_sizes = (
    final_target_data[
        final_target_data["cat3_slug"] == "house-villa-rent"
    ]["building_size"]
)

print(house_villa_sizes.describe())

print("\nPercentiles:")
print(
    house_villa_sizes.quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

) as file:
    )

count    6.427000e+04
mean     4.671729e+03
std      1.471001e+05
min      1.000000e+00
25%      8.000000e+01
50%      1.000000e+02
75%      1.500000e+02
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900        200.00
0.950        300.00
0.990       8181.66
0.995     111111.00
0.999    1111111.00
Name: building_size, dtype: float64


In [53]:
categories = [
    "shop-rent",
    "office-rent",
    "industry-agriculture-business-rent"
]

for category in categories:
    sizes = (
        final_target_data[
            final_target_data["cat3_slug"] == category
        ]["building_size"]
    )

    print("=" * 70)
    print(f"Category: {category}")
    print("=" * 70)

    print("\nStatistics:")
    print(sizes.describe())

    print("\nPercentiles:")
    print(
        sizes.quantile(
            [0.90, 0.95, 0.99, 0.995, 0.999]
        )
    )

    print("\nThreshold counts:")
    for threshold in [300, 500, 1000, 2000, 5000, 10000, 50000, 100000]:
        print(
            f"> {threshold:,} m²: "
            f"{(sizes > threshold).sum():,}"
        )

    print()

Category: shop-rent

Statistics:
count    4.556500e+04
mean     1.901329e+03
std      7.694285e+04
min      1.000000e+00
25%      2.000000e+01
50%      3.900000e+01
75%      8.000000e+01
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       180.0
0.950       300.0
0.990      1000.0
0.995      4000.0
0.999    350000.0
Name: building_size, dtype: float64

Threshold counts:
> 300 m²: 1,936
> 500 m²: 970
> 1,000 m²: 440
> 2,000 m²: 270
> 5,000 m²: 219
> 10,000 m²: 204
> 50,000 m²: 143
> 100,000 m²: 109

Category: office-rent

Statistics:
count    2.109500e+04
mean     1.957580e+03
std      1.008438e+05
min      1.000000e+00
25%      6.000000e+01
50%      9.000000e+01
75%      1.350000e+02
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       240.0
0.950       420.0
0.990      1500.0
0.995      2553.0
0.999    179060.0
Name: building_size, dtype: float64

Threshold counts:
> 300 m²: 1,447
> 500 m²: 801
> 1,000 m²: 313
> 2,000 m²: 1

In [54]:
categories = [
    "office-rent",
    "industry-agriculture-business-rent"
]

for category in categories:
    sizes = (
        final_target_data[
            final_target_data["cat3_slug"] == category
        ]["building_size"]
    )

    print("=" * 70)
    print(f"Category: {category}")
    print("=" * 70)

    print("\nStatistics:")
    print(sizes.describe())

    print("\nPercentiles:")
    print(
        sizes.quantile(
            [0.90, 0.95, 0.99, 0.995, 0.999]
        )
    )

    print("\nThreshold counts:")
    for threshold in [300, 500, 1000, 2000, 5000, 10000, 50000, 100000]:
        print(
            f"> {threshold:,} m²: "
            f"{(sizes > threshold).sum():,}"
        )

    print()

Category: office-rent

Statistics:
count    2.109500e+04
mean     1.957580e+03
std      1.008438e+05
min      1.000000e+00
25%      6.000000e+01
50%      9.000000e+01
75%      1.350000e+02
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       240.0
0.950       420.0
0.990      1500.0
0.995      2553.0
0.999    179060.0
Name: building_size, dtype: float64

Threshold counts:
> 300 m²: 1,447
> 500 m²: 801
> 1,000 m²: 313
> 2,000 m²: 128
> 5,000 m²: 77
> 10,000 m²: 70
> 50,000 m²: 54
> 100,000 m²: 46

Category: industry-agriculture-business-rent

Statistics:
count    8.970000e+03
mean     5.901341e+04
std      5.247246e+05
min      1.000000e+00
25%      2.000000e+02
50%      5.000000e+02
75%      2.000000e+03
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       13000.0
0.950       80000.0
0.990     1200000.0
0.995     3000000.0
0.999    10000000.0
Name: building_size, dtype: float64

Threshold counts:
> 300 m²: 5,441
> 500 m²: 4,

In [55]:
sizes = (
    final_target_data[
        final_target_data["cat3_slug"] == "industry-agriculture-business-rent"
    ]["building_size"]
)

print("=" * 70)
print("Category: industry-agriculture-business-rent")
print("=" * 70)

print("\nStatistics:")
print(sizes.describe())

print("\nPercentiles:")
print(
    sizes.quantile(
        [0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

print("\nThreshold counts:")
for threshold in [300, 500, 1000, 2000, 5000, 10000, 50000, 100000]:
    print(
        f"> {threshold:,} m²: "
        f"{(sizes > threshold).sum():,}"
    )

Category: industry-agriculture-business-rent

Statistics:
count    8.970000e+03
mean     5.901341e+04
std      5.247246e+05
min      1.000000e+00
25%      2.000000e+02
50%      5.000000e+02
75%      2.000000e+03
max      1.000000e+07
Name: building_size, dtype: float64

Percentiles:
0.900       13000.0
0.950       80000.0
0.990     1200000.0
0.995     3000000.0
0.999    10000000.0
Name: building_size, dtype: float64

Threshold counts:
> 300 m²: 5,441
> 500 m²: 4,196
> 1,000 m²: 2,933
> 2,000 m²: 2,015
> 5,000 m²: 1,253
> 10,000 m²: 957
> 50,000 m²: 545
> 100,000 m²: 365


In [56]:
categories = [
    "apartment-rent",
    "house-villa-rent",
    "shop-rent",
    "office-rent",
    "industry-agriculture-business-rent"
]

) as file:

    for category in categories:

        data = final_target_data[
            final_target_data["cat3_slug"] == category
        ]

        sizes = data["building_size"]

        output = []

        output.append("=" * 80)
        output.append(f"Category: {category}")
        output.append(f"Number of records: {len(data):,}")
        output.append("=" * 80)

        output.append("\nTop 20 largest building sizes:")

        top_20 = (
            data[
                [
                    "building_size",
                    "total_credit",
                    "city_slug",
                    "rooms_count"
                ]
            ]
            .sort_values("building_size", ascending=False)
            .head(20)
            .to_string(index=False)
        )

        output.append(top_20)

        output.append("\nThreshold counts:")

        for threshold in [
            1_000,
            2_000,
            5_000,
            10_000,
            50_000,
            100_000,
            500_000,
            1_000_000,
            5_000_000
        ]:
            count = (sizes > threshold).sum()

            output.append(
                f"> {threshold:,} m²: "
                f"{count:,} "
                f"({count / len(data) * 100:.3f}%)"
            )

        result = "\n".join(output)

        print(result)
        print()

print("Output saved to building_size_categories_check.txt")

Category: apartment-rent
Number of records: 210,650

Top 20 largest building sizes:
 building_size  total_credit      city_slug  rooms_count
    10000000.0  1.040744e+09       javanrud           دو
    10000000.0  1.453333e+07        ardabil           دو
    10000000.0  1.011111e+08      aligudarz           دو
     9999999.0  3.033333e+08        mashhad           دو
     5000000.0  1.500000e+08         saqqez           دو
     3690000.0  1.622850e+08        saravan           دو
     3600000.0  5.000000e+10         tehran پنج یا بیشتر
     3000000.0  3.833333e+08          zabol           دو
     2855455.0  1.083333e+08           baft           یک
     2222333.0  3.433333e+07       maragheh           دو
     2222222.0  3.803704e+08 bandar-ganaveh           سه
     2000000.0  3.463333e+09         rudbar         چهار
     2000000.0  9.444444e+07        zahedan    بدون اتاق
     2000000.0  3.833333e+07            qom پنج یا بیشتر
     1500000.0  6.500000e+07        isfahan           یک
    

In [57]:
size_limits = {
    "apartment-rent": 500,
    "house-villa-rent": 2_000,
    "shop-rent": 2_000,
    "office-rent": 2_000,
    "industry-agriculture-business-rent": 500_000
}

size_filter = pd.Series(True, index=final_target_data.index)

for category, limit in size_limits.items():
    size_filter &= ~(
        (final_target_data["cat3_slug"] == category) &
        (final_target_data["building_size"] > limit)
    )

final_target_data = final_target_data[size_filter].copy()

print("Number of records after building size cleaning:")
print(f"{len(final_target_data):,}")

Number of records after building size cleaning:
348,627


## 5. بررسی و مدیریت مقادیر گمشده

بررسی ویژگی‌های دودویی و دسته‌ای و تصمیم‌گیری درباره نحوه برخورد با Missing Values.


In [58]:
binary_features = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_electricity",
    "has_heating_system",
    "has_cooling_system"
]

output = []

for column in binary_features:
    output.append("=" * 60)
    output.append(f"Column: {column}")
    output.append("=" * 60)

    result = final_target_data[column].value_counts(dropna=False)

    output.append(result.to_string())
    output.append("")


# نمایش در خروجی
print("\n".join(output))


# ذخیره در فایل
) as file:
print("\nOutput saved to binary_features_check.txt")

Column: has_elevator
has_elevator
True     139690
NaN      117691
False     91246

Column: has_warehouse
has_warehouse
True     214775
False     79747
NaN       54105

Column: has_parking
has_parking
True     198486
False     96036
NaN       54105

Column: has_electricity
has_electricity
NaN      343287
False      3050
True       2290

Column: has_heating_system
has_heating_system
NaN              203831
shoofaj           71678
heater            55071
duct_split         8628
floor_heating      2352
split              2073
fan_coil           1978
fireplace          1727
unselect           1289

Column: has_cooling_system
has_cooling_system
NaN                209977
water_cooler        94502
air_conditioner     16494
split               12594
duct_split          10781
unselect             2320
fan_coil             1959


Output saved to binary_features_check.txt


In [59]:
print("df1 columns:")
print(df1.columns.tolist())

print("\nSample city names from df1:")
print(df1["نام شهر"].head(20).to_string(index=False))

print("\nSample city slugs from final_target_data:")
print(final_target_data["city_slug"].head(20).to_string(index=False))

df1 columns:
['نام شهر', 'دسته\u200cبندی']

Sample city names from df1:
            karaj
           tehran
          mashhad
            ahvaz
       kermanshah
    mahdasht-city
      pardis-city
     foolad-shahr
            rasht
          mahabad
           shiraz
          gonabad
             azna
andisheh-new-town
        chamestan
            babol
             arak
           pishva
           tabriz
            urmia

Sample city slugs from final_target_data:
           tehran
           tehran
            ahvaz
       kermanshah
           tehran
            rasht
          mahabad
           shiraz
andisheh-new-town
            karaj
           tehran
           tehran
           shiraz
       nasimshahr
             amol
             ilam
       kermanshah
          isfahan
           tehran
        najafabad


In [60]:
city_check = (
    df1.groupby("نام شهر")["دسته\u200cبندی"]
    .nunique()
)

print("Number of cities:", len(city_check))
print("Cities with multiple categories:", (city_check > 1).sum())

if (city_check > 1).sum() > 0:
    print("\nCities with multiple categories:")
    print(city_check[city_check > 1])

Number of cities: 240
Cities with multiple categories: 0


## 6. اضافه کردن نوع شهر

ادغام اطلاعات دسته‌بندی شهرها و بررسی شهرهای unmatched.


In [61]:
city_mapping = df1[
    [
        "نام شهر",
        "دسته\u200cبندی"
    ]
].rename(
    columns={
        "نام شهر": "city_slug",
        "دسته\u200cبندی": "city_type"
    }
)

final_target_data = final_target_data.merge(
    city_mapping,
    on="city_slug",
    how="left"
)

print(final_target_data["city_type"].value_counts(dropna=False))

city_type
کلان‌شهر    186102
شهر کوچک    154262
NaN           8263
Name: count, dtype: int64


In [62]:
missing_city_type = (
    final_target_data[
        final_target_data["city_type"].isna()
    ]["city_slug"]
    .value_counts(dropna=False)
)

print("Number of unmatched city slugs:")
print(missing_city_type.shape[0])

print("\nUnmatched cities:")
print(missing_city_type.to_string())

Number of unmatched city slugs:
175

Unmatched cities:
city_slug
andimeshk               503
langarud                379
nurabad                 330
aligudarz               305
qorveh                  301
garmsar                 298
fasa-city               271
bandar-Imam-khomeini    244
dehdasht                242
takestan                208
norabad                 190
bukan                   183
kangavar                178
kamyaran                138
meybod                  138
firuzabad               132
aleshtar                131
lordegan                130
ardakan                 118
eshtehard               116
shahedshahr             110
eqbaliyeh               103
bijar                    98
baghershahr              94
aliabad-katul            92
dehgolan                 92
eyvan                    84
siahkal                  84
ashkhaneh                83
sahneh                   79
ghayen                   77
zarghan                  74
abdanan                  73
naqadeh    

In [63]:
main_cities = set(
    final_target_data["city_slug"].dropna().unique()
)

city_mapping_cities = set(
    df1["نام شهر"].dropna().unique()
)

common_cities = main_cities.intersection(city_mapping_cities)

print("Cities in final_target_data:", len(main_cities))
print("Cities in df1:", len(city_mapping_cities))
print("Common cities:", len(common_cities))
print("Unmatched cities:", len(main_cities - city_mapping_cities))

Cities in final_target_data: 414
Cities in df1: 240
Common cities: 240
Unmatched cities: 174


In [64]:
final_target_data["city_type"] = (
    final_target_data["city_type"]
    .fillna("شهر کوچک")
)

print(final_target_data["city_type"].value_counts(dropna=False))

city_type
کلان‌شهر    186102
شهر کوچک    162525
Name: count, dtype: int64


In [65]:
binary_missing_features = [
    "has_elevator",
    "has_warehouse",
    "has_parking"
]

categorical_missing_features = [
    "has_heating_system",
    "has_cooling_system",
    "cat3_slug",
    "city_slug"
]


# Fill missing values in binary features with False
for col in binary_missing_features:
    final_target_data[col] = (
        final_target_data[col]
        .fillna(False)
        .infer_objects(copy=False)
    )


# Fill missing values in categorical features with "Unknown"
for col in categorical_missing_features:
    final_target_data[col] = (
        final_target_data[col]
        .fillna("Unknown")
    )


# Check the result
features_to_check = (
    binary_missing_features +
    categorical_missing_features
)

output = []

output.append("=" * 70)
output.append("Missing Values Check After Filling")
output.append("=" * 70)

for col in features_to_check:
    missing_count = final_target_data[col].isna().sum()
    missing_percentage = (
        final_target_data[col].isna().mean() * 100
    )

    output.append(
        f"\nFeature: {col}"
    )
    output.append(
        f"Dtype: {final_target_data[col].dtype}"
    )
    output.append(
        f"Missing: {missing_count}"
    )
    output.append(
        f"Missing percentage: {missing_percentage:.2f}%"
    )


result = "\n".join(output)

# Show in notebook
print(result)

# Save to file
) as file:


C:\Users\Amin\AppData\Local\Temp\ipykernel_9484\3770527764.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


Missing Values Check After Filling

Feature: has_elevator
Dtype: bool
Missing: 0
Missing percentage: 0.00%

Feature: has_warehouse
Dtype: bool
Missing: 0
Missing percentage: 0.00%

Feature: has_parking
Dtype: bool
Missing: 0
Missing percentage: 0.00%

Feature: has_heating_system
Dtype: object
Missing: 0
Missing percentage: 0.00%

Feature: has_cooling_system
Dtype: object
Missing: 0
Missing percentage: 0.00%

Feature: cat3_slug
Dtype: object
Missing: 0
Missing percentage: 0.00%

Feature: city_slug
Dtype: object
Missing: 0
Missing percentage: 0.00%


In [66]:
numeric_features = [
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score"
]

) as file:
    )


In [68]:
output = []

output.append("=" * 70)
output.append("Numeric Features Check")
output.append("=" * 70)

for col in numeric_features:
    output.append(f"\nFeature: {col}")
    output.append(f"Dtype: {final_target_data[col].dtype}")

    missing_count = final_target_data[col].isna().sum()
    missing_percentage = (
        final_target_data[col].isna().mean() * 100
    )

    output.append(f"Missing: {missing_count}")
    output.append(
        f"Missing percentage: {missing_percentage:.2f}%"
    )

    output.append("\nDescriptive statistics:")
    output.append(
        final_target_data[col].describe().to_string()
    )


result = "\n".join(output)

# نمایش در Notebook
print(result)

# ذخیره در فایل
) as file:


KeyError: 'rooms_count_numeric'

In [69]:
final_target_data["building_age"] = (
    final_target_data["building_age"]
    .fillna(final_target_data["building_age"].median())
)

final_target_data["rooms_count_numeric"] = (
    final_target_data["rooms_count_numeric"]
    .fillna(final_target_data["rooms_count_numeric"].median())
)

KeyError: 'rooms_count_numeric'

In [70]:
# Calculate median values before filling missing values
building_age_median = final_target_data["building_age"].median()
rooms_count_median = final_target_data["rooms_count_numeric"].median()


# Fill missing values with median
final_target_data["building_age"] = (
    final_target_data["building_age"]
    .fillna(building_age_median)
)

final_target_data["rooms_count_numeric"] = (
    final_target_data["rooms_count_numeric"]
    .fillna(rooms_count_median)
)


# Check the result
features_to_check = [
    "building_age",
    "rooms_count_numeric"
]

output = []

output.append("=" * 70)
output.append("Numeric Features Missing Value Handling")
output.append("=" * 70)

output.append(
    f"\nbuilding_age median used: {building_age_median}"
)

output.append(
    f"rooms_count_numeric median used: {rooms_count_median}"
)

for col in features_to_check:
    missing_count = final_target_data[col].isna().sum()
    missing_percentage = (
        final_target_data[col].isna().mean() * 100
    )

    output.append(f"\nFeature: {col}")
    output.append(f"Dtype: {final_target_data[col].dtype}")
    output.append(f"Missing: {missing_count}")
    output.append(
        f"Missing percentage: {missing_percentage:.2f}%"
    )


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


KeyError: 'rooms_count_numeric'

## 7. بررسی Outlierها

بررسی آماری ویژگی‌های عددی و اعمال فقط محدودیت‌های مورد تأیید پروژه.


In [71]:
output = []

output.append("=" * 70)
output.append("Outlier Detection - IQR Method")
output.append("=" * 70)

for col in numeric_features:
    q1 = final_target_data[col].quantile(0.25)
    q3 = final_target_data[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (final_target_data[col] < lower_bound) |
        (final_target_data[col] > upper_bound)
    )

    outlier_count = outlier_mask.sum()
    outlier_percentage = (
        outlier_count / len(final_target_data) * 100
    )

    output.append(f"\nFeature: {col}")
    output.append("-" * 70)

    output.append(f"Q1: {q1}")
    output.append(f"Q3: {q3}")
    output.append(f"IQR: {iqr}")
    output.append(f"Lower bound: {lower_bound}")
    output.append(f"Upper bound: {upper_bound}")
    output.append(f"Outlier count: {outlier_count}")
    output.append(
        f"Outlier percentage: {outlier_percentage:.2f}%"
    )


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


KeyError: 'rooms_count_numeric'

In [72]:
size_limits = {
    "apartment-rent": 500,
    "house-villa-rent": 2_000,
    "shop-rent": 2_000,
    "office-rent": 2_000,
    "industry-agriculture-business-rent": 500_000
}

output = []

output.append("=" * 70)
output.append("Final Building Size Validation")
output.append("=" * 70)

for category, limit in size_limits.items():

    category_data = final_target_data[
        final_target_data["cat3_slug"] == category
    ]

    invalid_count = (
        category_data["building_size"] > limit
    ).sum()

    output.append(
        f"\nCategory: {category}"
    )
    output.append(
        f"Limit: {limit}"
    )
    output.append(
        f"Records: {len(category_data)}"
    )
    output.append(
        f"Invalid records above limit: {invalid_count}"
    )

    if len(category_data) > 0:
        output.append(
            f"Maximum building_size: "
            f"{category_data['building_size'].max()}"
        )


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


Final Building Size Validation

Category: apartment-rent
Limit: 500
Records: 209969
Invalid records above limit: 0
Maximum building_size: 500.0

Category: house-villa-rent
Limit: 2000
Records: 63586
Invalid records above limit: 0
Maximum building_size: 2000.0

Category: shop-rent
Limit: 2000
Records: 45295
Invalid records above limit: 0
Maximum building_size: 2000.0

Category: office-rent
Limit: 2000
Records: 20967
Invalid records above limit: 0
Maximum building_size: 2000.0

Category: industry-agriculture-business-rent
Limit: 500000
Records: 8810
Invalid records above limit: 0
Maximum building_size: 500000.0


## 8. تعریف ویژگی‌های اولیه مدل

تعریف ویژگی‌های عددی و دسته‌ای قبل از ورود به مدل‌سازی.


In [73]:

# Final feature lists

numeric_features = [
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score"
]

categorical_features = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_heating_system",
    "has_cooling_system",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "city_type"
]

features = numeric_features + categorical_features

target = "total_credit"


# Create X and y
X = final_target_data[features].copy()
y = final_target_data[target].copy()


# Create a report
output = []

output.append("=" * 70)
output.append("Final Features and Target")
output.append("=" * 70)

output.append(f"\nNumber of features: {len(features)}")
output.append("\nNumeric features:")

for col in numeric_features:
    output.append(f"- {col}")

output.append("\nCategorical / binary features:")

for col in categorical_features:
    output.append(f"- {col}")

output.append(f"\nTarget:")
output.append(f"- {target}")

output.append("\nShapes:")
output.append(f"X shape: {X.shape}")
output.append(f"y shape: {y.shape}")


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


KeyError: "['rooms_count_numeric'] not in index"

## 9. تقسیم داده و آماده‌سازی برای مدل‌سازی

تقسیم Train/Validation/Test با random_state=42 و آماده‌سازی One-Hot Encoding.


In [74]:


# First split:
# 80% Train
# 20% Temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# Second split:
# 10% Validation
# 10% Test
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)


# Check the result
output = []

output.append("=" * 70)
output.append("Train / Validation / Test Split")
output.append("=" * 70)

output.append(f"\nX_train shape: {X_train.shape}")
output.append(f"X_validation shape: {X_validation.shape}")
output.append(f"X_test shape: {X_test.shape}")

output.append(f"\ny_train shape: {y_train.shape}")
output.append(f"y_validation shape: {y_validation.shape}")
output.append(f"y_test shape: {y_test.shape}")

output.append(
    f"\nTrain percentage: "
    f"{len(X_train) / len(X) * 100:.2f}%"
)

output.append(
    f"Validation percentage: "
    f"{len(X_validation) / len(X) * 100:.2f}%"
)

output.append(
    f"Test percentage: "
    f"{len(X_test) / len(X) * 100:.2f}%"
)


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


NameError: name 'X' is not defined

In [75]:


categorical_features = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_heating_system",
    "has_cooling_system",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "city_type"
]


# Create encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)


# Fit ONLY on training data
encoder.fit(X_train[categorical_features])


# Transform train, validation and test
X_train_encoded = encoder.transform(
    X_train[categorical_features]
)

X_validation_encoded = encoder.transform(
    X_validation[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)


# Check the result
output = []

output.append("=" * 70)
output.append("One-Hot Encoding Check")
output.append("=" * 70)

output.append(
    f"\nOriginal categorical features: "
    f"{len(categorical_features)}"
)

output.append(
    f"Encoded features: "
    f"{X_train_encoded.shape[1]}"
)

output.append(
    f"\nX_train encoded shape: "
    f"{X_train_encoded.shape}"
)

output.append(
    f"X_validation encoded shape: "
    f"{X_validation_encoded.shape}"
)

output.append(
    f"X_test encoded shape: "
    f"{X_test_encoded.shape}"
)

output.append(
    f"\nEncoder type: "
    f"{type(encoder).__name__}"
)

output.append(
    f"Sparse output: "
    f"{encoder.sparse_output}"
)


result = "\n".join(output)

# Show in Notebook
print(result)

# Save to file
) as file:


NameError: name 'X_train' is not defined

In [76]:

numeric_features = [
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score"
]

X_train_numeric = csr_matrix(
    X_train[numeric_features].values
)

X_validation_numeric = csr_matrix(
    X_validation[numeric_features].values
)

X_test_numeric = csr_matrix(
    X_test[numeric_features].values
)

X_train_final = hstack([
    X_train_numeric,
    X_train_encoded
]).tocsr()

X_validation_final = hstack([
    X_validation_numeric,
    X_validation_encoded
]).tocsr()

X_test_final = hstack([
    X_test_numeric,
    X_test_encoded
]).tocsr()

output = []
output.append("=" * 70)
output.append("Final Feature Matrix Check")
output.append("=" * 70)
output.append("")

output.append(f"Numeric features: {len(numeric_features)}")
output.append(f"Encoded categorical features: {X_train_encoded.shape[1]}")
output.append(
    f"Total final features: {X_train_final.shape[1]}"
)
output.append("")

output.append(
    f"X_train final shape: {X_train_final.shape}"
)
output.append(
    f"X_validation final shape: {X_validation_final.shape}"
)
output.append(
    f"X_test final shape: {X_test_final.shape}"
)
output.append("")

output.append(f"X_train type: {type(X_train_final).__name__}")
output.append(f"X_validation type: {type(X_validation_final).__name__}")
output.append(f"X_test type: {type(X_test_final).__name__}")

) as file:
print("\n".join(output))



NameError: name 'X_train' is not defined

In [77]:
numeric_features = [
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score"
]

output = []
output.append("=" * 70)
output.append("Numeric Features Scale Check")
output.append("=" * 70)
output.append("")

for col in numeric_features:
    series = X_train[col]

    output.append(f"Feature: {col}")
    output.append(f"Min: {series.min()}")
    output.append(f"Max: {series.max()}")
    output.append(f"Mean: {series.mean()}")
    output.append(f"Std: {series.std()}")
    output.append("")

) as file:
print("\n".join(output))



NameError: name 'X_train' is not defined

In [78]:
numeric_features = [
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score"
]

scaler = StandardScaler()

# Fit فقط روی داده‌های Train
X_train_numeric_scaled = scaler.fit_transform(
    X_train[numeric_features]
)

# فقط Transform روی Validation و Test
X_validation_numeric_scaled = scaler.transform(
    X_validation[numeric_features]
)

X_test_numeric_scaled = scaler.transform(
    X_test[numeric_features]
)

# تبدیل به sparse
X_train_numeric_scaled = csr_matrix(
    X_train_numeric_scaled
)

X_validation_numeric_scaled = csr_matrix(
    X_validation_numeric_scaled
)

X_test_numeric_scaled = csr_matrix(
    X_test_numeric_scaled
)

# ترکیب Numeric + One-Hot
X_train_scaled = hstack([
    X_train_numeric_scaled,
    X_train_encoded
]).tocsr()

X_validation_scaled = hstack([
    X_validation_numeric_scaled,
    X_validation_encoded
]).tocsr()

X_test_scaled = hstack([
    X_test_numeric_scaled,
    X_test_encoded
]).tocsr()

output = []
output.append("=" * 70)
output.append("Standard Scaling Check")
output.append("=" * 70)
output.append("")

output.append("Scaler: StandardScaler")
output.append("Scaler fitted on: X_train only")
output.append("")

output.append(
    f"X_train scaled shape: {X_train_scaled.shape}"
)
output.append(
    f"X_validation scaled shape: {X_validation_scaled.shape}"
)
output.append(
    f"X_test scaled shape: {X_test_scaled.shape}"
)
output.append("")

output.append(
    f"X_train type: {type(X_train_scaled).__name__}"
)
output.append(
    f"X_validation type: {type(X_validation_scaled).__name__}"
)
output.append(
    f"X_test type: {type(X_test_scaled).__name__}"
)
output.append("")

output.append("Scaler means:")
for col, mean in zip(numeric_features, scaler.mean_):
    output.append(f"{col}: {mean}")

output.append("")
output.append("Scaler standard deviations:")
for col, scale in zip(numeric_features, scaler.scale_):
    output.append(f"{col}: {scale}")

) as file:
print("\n".join(output))



NameError: name 'X_train' is not defined

In [105]:

# ساخت مدل پایه
decision_tree_model = DecisionTreeRegressor(
    random_state=42
)

# آموزش فقط روی Train
decision_tree_model.fit(
    X_train_final,
    y_train
)

# پیش‌بینی روی Validation
y_validation_pred_dt = decision_tree_model.predict(
    X_validation_final
)

# محاسبه معیارها
dt_r2 = r2_score(
    y_validation,
    y_validation_pred_dt
)

dt_mae = mean_absolute_error(
    y_validation,
    y_validation_pred_dt
)

dt_mse = mean_squared_error(
    y_validation,
    y_validation_pred_dt
)

dt_rmse = dt_mse ** 0.5

output = []
output.append("=" * 70)
output.append("Decision Tree Regression - Validation Results")
output.append("=" * 70)
output.append("")

output.append(f"R²: {dt_r2:.6f}")
output.append(f"MAE: {dt_mae:.2f}")
output.append(f"MSE: {dt_mse:.2f}")
output.append(f"RMSE: {dt_rmse:.2f}")
output.append("")

output.append(
    f"Tree depth: {decision_tree_model.get_depth()}"
)
output.append(
    f"Number of leaves: {decision_tree_model.get_n_leaves()}"
)

) as file:
print("\n".join(output))



Decision Tree Regression - Validation Results

R²: -0.986683
MAE: 1063468570.08
MSE: 153756217079817764864.00
RMSE: 12399847461.96

Tree depth: 185
Number of leaves: 203955


In [106]:

depths = [5, 10, 15, 20]

results = []

for depth in depths:
    decision_tree_model = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42
    )

    decision_tree_model.fit(
        X_train_final,
        y_train
    )

    y_validation_pred_dt = decision_tree_model.predict(
        X_validation_final
    )

    r2 = r2_score(
        y_validation,
        y_validation_pred_dt
    )

    mae = mean_absolute_error(
        y_validation,
        y_validation_pred_dt
    )

    mse = mean_squared_error(
        y_validation,
        y_validation_pred_dt
    )

    rmse = mse ** 0.5

    results.append({
        "max_depth": depth,
        "r2": r2,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "actual_depth": decision_tree_model.get_depth(),
        "n_leaves": decision_tree_model.get_n_leaves()
    })


) as f:

    for result in results:
print("Decision Tree depth tuning completed.")

for result in results:
    print(
        f"max_depth={result['max_depth']} | "
        f"R2={result['r2']:.6f} | "
        f"MAE={result['mae']:,.2f} | "
        f"RMSE={result['rmse']:,.2f}"
    )



Decision Tree depth tuning completed.
max_depth=5 | R2=0.037571 | MAE=897,431,589.11 | RMSE=8,630,511,009.60
max_depth=10 | R2=-0.191494 | MAE=892,182,473.64 | RMSE=9,602,801,487.61
max_depth=15 | R2=-0.384649 | MAE=914,830,091.84 | RMSE=10,351,944,232.19
max_depth=20 | R2=-0.598513 | MAE=932,457,045.46 | RMSE=11,122,698,953.74


In [107]:

leaf_sizes = [1, 5, 10, 20, 50]

results = []

for leaf_size in leaf_sizes:
    decision_tree_model = DecisionTreeRegressor(
        max_depth=5,
        min_samples_leaf=leaf_size,
        random_state=42
    )

    decision_tree_model.fit(
        X_train_final,
        y_train
    )

    y_validation_pred_dt = decision_tree_model.predict(
        X_validation_final
    )

    r2 = r2_score(
        y_validation,
        y_validation_pred_dt
    )

    mae = mean_absolute_error(
        y_validation,
        y_validation_pred_dt
    )

    mse = mean_squared_error(
        y_validation,
        y_validation_pred_dt
    )

    rmse = mse ** 0.5

    results.append({
        "min_samples_leaf": leaf_size,
        "r2": r2,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "actual_depth": decision_tree_model.get_depth(),
        "n_leaves": decision_tree_model.get_n_leaves()
    })


) as f:

    for result in results:
        )
print("Decision Tree min_samples_leaf tuning completed.")

for result in results:
    print(
        f"min_samples_leaf={result['min_samples_leaf']} | "
        f"R2={result['r2']:.6f} | "
        f"MAE={result['mae']:,.2f} | "
        f"RMSE={result['rmse']:,.2f}"
    )



Decision Tree min_samples_leaf tuning completed.
min_samples_leaf=1 | R2=0.037571 | MAE=897,431,589.11 | RMSE=8,630,511,009.60
min_samples_leaf=5 | R2=0.034927 | MAE=896,009,691.09 | RMSE=8,642,354,347.77
min_samples_leaf=10 | R2=0.035495 | MAE=895,923,039.64 | RMSE=8,639,813,040.82
min_samples_leaf=20 | R2=0.036140 | MAE=892,557,203.61 | RMSE=8,636,922,431.93
min_samples_leaf=50 | R2=0.038850 | MAE=881,875,012.12 | RMSE=8,624,773,838.58


In [108]:

max_features_values = [1.0, 0.7, 0.5, 0.3]

results = []

for max_features_value in max_features_values:
    decision_tree_model = DecisionTreeRegressor(
        max_depth=5,
        min_samples_leaf=50,
        max_features=max_features_value,
        random_state=42
    )

    decision_tree_model.fit(
        X_train_final,
        y_train
    )

    y_validation_pred_dt = decision_tree_model.predict(
        X_validation_final
    )

    r2 = r2_score(
        y_validation,
        y_validation_pred_dt
    )

    mae = mean_absolute_error(
        y_validation,
        y_validation_pred_dt
    )

    mse = mean_squared_error(
        y_validation,
        y_validation_pred_dt
    )

    rmse = mse ** 0.5

    results.append({
        "max_features": max_features_value,
        "r2": r2,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "actual_depth": decision_tree_model.get_depth(),
        "n_leaves": decision_tree_model.get_n_leaves()
    })


) as f:

    for result in results:
        )
print("Decision Tree max_features tuning completed.")

for result in results:
    print(
        f"max_features={result['max_features']} | "
        f"R2={result['r2']:.6f} | "
        f"MAE={result['mae']:,.2f} | "
        f"RMSE={result['rmse']:,.2f}"
    )



Decision Tree max_features tuning completed.
max_features=1.0 | R2=0.038850 | MAE=881,875,012.12 | RMSE=8,624,773,838.58
max_features=0.7 | R2=0.039220 | MAE=911,397,181.60 | RMSE=8,623,110,578.95
max_features=0.5 | R2=0.034566 | MAE=920,060,635.41 | RMSE=8,643,972,476.65
max_features=0.3 | R2=0.031925 | MAE=937,340,801.79 | RMSE=8,655,784,880.03


## 11. Random Forest

اجرای Random Forest و بررسی عملکرد آن.


In [112]:

random_forest_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_final,
    y_train
)

y_validation_pred_rf = (
    random_forest_model.predict(
        X_validation_final
    )
)

rf_r2 = r2_score(
    y_validation,
    y_validation_pred_rf
)

rf_mae = mean_absolute_error(
    y_validation,
    y_validation_pred_rf
)

rf_mse = mean_squared_error(
    y_validation,
    y_validation_pred_rf
)

rf_rmse = rf_mse ** 0.5


print("Random Forest - Baseline")
print("-" * 50)
print(f"R2: {rf_r2:.6f}")
print(f"MAE: {rf_mae:,.2f}")
print(f"MSE: {rf_mse:,.2f}")
print(f"RMSE: {rf_rmse:,.2f}")


) as f:



Random Forest - Baseline
--------------------------------------------------
R2: -0.203346
MAE: 941,695,564.37
MSE: 93,131,116,446,998,495,232.00
RMSE: 9,650,446,437.70


In [113]:
# Target جدید
target = "rent_value"

y_rent = final_target_data[target].copy()


# استفاده از همان Train / Validation / Test قبلی
y_train_rent = y_rent.loc[X_train.index]
y_validation_rent = y_rent.loc[X_validation.index]
y_test_rent = y_rent.loc[X_test.index]


# بررسی ابعاد
print("Target:", target)
print("-" * 50)

print("y_train_rent shape:", y_train_rent.shape)
print("y_validation_rent shape:", y_validation_rent.shape)
print("y_test_rent shape:", y_test_rent.shape)


# بررسی Missing
print("\nMissing values:")
print("Train:", y_train_rent.isna().sum())
print("Validation:", y_validation_rent.isna().sum())
print("Test:", y_test_rent.isna().sum())


# بررسی آماری target
print("\nTarget statistics:")
print(
    y_rent.describe()
)


# ذخیره گزارش
) as f:

    )

    )

    )

    )

    )
    )
    )

    )

Target: rent_value
--------------------------------------------------
y_train_rent shape: (278901,)
y_validation_rent shape: (34863,)
y_test_rent shape: (34863,)

Missing values:
Train: 0
Validation: 0
Test: 0

Target statistics:
count    3.486270e+05
mean     2.121001e+07
std      2.428930e+08
min      0.000000e+00
25%      1.111110e+05
50%      5.000000e+06
75%      1.200000e+07
max      1.300000e+10
Name: rent_value, dtype: float64


In [115]:

random_forest_rent_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_rent_model.fit(
    X_train_final,
    y_train_rent
)

y_validation_pred_rf_rent = (
    random_forest_rent_model.predict(
        X_validation_final
    )
)


rf_rent_r2 = r2_score(
    y_validation_rent,
    y_validation_pred_rf_rent
)

rf_rent_mae = mean_absolute_error(
    y_validation_rent,
    y_validation_pred_rf_rent
)

rf_rent_mse = mean_squared_error(
    y_validation_rent,
    y_validation_pred_rf_rent
)

rf_rent_rmse = rf_rent_mse ** 0.5


print("Random Forest - Rent Value Baseline")
print("-" * 50)
print(f"R2: {rf_rent_r2:.6f}")
print(f"MAE: {rf_rent_mae:,.2f}")
print(f"MSE: {rf_rent_mse:,.2f}")
print(f"RMSE: {rf_rent_rmse:,.2f}")


) as f:



Random Forest - Rent Value Baseline
--------------------------------------------------
R2: -0.271033
MAE: 27,490,393.93
MSE: 83,673,240,176,444,080.00
RMSE: 289,263,271.39


## 12. تعریف نهایی Target یعنی rent_value

مقادیر rent_value برابر صفر به‌عنوان «توافقی» از داده آموزشی حذف می‌شوند و مقدار مثبت برای پیش‌بینی استفاده می‌شود.


In [116]:
# بررسی وضعیت target یعنی rent_value

rent_target_check = final_target_data["rent_value"]

print("Rent Value Target Check")
print("-" * 60)

# تعداد صفرها
zero_count = (
    rent_target_check == 0
).sum()

zero_percentage = (
    zero_count / len(rent_target_check)
) * 100

print(f"Zero values: {zero_count:,}")
print(f"Zero percentage: {zero_percentage:.2f}%")


# تعداد مقادیر بسیار بزرگ
large_100m = (
    rent_target_check > 100_000_000
).sum()

large_500m = (
    rent_target_check > 500_000_000
).sum()

large_1b = (
    rent_target_check > 1_000_000_000
).sum()

print("\nLarge values:")
print(f"> 100M: {large_100m:,}")
print(f"> 500M: {large_500m:,}")
print(f"> 1B: {large_1b:,}")


# صدک‌ها
percentiles = (
    rent_target_check
    .quantile([
        0,
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.995,
        0.999,
        1
    ])
)

print("\nPercentiles:")
print(percentiles)


# بررسی صفرها بر اساس دسته‌بندی ملک
zero_by_category = (
    final_target_data[
        final_target_data["rent_value"] == 0
    ]
    .groupby("cat3_slug")
    .size()
    .sort_values(ascending=False)
)

print("\nZero rent values by category:")
print(zero_by_category)


# ذخیره گزارش
) as f:

    )

    )

    )

    )
    )
    )

    )

    )



Rent Value Target Check
------------------------------------------------------------
Zero values: 59,010
Zero percentage: 16.93%

Large values:
> 100M: 8,239
> 500M: 949
> 1B: 595

Percentiles:
0.000    0.000000e+00
0.010    0.000000e+00
0.050    0.000000e+00
0.100    0.000000e+00
0.250    1.111110e+05
0.500    5.000000e+06
0.750    1.200000e+07
0.900    3.000000e+07
0.950    5.500000e+07
0.990    2.000000e+08
0.995    3.000000e+08
0.999    1.200000e+09
1.000    1.300000e+10
Name: rent_value, dtype: float64

Zero rent values by category:
cat3_slug
apartment-rent                        47730
house-villa-rent                       7521
shop-rent                              1813
office-rent                            1607
industry-agriculture-business-rent      339
dtype: int64


In [117]:
zero_rent = final_target_data[
    final_target_data["rent_value"] == 0
].copy()

nonzero_rent = final_target_data[
    final_target_data["rent_value"] > 0
].copy()

zero_rent_check = zero_rent.groupby("cat3_slug").agg(
    count=("rent_value", "size"),
    median_credit=("credit_value", "median"),
    mean_credit=("credit_value", "mean"),
    missing_credit=("credit_value", lambda x: x.isna().sum())
)

nonzero_rent_check = nonzero_rent.groupby("cat3_slug").agg(
    count=("rent_value", "size"),
    median_rent=("rent_value", "median"),
    median_credit=("credit_value", "median"),
    missing_credit=("credit_value", lambda x: x.isna().sum())
)

) as file:
    )

    )


In [120]:
rent_target_data = final_target_data[
    final_target_data["rent_value"] > 0
].copy()

rent_value_summary = rent_target_data["rent_value"].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.995,
        0.999
    ]
)

) as file:

    )

    )

    )

print(
    f"Total records: {len(rent_target_data):,}"
)

print(
    f"Removed zero values: "
    f"{len(final_target_data) - len(rent_target_data):,}"
)

print(rent_value_summary)


Total records: 289,617
Removed zero values: 59,010
count    2.896170e+05
mean     2.553159e+07
std      2.662845e+08
min      1.000000e+00
1%       1.000000e+05
5%       1.000000e+05
10%      1.500000e+05
25%      3.000000e+06
50%      6.300000e+06
75%      1.500000e+07
90%      3.500000e+07
95%      6.000000e+07
99%      2.250000e+08
99.5%    3.600000e+08
99.9%    1.500000e+09
max      1.300000e+10
Name: rent_value, dtype: float64


In [121]:
large_rent_check = (
    rent_target_data
    .groupby("cat3_slug")["rent_value"]
    .agg(
        count="count",
        median="median",
        q90=lambda x: x.quantile(0.90),
        q95=lambda x: x.quantile(0.95),
        q99=lambda x: x.quantile(0.99),
        q995=lambda x: x.quantile(0.995),
        q999=lambda x: x.quantile(0.999),
        max="max"
    )
)

) as file:

    )
    )

    )

print(large_rent_check)



                                     count      median          q90  \
cat3_slug                                                             
apartment-rent                      162239   6000000.0   30000000.0   
house-villa-rent                     56065   4000000.0   15000000.0   
industry-agriculture-business-rent    8471  17000000.0  100000000.0   
office-rent                          19360  15000000.0   70000000.0   
shop-rent                            43482  10000000.0   60000000.0   

                                            q95           q99          q995  \
cat3_slug                                                                     
apartment-rent                       46000000.0  1.400000e+08  2.000000e+08   
house-villa-rent                     30000000.0  1.900000e+08  3.768000e+08   
industry-agriculture-business-rent  150000000.0  1.111111e+09  1.500000e+09   
office-rent                         130000000.0  4.500000e+08  9.000000e+08   
shop-rent                   

In [122]:
category_quantiles = (
    rent_target_data
    .groupby("cat3_slug")["rent_value"]
    .quantile(0.999)
)

extreme_rent_rows = []

for category, threshold in category_quantiles.items():

    category_data = rent_target_data[
        rent_target_data["cat3_slug"] == category
    ]

    extreme_rows = category_data[
        category_data["rent_value"] >= threshold
    ].copy()

    extreme_rent_rows.append(
        extreme_rows
    )

extreme_rent_data = pd.concat(
    extreme_rent_rows
)

columns_to_check = [
    "cat3_slug",
    "city_slug",
    "city_type",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "luxury_score",
    "credit_value",
    "rent_value"
]

extreme_rent_data = (
    extreme_rent_data[columns_to_check]
    .sort_values(
        ["cat3_slug", "rent_value"],
        ascending=[True, False]
    )
)

) as file:

    )

print(
    f"Extreme records: {len(extreme_rent_data):,}"
)

print(
    extreme_rent_data.to_string(index=False)
)



Extreme records: 360
                         cat3_slug            city_slug city_type  building_size  building_age  rooms_count_numeric  luxury_score  credit_value   rent_value
                    apartment-rent               tehran  کلان‌شهر           45.0           8.0                  1.0             0  1.000000e+08 1.300000e+10
                    apartment-rent               tehran  کلان‌شهر           70.0          17.0                  2.0             0  4.000000e+08 1.300000e+10
                    apartment-rent              mashhad  کلان‌شهر          115.0          14.0                  2.0             0  2.000000e+08 1.290000e+10
                    apartment-rent               tehran  کلان‌شهر           52.0          25.0                  1.0             0  1.000000e+08 1.250000e+10
                    apartment-rent               tehran  کلان‌شهر           97.0          16.0                  2.0             0  8.000000e+08 1.250000e+10
                    apartment-rent   

In [124]:
category_quantiles = (
    rent_target_data
    .groupby("cat3_slug")["rent_value"]
    .quantile(0.999)
)

summary = []

for category, threshold in category_quantiles.items():

    category_data = rent_target_data[
        rent_target_data["cat3_slug"] == category
    ]

    extreme_data = category_data[
        category_data["rent_value"] >= threshold
    ]

    summary.append({
        "category": category,
        "threshold_999": threshold,
        "extreme_count": len(extreme_data),
        "max_rent": extreme_data["rent_value"].max(),
        "median_rent": extreme_data["rent_value"].median(),
        "median_building_size": (
            extreme_data["building_size"].median()
        ),
        "median_credit": (
            extreme_data["credit_value"].median()
        )
    })

extreme_summary = pd.DataFrame(summary)

) as file:

    )

print(extreme_summary)



                             category  threshold_999  extreme_count  \
0                      apartment-rent   1.111111e+09            178   
1                    house-villa-rent   1.111111e+09             99   
2  industry-agriculture-business-rent   1.111111e+10             18   
3                         office-rent   2.000000e+09             21   
4                           shop-rent   2.603800e+09             44   

       max_rent   median_rent  median_building_size  median_credit  
0  1.300000e+10  4.000000e+09                  90.0   2.000000e+08  
1  1.200000e+10  1.600000e+09                 120.0   2.200000e+08  
2  1.111111e+10  1.111111e+10                 665.0   8.055556e+08  
3  1.111111e+10  1.050000e+10                 150.0   1.111111e+09  
4  1.250000e+10  1.111111e+10                  40.0   1.111111e+09  


## 13. بازسازی Train/Validation/Test برای Target نهایی

تقسیم مجدد داده پس از تعیین مجموعه هدف.


In [125]:

X = rent_target_data[features].copy()
y = rent_target_data["rent_value"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

y_train_rent = y_train
y_validation_rent = y_validation
y_test_rent = y_test

print(
    f"X_train: {X_train.shape}"
)

print(
    f"X_validation: {X_validation.shape}"
)

print(
    f"X_test: {X_test.shape}"
)

print(
    f"y_train: {y_train.shape}"
)

print(
    f"y_validation: {y_validation.shape}"
)

print(
    f"y_test: {y_test.shape}"
)


X_train: (231693, 13)
X_validation: (28962, 13)
X_test: (28962, 13)
y_train: (231693,)
y_validation: (28962,)
y_test: (28962,)


In [126]:

categorical_features = [
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "has_heating_system",
    "has_cooling_system",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "city_type"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder.fit(
    X_train[categorical_features]
)

X_train_encoded = encoder.transform(
    X_train[categorical_features]
)

X_validation_encoded = encoder.transform(
    X_validation[categorical_features]
)

X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

print(
    f"X_train_encoded: {X_train_encoded.shape}"
)

print(
    f"X_validation_encoded: {X_validation_encoded.shape}"
)

print(
    f"X_test_encoded: {X_test_encoded.shape}"
)


X_train_encoded: (231693, 442)
X_validation_encoded: (28962, 442)
X_test_encoded: (28962, 442)


In [127]:

X_train_numeric = csr_matrix(
    X_train[numeric_features].values
)

X_validation_numeric = csr_matrix(
    X_validation[numeric_features].values
)

X_test_numeric = csr_matrix(
    X_test[numeric_features].values
)

X_train_final = hstack(
    [
        X_train_numeric,
        X_train_encoded
    ]
).tocsr()

X_validation_final = hstack(
    [
        X_validation_numeric,
        X_validation_encoded
    ]
).tocsr()

X_test_final = hstack(
    [
        X_test_numeric,
        X_test_encoded
    ]
).tocsr()

print(
    f"X_train_final: {X_train_final.shape}"
)

print(
    f"X_validation_final: {X_validation_final.shape}"
)

print(
    f"X_test_final: {X_test_final.shape}"
)


X_train_final: (231693, 446)
X_validation_final: (28962, 446)
X_test_final: (28962, 446)


In [130]:

random_forest_rent_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_rent_model.fit(
    X_train_final,
    y_train_rent
)

y_validation_pred_rf_rent = (
    random_forest_rent_model.predict(
        X_validation_final
    )
)

r2 = r2_score(
    y_validation_rent,
    y_validation_pred_rf_rent
)

mae = mean_absolute_error(
    y_validation_rent,
    y_validation_pred_rf_rent
)

mse = mean_squared_error(
    y_validation_rent,
    y_validation_pred_rf_rent
)

rmse = mse ** 0.5

) as file:

    )

    )

    )

    )

    )

print(
    f"R2: {r2:.6f}"
)

print(
    f"MAE: {mae:,.2f}"
)

print(
    f"MSE: {mse:,.2f}"
)

print(
    f"RMSE: {rmse:,.2f}"
)

R2: -0.778669
MAE: 28,172,778.93
MSE: 64,205,547,945,681,064.00
RMSE: 253,388,136.95


In [131]:
import numpy as np

validation_error = (
    y_validation_rent
    - y_validation_pred_rf_rent
)

absolute_error = np.abs(
    validation_error
)

error_percentage = (
    absolute_error
    / y_validation_rent
) * 100

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

print(
    f"Mean Absolute Error: {absolute_error.mean():,.2f}"
)

print(
    f"Median Absolute Error: {np.median(absolute_error):,.2f}"
)

print(
    f"90th Percentile Error: "
    f"{np.percentile(absolute_error, 90):,.2f}"
)

print(
    f"95th Percentile Error: "
    f"{np.percentile(absolute_error, 95):,.2f}"
)

print(
    f"99th Percentile Error: "
    f"{np.percentile(absolute_error, 99):,.2f}"
)

print(
    f"Median Relative Error: "
    f"{np.median(error_percentage):.2f}%"
)

print(
    f"Mean Relative Error: "
    f"{error_percentage.mean():.2f}%"
)


Mean Absolute Error: 28,172,778.93
Median Absolute Error: 4,037,861.11
90th Percentile Error: 29,037,766.71
95th Percentile Error: 63,829,100.00
99th Percentile Error: 332,265,771.09
Median Relative Error: 49.32%
Mean Relative Error: 3507.36%


## 14. Random Forest با Target لگاریتمی

استفاده از log1p برای کاهش اثر چولگی شدید rent_value.


In [133]:


y_train_log = np.log1p(
    y_train_rent
)

random_forest_log_rent_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_log_rent_model.fit(
    X_train_final,
    y_train_log
)

y_validation_pred_log = (
    random_forest_log_rent_model.predict(
        X_validation_final
    )
)

y_validation_pred_log_rent = np.expm1(
    y_validation_pred_log
)

r2_log_rent = r2_score(
    y_validation_rent,
    y_validation_pred_log_rent
)

mae_log_rent = mean_absolute_error(
    y_validation_rent,
    y_validation_pred_log_rent
)

mse_log_rent = mean_squared_error(
    y_validation_rent,
    y_validation_pred_log_rent
)

rmse_log_rent = mse_log_rent ** 0.5

) as file:

    )

    )

    )

    )

    )

print(
    f"R2: {r2_log_rent:.6f}"
)

print(
    f"MAE: {mae_log_rent:,.2f}"
)

print(
    f"MSE: {mse_log_rent:,.2f}"
)

print(
    f"RMSE: {rmse_log_rent:,.2f}"
)


R2: 0.014278
MAE: 17,332,644.20
MSE: 35,582,123,961,134,500.00
RMSE: 188,632,245.28


In [134]:
feature_names = (
    numeric_features
    + list(
        encoder.get_feature_names_out(
            categorical_features
        )
    )
)

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": (
        random_forest_log_rent_model
        .feature_importances_
    )
})

feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

) as file:

    )

    )

print(
    feature_importance.head(30).to_string(
        index=False
    )
)


                           feature  importance
                     building_size    0.266484
                      building_age    0.175749
                  city_slug_tehran    0.062384
        cat2_slug_residential-rent    0.048452
               rooms_count_numeric    0.047757
               has_warehouse_False    0.013677
                has_warehouse_True    0.013644
                  has_parking_True    0.012226
        has_heating_system_shoofaj    0.011612
                 has_parking_False    0.011381
        has_heating_system_Unknown    0.010902
   has_cooling_system_water_cooler    0.010642
                 has_elevator_True    0.010325
        has_cooling_system_Unknown    0.010281
                has_elevator_False    0.009577
         has_heating_system_heater    0.008556
         cat2_slug_commercial-rent    0.008530
has_cooling_system_air_conditioner    0.008132
          cat3_slug_apartment-rent    0.006966
        cat3_slug_house-villa-rent    0.006240
             

## 15. تحلیل Neighborhood و اطلاعات مکانی

بررسی نقش محله و مختصات جغرافیایی و آماده‌سازی آزمایش‌های مربوط به Neighborhood.


In [135]:
neighborhood_analysis = pd.DataFrame({
    "missing": [
        rent_target_data["neighborhood_slug"].isna().sum()
    ],
    "non_missing": [
        rent_target_data["neighborhood_slug"].notna().sum()
    ],
    "unique": [
        rent_target_data["neighborhood_slug"].nunique()
    ]
})

missing_percentage = (
    rent_target_data["neighborhood_slug"].isna().mean()
    * 100
)

neighborhood_counts = (
    rent_target_data["neighborhood_slug"]
    .value_counts(dropna=True)
)

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

    )

print(
    f"Total rows: {len(rent_target_data):,}"
)

print(
    f"Missing: "
    f"{neighborhood_analysis.loc[0, 'missing']:,}"
)

print(
    f"Missing percentage: "
    f"{missing_percentage:.2f}%"
)

print(
    f"Unique neighborhoods: "
    f"{neighborhood_analysis.loc[0, 'unique']:,}"
)

print("\nTop 30 neighborhoods:")

print(
    neighborhood_counts
    .head(30)
    .to_string()
)


Total rows: 289,617
Missing: 145,541
Missing percentage: 50.25%
Unique neighborhoods: 1,104

Top 30 neighborhoods:
neighborhood_slug
saadat-abad        2098
ghasemabad         2053
poonak             1800
elahiyehblvd       1661
west-tehranpars    1463
gohardasht         1347
bolvartoos         1347
azadshahr          1227
elahiyeh           1144
niavaran           1094
zafaraniyeh        1082
tohid-ghom         1067
hashemieh          1019
kooy-e-ferdos       962
azimieh             954
farmaniyeh          939
pardisan-ghom       890
marlik              888
north-tabars        842
east-tehranpars     830
pasdaran            826
qeytariyeh          785
farhangshahr        762
marzdaran           757
velenjak            731
jordan              719
moaliabad           709
chitgar-lake        705
baghestan           681
sarsabil            678


In [136]:
latitude_missing = (
    rent_target_data["location_latitude"]
    .isna()
)

longitude_missing = (
    rent_target_data["location_longitude"]
    .isna()
)

both_missing = (
    latitude_missing
    & longitude_missing
)

both_available = (
    ~latitude_missing
    & ~longitude_missing
)

latitude_only = (
    ~latitude_missing
    & longitude_missing
)

longitude_only = (
    latitude_missing
    & ~longitude_missing
)

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

    )

print(
    f"Total rows: "
    f"{len(rent_target_data):,}"
)

print(
    f"Both coordinates available: "
    f"{both_available.sum():,} "
    f"({both_available.mean() * 100:.2f}%)"
)

print(
    f"Both coordinates missing: "
    f"{both_missing.sum():,} "
    f"({both_missing.mean() * 100:.2f}%)"
)

print(
    f"Only latitude available: "
    f"{latitude_only.sum():,}"
)

print(
    f"Only longitude available: "
    f"{longitude_only.sum():,}"
)

print("\nLatitude:")
print(
    rent_target_data["location_latitude"]
    .describe()
)

print("\nLongitude:")
print(
    rent_target_data["location_longitude"]
    .describe()
)


Total rows: 289,617
Both coordinates available: 191,976 (66.29%)
Both coordinates missing: 97,641 (33.71%)
Only latitude available: 0
Only longitude available: 0

Latitude:
count    191976.000000
mean         34.863767
std           2.376663
min          23.636976
25%          34.340694
50%          35.719757
75%          35.964994
max          39.808537
Name: location_latitude, dtype: float64

Longitude:
count    191976.000000
mean         51.830161
std           3.334789
min          43.305359
25%          50.864252
50%          51.377592
75%          51.803175
max          74.511620
Name: location_longitude, dtype: float64


In [137]:

neighborhood_data = rent_target_data.copy()

neighborhood_data["neighborhood_slug"] = (
    neighborhood_data["neighborhood_slug"]
    .fillna("Unknown")
)

features_with_neighborhood = (
    numeric_features
    + categorical_features
    + ["neighborhood_slug"]
)

X_neighborhood = neighborhood_data[
    features_with_neighborhood
].copy()

y_neighborhood = neighborhood_data[
    "rent_value"
].copy()

X_train_n, X_temp_n, y_train_n, y_temp_n = train_test_split(
    X_neighborhood,
    y_neighborhood,
    test_size=0.20,
    random_state=42
)

X_validation_n, X_test_n, y_validation_n, y_test_n = (
    train_test_split(
        X_temp_n,
        y_temp_n,
        test_size=0.50,
        random_state=42
    )
)

categorical_features_n = (
    categorical_features
    + ["neighborhood_slug"]
)

encoder_n = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_n.fit(
    X_train_n[categorical_features_n]
)

X_train_cat_n = encoder_n.transform(
    X_train_n[categorical_features_n]
)

X_validation_cat_n = encoder_n.transform(
    X_validation_n[categorical_features_n]
)

X_test_cat_n = encoder_n.transform(
    X_test_n[categorical_features_n]
)

X_train_num_n = csr_matrix(
    X_train_n[numeric_features].values
)

X_validation_num_n = csr_matrix(
    X_validation_n[numeric_features].values
)

X_test_num_n = csr_matrix(
    X_test_n[numeric_features].values
)

X_train_final_n = hstack([
    X_train_num_n,
    X_train_cat_n
]).tocsr()

X_validation_final_n = hstack([
    X_validation_num_n,
    X_validation_cat_n
]).tocsr()

X_test_final_n = hstack([
    X_test_num_n,
    X_test_cat_n
]).tocsr()

y_train_log_n = np.log1p(
    y_train_n
)

random_forest_neighborhood_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_neighborhood_model.fit(
    X_train_final_n,
    y_train_log_n
)

y_validation_pred_log_n = (
    random_forest_neighborhood_model.predict(
        X_validation_final_n
    )
)

y_validation_pred_n = np.expm1(
    y_validation_pred_log_n
)

r2_neighborhood = r2_score(
    y_validation_n,
    y_validation_pred_n
)

mae_neighborhood = mean_absolute_error(
    y_validation_n,
    y_validation_pred_n
)

mse_neighborhood = mean_squared_error(
    y_validation_n,
    y_validation_pred_n
)

rmse_neighborhood = mse_neighborhood ** 0.5

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

print(
    f"R2: {r2_neighborhood:.6f}"
)

print(
    f"MAE: {mae_neighborhood:,.2f}"
)

print(
    f"RMSE: {rmse_neighborhood:,.2f}"
)


R2: 0.026356
MAE: 16,168,354.70
RMSE: 187,473,027.45


In [138]:


geo_data = rent_target_data.copy()

geo_data["neighborhood_slug"] = (
    geo_data["neighborhood_slug"]
    .fillna("Unknown")
)

geo_data["coordinates_available"] = (
    geo_data["location_latitude"].notna()
    & geo_data["location_longitude"].notna()
).astype(int)

features_with_geo = (
    numeric_features
    + ["location_latitude", "location_longitude",
       "coordinates_available"]
    + categorical_features
    + ["neighborhood_slug"]
)

X_geo = geo_data[
    features_with_geo
].copy()

y_geo = geo_data[
    "rent_value"
].copy()

X_train_g, X_temp_g, y_train_g, y_temp_g = train_test_split(
    X_geo,
    y_geo,
    test_size=0.20,
    random_state=42
)

X_validation_g, X_test_g, y_validation_g, y_test_g = (
    train_test_split(
        X_temp_g,
        y_temp_g,
        test_size=0.50,
        random_state=42
    )
)

categorical_features_g = (
    categorical_features
    + ["neighborhood_slug"]
)

encoder_g = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_g.fit(
    X_train_g[categorical_features_g]
)

X_train_cat_g = encoder_g.transform(
    X_train_g[categorical_features_g]
)

X_validation_cat_g = encoder_g.transform(
    X_validation_g[categorical_features_g]
)

X_test_cat_g = encoder_g.transform(
    X_test_g[categorical_features_g]
)

numeric_features_g = (
    numeric_features
    + [
        "location_latitude",
        "location_longitude",
        "coordinates_available"
    ]
)

train_geo_medians = (
    X_train_g[
        ["location_latitude", "location_longitude"]
    ]
    .median()
)

X_train_g[
    ["location_latitude", "location_longitude"]
] = X_train_g[
    ["location_latitude", "location_longitude"]
].fillna(train_geo_medians)

X_validation_g[
    ["location_latitude", "location_longitude"]
] = X_validation_g[
    ["location_latitude", "location_longitude"]
].fillna(train_geo_medians)

X_test_g[
    ["location_latitude", "location_longitude"]
] = X_test_g[
    ["location_latitude", "location_longitude"]
].fillna(train_geo_medians)

X_train_num_g = csr_matrix(
    X_train_g[numeric_features_g].values
)

X_validation_num_g = csr_matrix(
    X_validation_g[numeric_features_g].values
)

X_test_num_g = csr_matrix(
    X_test_g[numeric_features_g].values
)

X_train_final_g = hstack([
    X_train_num_g,
    X_train_cat_g
]).tocsr()

X_validation_final_g = hstack([
    X_validation_num_g,
    X_validation_cat_g
]).tocsr()

X_test_final_g = hstack([
    X_test_num_g,
    X_test_cat_g
]).tocsr()

y_train_log_g = np.log1p(
    y_train_g
)

random_forest_geo_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_forest_geo_model.fit(
    X_train_final_g,
    y_train_log_g
)

y_validation_pred_log_g = (
    random_forest_geo_model.predict(
        X_validation_final_g
    )
)

y_validation_pred_g = np.expm1(
    y_validation_pred_log_g
)

r2_geo = r2_score(
    y_validation_g,
    y_validation_pred_g
)

mae_geo = mean_absolute_error(
    y_validation_g,
    y_validation_pred_g
)

mse_geo = mean_squared_error(
    y_validation_g,
    y_validation_pred_g
)

rmse_geo = mse_geo ** 0.5

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

print(
    f"R2: {r2_geo:.6f}"
)

print(
    f"MAE: {mae_geo:,.2f}"
)

print(
    f"RMSE: {rmse_geo:,.2f}"
)


R2: 0.025013
MAE: 16,089,975.06
RMSE: 187,602,257.87


In [139]:

city_analysis = X_validation_n.copy()

city_analysis["actual_rent"] = y_validation_n.values

city_analysis["predicted_rent"] = (
    y_validation_pred_n
)

city_results = []

for city in city_analysis["city_slug"].unique():

    city_data = city_analysis[
        city_analysis["city_slug"] == city
    ]

    if len(city_data) < 30:
        continue

    city_r2 = r2_score(
        city_data["actual_rent"],
        city_data["predicted_rent"]
    )

    city_mae = mean_absolute_error(
        city_data["actual_rent"],
        city_data["predicted_rent"]
    )

    city_results.append({
        "city_slug": city,
        "sample_count": len(city_data),
        "r2": city_r2,
        "mae": city_mae
    })

city_results_df = pd.DataFrame(
    city_results
)

city_results_df = city_results_df.sort_values(
    "sample_count",
    ascending=False
)

) as file:

    )

    )

    )

    )

    )

    )

    )

    )

print(
    city_results_df.head(30)
)


            city_slug  sample_count        r2           mae
4              tehran          6511  0.209800  2.678033e+07
13            mashhad          2566  0.261487  1.024259e+07
11              karaj          1422  0.151566  9.010083e+06
6              shiraz          1295  0.001463  1.974763e+07
7             isfahan          1251  0.000433  1.400528e+07
5              tabriz           737  0.499944  9.876836e+06
8                 qom           640  0.239517  4.369523e+06
12         kermanshah           592  0.264375  6.070185e+06
10              urmia           466  0.267901  6.780999e+06
61             kerman           443 -0.002975  3.252103e+07
42              rasht           412  0.236053  7.359665e+06
15               yazd           396 -0.226989  5.555943e+06
9               ahvaz           391 -0.003558  2.606367e+07
37               arak           359 -0.016620  7.397007e+06
28            hamedan           346 -0.006966  7.008098e+07
3        bandar-abbas           305  0.0

In [140]:
city_target_analysis = (
    X_validation_n[
        ["city_slug"]
    ]
    .copy()
)

city_target_analysis["actual_rent"] = (
    y_validation_n.values
)

city_target_results = (
    city_target_analysis
    .groupby("city_slug")
    .agg(
        sample_count=("actual_rent", "size"),
        median_rent=("actual_rent", "median"),
        mean_rent=("actual_rent", "mean")
    )
    .reset_index()
)

city_target_results = city_target_results.merge(
    city_results_df[
        [
            "city_slug",
            "r2",
            "mae"
        ]
    ],
    on="city_slug",
    how="left"
)

city_target_results = city_target_results[
    city_target_results["sample_count"] >= 100
].copy()

city_target_results = city_target_results.sort_values(
    "sample_count",
    ascending=False
)

) as file:

    )

    )

print(
    city_target_results.head(30)
)


             city_slug  sample_count  median_rent     mean_rent        r2  \
333             tehran          6511   15000000.0  4.222676e+07  0.209800   
210            mashhad          2566    7500000.0  1.611424e+07  0.261487   
152              karaj          1422    6650000.0  1.344878e+07  0.151566   
312             shiraz          1295   10000000.0  2.759185e+07  0.001463   
138            isfahan          1251    8000000.0  2.050045e+07  0.000433   
326             tabriz           737    7000000.0  1.602962e+07  0.499944   
261                qom           640    4150000.0  6.564355e+06  0.239517   
160         kermanshah           592    4000000.0  9.124531e+06  0.264375   
341              urmia           466    8000000.0  1.227265e+07  0.267901   
159             kerman           443    6800000.0  3.667849e+07 -0.002975   
271              rasht           412    6000000.0  1.216103e+07  0.236053   
345               yazd           396    7000000.0  9.666824e+06 -0.226989   

In [141]:


y_validation_log_actual = np.log1p(
    y_validation_n
)

y_validation_log_predicted = (
    y_validation_pred_log_n
)

r2_log_scale = r2_score(
    y_validation_log_actual,
    y_validation_log_predicted
)

mae_log_scale = mean_absolute_error(
    y_validation_log_actual,
    y_validation_log_predicted
)

mse_log_scale = mean_squared_error(
    y_validation_log_actual,
    y_validation_log_predicted
)

rmse_log_scale = mse_log_scale ** 0.5

) as file:

    )

    )

    )

    )

    )

print(
    f"R2: {r2_log_scale:.6f}"
)

print(
    f"MAE: {mae_log_scale:.6f}"
)

print(
    f"RMSE: {rmse_log_scale:.6f}"
)


R2: 0.154607
MAE: 1.085692
RMSE: 1.622197


## 16. تنظیم پارامترهای Random Forest

مقایسه پارامترهای اصلی و انتخاب تنظیمات مناسب برای Random Forest.


In [142]:

rf_tuning_results = []

max_depth_values = [
    5,
    10,
    15,
    20,
    None
]

min_samples_leaf_values = [
    1,
    5,
    10,
    20
]

for max_depth in max_depth_values:

    for min_samples_leaf in min_samples_leaf_values:

        rf_model = RandomForestRegressor(
            n_estimators=100,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(
            X_train_final_n,
            y_train_log_n
        )

        y_validation_pred_log_tuning = (
            rf_model.predict(
                X_validation_final_n
            )
        )

        r2_log = r2_score(
            y_validation_log_actual,
            y_validation_pred_log_tuning
        )

        rf_tuning_results.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "r2_log": r2_log
        })

rf_tuning_results_df = pd.DataFrame(
    rf_tuning_results
)

rf_tuning_results_df = (
    rf_tuning_results_df
    .sort_values(
        "r2_log",
        ascending=False
    )
    .reset_index(drop=True)
)

) as file:

    )

    )

print(
    rf_tuning_results_df
)

    max_depth  min_samples_leaf    r2_log
0        20.0                 5  0.245419
1        20.0                10  0.244540
2        15.0                 5  0.242623
3         NaN                10  0.242211
4        20.0                20  0.242110
5        15.0                10  0.241910
6         NaN                20  0.241908
7        15.0                20  0.240293
8        15.0                 1  0.237674
9        20.0                 1  0.236760
10        NaN                 5  0.236539
11       10.0                 5  0.232258
12       10.0                10  0.231737
13       10.0                20  0.231498
14       10.0                 1  0.230198
15        5.0                 5  0.197860
16        5.0                10  0.197842
17        5.0                20  0.197835
18        5.0                 1  0.197802
19        NaN                 1  0.154607


In [143]:

n_estimators_values = [
    100,
    200,
    300,
    500
]

n_estimators_results = []

for n_estimators in n_estimators_values:

    rf_model_n = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=20,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_model_n.fit(
        X_train_final_n,
        y_train_log_n
    )

    y_validation_pred_log_n_estimators = (
        rf_model_n.predict(
            X_validation_final_n
        )
    )

    r2_log_n_estimators = r2_score(
        y_validation_log_actual,
        y_validation_pred_log_n_estimators
    )

    n_estimators_results.append({
        "n_estimators": n_estimators,
        "r2_log": r2_log_n_estimators
    })

n_estimators_results_df = pd.DataFrame(
    n_estimators_results
)

n_estimators_results_df = (
    n_estimators_results_df
    .sort_values(
        "r2_log",
        ascending=False
    )
    .reset_index(drop=True)
)

) as file:

    )

    )

print(
    n_estimators_results_df
)


   n_estimators    r2_log
0           500  0.246030
1           200  0.245754
2           300  0.245722
3           100  0.245419


## 17. بررسی اثر credit_value

بررسی رابطه credit_value با rent_value و اضافه کردن آن به مدل.


In [144]:

credit_data = rent_target_data.copy()

credit_data["neighborhood_slug"] = (
    credit_data["neighborhood_slug"]
    .fillna("Unknown")
)

features_with_credit = (
    numeric_features
    + ["credit_value"]
    + categorical_features
    + ["neighborhood_slug"]
)

X_credit = credit_data[
    features_with_credit
].copy()

y_credit = credit_data[
    "rent_value"
].copy()

X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_credit,
    y_credit,
    test_size=0.20,
    random_state=42
)

X_validation_c, X_test_c, y_validation_c, y_test_c = (
    train_test_split(
        X_temp_c,
        y_temp_c,
        test_size=0.50,
        random_state=42
    )
)

categorical_features_c = (
    categorical_features
    + ["neighborhood_slug"]
)

encoder_c = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_c.fit(
    X_train_c[categorical_features_c]
)

X_train_cat_c = encoder_c.transform(
    X_train_c[categorical_features_c]
)

X_validation_cat_c = encoder_c.transform(
    X_validation_c[categorical_features_c]
)

X_test_cat_c = encoder_c.transform(
    X_test_c[categorical_features_c]
)

numeric_features_c = (
    numeric_features
    + ["credit_value"]
)

X_train_num_c = csr_matrix(
    X_train_c[numeric_features_c].values
)

X_validation_num_c = csr_matrix(
    X_validation_c[numeric_features_c].values
)

X_test_num_c = csr_matrix(
    X_test_c[numeric_features_c].values
)

X_train_final_c = hstack([
    X_train_num_c,
    X_train_cat_c
]).tocsr()

X_validation_final_c = hstack([
    X_validation_num_c,
    X_validation_cat_c
]).tocsr()

X_test_final_c = hstack([
    X_test_num_c,
    X_test_cat_c
]).tocsr()

y_train_log_c = np.log1p(
    y_train_c
)

rf_credit_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_credit_model.fit(
    X_train_final_c,
    y_train_log_c
)

y_validation_pred_log_c = (
    rf_credit_model.predict(
        X_validation_final_c
    )
)

y_validation_log_actual_c = np.log1p(
    y_validation_c
)

r2_credit_log = r2_score(
    y_validation_log_actual_c,
    y_validation_pred_log_c
)

) as file:

    )

    )

    )

    )

    )

print(
    f"R2 on log scale: "
    f"{r2_credit_log:.6f}"
)


R2 on log scale: 0.532934


In [145]:
import pandas as pd

feature_names_c = (
    numeric_features_c
    + list(
        encoder_c.get_feature_names_out(
            categorical_features_c
        )
    )
)

feature_importance_c = pd.DataFrame({
    "feature": feature_names_c,
    "importance": rf_credit_model.feature_importances_
})

feature_importance_c = (
    feature_importance_c
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

) as file:

    )

print(
    feature_importance_c.head(30)
)


                            feature  importance
0                      credit_value    0.345143
1                     building_size    0.220903
2                  city_slug_tehran    0.081200
3        cat2_slug_residential-rent    0.061790
4                      building_age    0.061563
5               rooms_count_numeric    0.022024
6                 has_elevator_True    0.017568
7         cat2_slug_commercial-rent    0.011967
8         neighborhood_slug_Unknown    0.009919
9                has_elevator_False    0.009172
10              cat3_slug_shop-rent    0.008950
11                 has_parking_True    0.008228
12                   city_slug_kish    0.007795
13       has_heating_system_shoofaj    0.007694
14                has_parking_False    0.006751
15  has_cooling_system_water_cooler    0.006398
16        has_heating_system_heater    0.006331
17           city_slug_bandar-abbas    0.005171
18               city_type_کلان‌شهر    0.005119
19                city_slug_mashhad    0

In [146]:
group_importance = []

for feature_group in numeric_features_c:
    importance = feature_importance_c.loc[
        feature_importance_c["feature"] == feature_group,
        "importance"
    ].sum()

    group_importance.append({
        "feature_group": feature_group,
        "importance": importance
    })


for feature_group in categorical_features_c:

    importance = feature_importance_c[
        feature_importance_c["feature"].str.startswith(
            feature_group + "_"
        )
    ]["importance"].sum()

    group_importance.append({
        "feature_group": feature_group,
        "importance": importance
    })


group_importance_df = pd.DataFrame(
    group_importance
)

group_importance_df = (
    group_importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

) as file:

    )

print(
    group_importance_df
)


          feature_group  importance
0          credit_value    0.345143
1         building_size    0.220903
2             city_slug    0.139018
3             cat2_slug    0.073757
4          building_age    0.061563
5     neighborhood_slug    0.028090
6          has_elevator    0.026740
7   rooms_count_numeric    0.022024
8    has_heating_system    0.018978
9             cat3_slug    0.018154
10   has_cooling_system    0.016021
11          has_parking    0.014978
12            city_type    0.007493
13        has_warehouse    0.006984
14         luxury_score    0.000152


In [147]:
import pandas as pd

credit_rent_data = rent_target_data[
    ["credit_value", "rent_value"]
].copy()

credit_rent_data = credit_rent_data.dropna(
    subset=["credit_value", "rent_value"]
)

pearson_correlation = (
    credit_rent_data[
        "credit_value"
    ].corr(
        credit_rent_data["rent_value"],
        method="pearson"
    )
)

spearman_correlation = (
    credit_rent_data[
        "credit_value"
    ].corr(
        credit_rent_data["rent_value"],
        method="spearman"
    )
)

credit_rent_data["credit_group"] = pd.qcut(
    credit_rent_data["credit_value"],
    q=10,
    duplicates="drop"
)

credit_group_analysis = (
    credit_rent_data
    .groupby(
        "credit_group",
        observed=True
    )["rent_value"]
    .agg(
        count="count",
        median="median",
        mean="mean"
    )
    .reset_index()
)

) as file:

    )

    )

    )

    )

    )

    )

print(
    f"Pearson correlation: "
    f"{pearson_correlation:.6f}"
)

print(
    f"Spearman correlation: "
    f"{spearman_correlation:.6f}"
)

print(
    "\nRent statistics by Credit Value decile:"
)

print(
    credit_group_analysis
)


Pearson correlation: 0.121689
Spearman correlation: 0.216139

Rent statistics by Credit Value decile:
                   credit_group  count      median          mean
0          (-0.001, 30000000.0]  29322   3000000.0  1.735045e+07
1      (30000000.0, 70000000.0]  31502   5000000.0  9.675427e+06
2     (70000000.0, 100000000.0]  39769   7500000.0  1.530540e+07
3    (100000000.0, 150000000.0]  25392   6000000.0  2.205021e+07
4    (150000000.0, 200000000.0]  34591   8000000.0  1.872398e+07
5    (200000000.0, 250000002.2]  13194   5000000.0  1.708619e+07
6    (250000002.2, 350000000.0]  31497   7000000.0  1.555815e+07
7    (350000000.0, 500000000.0]  35898  11000000.0  2.454487e+07
8    (500000000.0, 800000000.0]  20052   6500000.0  2.170697e+07
9  (800000000.0, 99999999999.0]  28400  24000000.0  9.622283e+07


## 18. تعریف مجموعه ویژگی‌های Core

ساخت مجموعه ویژگی فشرده برای مقایسه منصفانه‌تر مدل‌های درختی.


In [148]:

core_numeric_features = [
    "building_size",
    "building_age",
    "credit_value"
]

core_categorical_features = [
    "city_slug",
    "cat2_slug"
]

core_features = (
    core_numeric_features
    + core_categorical_features
)

core_data = rent_target_data[
    core_features + ["rent_value"]
].copy()

X_core = core_data[
    core_features
].copy()

y_core = core_data[
    "rent_value"
].copy()

X_train_core, X_temp_core, y_train_core, y_temp_core = (
    train_test_split(
        X_core,
        y_core,
        test_size=0.20,
        random_state=42
    )
)

X_validation_core, X_test_core, y_validation_core, y_test_core = (
    train_test_split(
        X_temp_core,
        y_temp_core,
        test_size=0.50,
        random_state=42
    )
)

encoder_core = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_core.fit(
    X_train_core[
        core_categorical_features
    ]
)

X_train_core_cat = encoder_core.transform(
    X_train_core[
        core_categorical_features
    ]
)

X_validation_core_cat = encoder_core.transform(
    X_validation_core[
        core_categorical_features
    ]
)

X_test_core_cat = encoder_core.transform(
    X_test_core[
        core_categorical_features
    ]
)

X_train_core_num = csr_matrix(
    X_train_core[
        core_numeric_features
    ].values
)

X_validation_core_num = csr_matrix(
    X_validation_core[
        core_numeric_features
    ].values
)

X_test_core_num = csr_matrix(
    X_test_core[
        core_numeric_features
    ].values
)

X_train_core_final = hstack([
    X_train_core_num,
    X_train_core_cat
]).tocsr()

X_validation_core_final = hstack([
    X_validation_core_num,
    X_validation_core_cat
]).tocsr()

X_test_core_final = hstack([
    X_test_core_num,
    X_test_core_cat
]).tocsr()

y_train_core_log = np.log1p(
    y_train_core
)

y_validation_core_log = np.log1p(
    y_validation_core
)

rf_core_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_core_model.fit(
    X_train_core_final,
    y_train_core_log
)

y_validation_core_pred_log = (
    rf_core_model.predict(
        X_validation_core_final
    )
)

r2_core_log = r2_score(
    y_validation_core_log,
    y_validation_core_pred_log
)

) as file:

    )

    )

    )

    )

    )

    )

    )

print(
    f"R2 on log scale: "
    f"{r2_core_log:.6f}"
)

R2 on log scale: 0.473057


In [149]:

core_numeric_features_n = [
    "building_size",
    "building_age",
    "credit_value"
]

core_categorical_features_n = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug"
]

core_features_n = (
    core_numeric_features_n
    + core_categorical_features_n
)

core_data_n = rent_target_data[
    core_features_n + ["rent_value"]
].copy()

core_data_n["neighborhood_slug"] = (
    core_data_n["neighborhood_slug"]
    .fillna("Unknown")
)

X_core_n = core_data_n[
    core_features_n
].copy()

y_core_n = core_data_n[
    "rent_value"
].copy()

X_train_core_n, X_temp_core_n, y_train_core_n, y_temp_core_n = (
    train_test_split(
        X_core_n,
        y_core_n,
        test_size=0.20,
        random_state=42
    )
)

X_validation_core_n, X_test_core_n, y_validation_core_n, y_test_core_n = (
    train_test_split(
        X_temp_core_n,
        y_temp_core_n,
        test_size=0.50,
        random_state=42
    )
)

encoder_core_n = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_core_n.fit(
    X_train_core_n[
        core_categorical_features_n
    ]
)

X_train_core_n_cat = encoder_core_n.transform(
    X_train_core_n[
        core_categorical_features_n
    ]
)

X_validation_core_n_cat = encoder_core_n.transform(
    X_validation_core_n[
        core_categorical_features_n
    ]
)

X_test_core_n_cat = encoder_core_n.transform(
    X_test_core_n[
        core_categorical_features_n
    ]
)

X_train_core_n_num = csr_matrix(
    X_train_core_n[
        core_numeric_features_n
    ].values
)

X_validation_core_n_num = csr_matrix(
    X_validation_core_n[
        core_numeric_features_n
    ].values
)

X_test_core_n_num = csr_matrix(
    X_test_core_n[
        core_numeric_features_n
    ].values
)

X_train_core_n_final = hstack([
    X_train_core_n_num,
    X_train_core_n_cat
]).tocsr()

X_validation_core_n_final = hstack([
    X_validation_core_n_num,
    X_validation_core_n_cat
]).tocsr()

X_test_core_n_final = hstack([
    X_test_core_n_num,
    X_test_core_n_cat
]).tocsr()

y_train_core_n_log = np.log1p(
    y_train_core_n
)

y_validation_core_n_log = np.log1p(
    y_validation_core_n
)

rf_core_neighborhood_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_core_neighborhood_model.fit(
    X_train_core_n_final,
    y_train_core_n_log
)

y_validation_core_n_pred_log = (
    rf_core_neighborhood_model.predict(
        X_validation_core_n_final
    )
)

r2_core_neighborhood_log = r2_score(
    y_validation_core_n_log,
    y_validation_core_n_pred_log
)

) as file:

    )

    )

    )

    )

    )

    )

    )

print(
    f"R2 on log scale: "
    f"{r2_core_neighborhood_log:.6f}"
)


R2 on log scale: 0.491110


In [150]:
# -----------------------------
# 1. انتخاب ویژگی‌ها
# -----------------------------

core_numeric_features_r = [
    "building_size",
    "building_age",
    "credit_value",
    "rooms_count_numeric"
]

core_categorical_features_r = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug"
]

core_features_r = (
    core_numeric_features_r +
    core_categorical_features_r
)


# -----------------------------
# 2. آماده‌سازی داده
# -----------------------------

core_data_r = rent_target_data[
    core_features_r + ["rent_value"]
].copy()

core_data_r["neighborhood_slug"] = (
    core_data_r["neighborhood_slug"]
    .fillna("Unknown")
)


# -----------------------------
# 3. جدا کردن X و y
# -----------------------------

X_r = core_data_r[core_features_r].copy()
y_r = core_data_r["rent_value"].copy()


# -----------------------------
# 4. تقسیم داده
# -----------------------------

X_train_r, X_temp_r, y_train_r, y_temp_r = train_test_split(
    X_r,
    y_r,
    test_size=0.20,
    random_state=42
)

X_validation_r, X_test_r, y_validation_r, y_test_r = train_test_split(
    X_temp_r,
    y_temp_r,
    test_size=0.50,
    random_state=42
)


# -----------------------------
# 5. One-Hot Encoding
# -----------------------------

encoder_r = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_r = encoder_r.fit_transform(
    X_train_r[core_categorical_features_r]
)

X_validation_cat_r = encoder_r.transform(
    X_validation_r[core_categorical_features_r]
)

X_test_cat_r = encoder_r.transform(
    X_test_r[core_categorical_features_r]
)


# -----------------------------
# 6. تبدیل ویژگی‌های عددی
# -----------------------------

X_train_num_r = X_train_r[
    core_numeric_features_r
].values

X_validation_num_r = X_validation_r[
    core_numeric_features_r
].values

X_test_num_r = X_test_r[
    core_numeric_features_r
].values


# -----------------------------
# 7. ترکیب ویژگی‌ها
# -----------------------------

X_train_r_final = hstack([
    X_train_num_r,
    X_train_cat_r
])

X_validation_r_final = hstack([
    X_validation_num_r,
    X_validation_cat_r
])

X_test_r_final = hstack([
    X_test_num_r,
    X_test_cat_r
])


# -----------------------------
# 8. تبدیل target به لگاریتم
# -----------------------------

y_train_log_r = np.log1p(y_train_r)
y_validation_log_r = np.log1p(y_validation_r)


# -----------------------------
# 9. ساخت مدل
# -----------------------------

rf_core_rooms_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


# -----------------------------
# 10. آموزش
# -----------------------------

rf_core_rooms_model.fit(
    X_train_r_final,
    y_train_log_r
)


# -----------------------------
# 11. پیش‌بینی
# -----------------------------

y_validation_pred_log_r = (
    rf_core_rooms_model.predict(
        X_validation_r_final
    )
)


# -----------------------------
# 12. محاسبه R²
# -----------------------------

r2_core_rooms_log = r2_score(
    y_validation_log_r,
    y_validation_pred_log_r
)

print(
    "R² (log scale):",
    r2_core_rooms_log
)


R² (log scale): 0.5024795312113179


In [151]:
# -----------------------------
# 1. انتخاب ویژگی‌ها
# -----------------------------

core_numeric_features_l = [
    "building_size",
    "building_age",
    "credit_value",
    "rooms_count_numeric",
    "luxury_score"
]

core_categorical_features_l = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug"
]

core_features_l = (
    core_numeric_features_l
    + core_categorical_features_l
)


# -----------------------------
# 2. آماده‌سازی داده
# -----------------------------

core_data_l = rent_target_data[
    core_features_l + ["rent_value"]
].copy()

core_data_l["neighborhood_slug"] = (
    core_data_l["neighborhood_slug"]
    .fillna("Unknown")
)


# -----------------------------
# 3. جدا کردن X و y
# -----------------------------

X_core_l = core_data_l[
    core_features_l
].copy()

y_core_l = core_data_l[
    "rent_value"
].copy()


# -----------------------------
# 4. تقسیم داده
# -----------------------------

X_train_core_l, X_temp_core_l, y_train_core_l, y_temp_core_l = (
    train_test_split(
        X_core_l,
        y_core_l,
        test_size=0.20,
        random_state=42
    )
)

X_validation_core_l, X_test_core_l, y_validation_core_l, y_test_core_l = (
    train_test_split(
        X_temp_core_l,
        y_temp_core_l,
        test_size=0.50,
        random_state=42
    )
)


# -----------------------------
# 5. One-Hot Encoding
# -----------------------------

encoder_core_l = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_core_l.fit(
    X_train_core_l[
        core_categorical_features_l
    ]
)

X_train_core_l_cat = encoder_core_l.transform(
    X_train_core_l[
        core_categorical_features_l
    ]
)

X_validation_core_l_cat = encoder_core_l.transform(
    X_validation_core_l[
        core_categorical_features_l
    ]
)

X_test_core_l_cat = encoder_core_l.transform(
    X_test_core_l[
        core_categorical_features_l
    ]
)


# -----------------------------
# 6. ویژگی‌های عددی
# -----------------------------

X_train_core_l_num = csr_matrix(
    X_train_core_l[
        core_numeric_features_l
    ].values
)

X_validation_core_l_num = csr_matrix(
    X_validation_core_l[
        core_numeric_features_l
    ].values
)

X_test_core_l_num = csr_matrix(
    X_test_core_l[
        core_numeric_features_l
    ].values
)


# -----------------------------
# 7. ترکیب ویژگی‌ها
# -----------------------------

X_train_core_l_final = hstack([
    X_train_core_l_num,
    X_train_core_l_cat
]).tocsr()

X_validation_core_l_final = hstack([
    X_validation_core_l_num,
    X_validation_core_l_cat
]).tocsr()

X_test_core_l_final = hstack([
    X_test_core_l_num,
    X_test_core_l_cat
]).tocsr()


# -----------------------------
# 8. تبدیل target به لگاریتم
# -----------------------------

y_train_core_l_log = np.log1p(
    y_train_core_l
)

y_validation_core_l_log = np.log1p(
    y_validation_core_l
)


# -----------------------------
# 9. ساخت مدل
# -----------------------------

rf_core_luxury_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


# -----------------------------
# 10. آموزش
# -----------------------------

rf_core_luxury_model.fit(
    X_train_core_l_final,
    y_train_core_l_log
)


# -----------------------------
# 11. پیش‌بینی
# -----------------------------

y_validation_core_l_pred_log = (
    rf_core_luxury_model.predict(
        X_validation_core_l_final
    )
)


# -----------------------------
# 12. محاسبه R²
# -----------------------------

r2_core_luxury_log = r2_score(
    y_validation_core_l_log,
    y_validation_core_l_pred_log
)

print(
    f"R² (log scale): "
    f"{r2_core_luxury_log:.6f}"
)


R² (log scale): 0.502480


## 19. مقایسه Decision Tree و XGBoost

اجرای Decision Tree و XGBoost روی مجموعه Core.


In [152]:
# -----------------------------
# 1. ساخت مدل Decision Tree
# -----------------------------

decision_tree_model = DecisionTreeRegressor(
    max_depth=20,
    min_samples_leaf=5,
    random_state=42
)


# -----------------------------
# 2. آموزش مدل
# -----------------------------

decision_tree_model.fit(
    X_train_core_n_final,
    y_train_core_n_log
)


# -----------------------------
# 3. پیش‌بینی روی Validation
# -----------------------------

y_validation_dt_pred_log = (
    decision_tree_model.predict(
        X_validation_core_n_final
    )
)


# -----------------------------
# 4. محاسبه R²
# -----------------------------

r2_decision_tree_log = r2_score(
    y_validation_core_n_log,
    y_validation_dt_pred_log
)

print(
    f"Decision Tree R² (log scale): "
    f"{r2_decision_tree_log:.6f}"
)


Decision Tree R² (log scale): 0.423577


In [154]:
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# -----------------------------
# 2. آموزش مدل
# -----------------------------

xgb_model.fit(
    X_train_core_n_final,
    y_train_core_n_log
)


# -----------------------------
# 3. پیش‌بینی روی Validation
# -----------------------------

y_validation_xgb_pred_log = (
    xgb_model.predict(
        X_validation_core_n_final
    )
)


# -----------------------------
# 4. محاسبه R²
# -----------------------------

r2_xgb_log = r2_score(
    y_validation_core_n_log,
    y_validation_xgb_pred_log
)

print(
    f"XGBoost R² (log scale): "
    f"{r2_xgb_log:.6f}"
)


XGBoost R² (log scale): 0.498487


## 20. تنظیم اولیه و Fine-Tuning مدل XGBoost

تنظیم مرحله‌ای max_depth، learning_rate، n_estimators، min_child_weight، gamma، subsample و colsample_bytree.


In [161]:
# 1. محدوده پارامترها


param_grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [300, 500, 800]
}

results = []

best_r2 = -np.inf
best_params = None
best_model = None


# -----------------------------
# 2. اجرای ترکیب پارامترها
# -----------------------------

for max_depth, learning_rate, n_estimators in product(
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["n_estimators"]
):

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_core_n_final,
        y_train_core_n_log
    )

    y_pred = model.predict(
        X_validation_core_n_final
    )

    r2 = r2_score(
        y_validation_core_n_log,
        y_pred
    )

    results.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "r2": r2
    })

    if r2 > best_r2:
        best_r2 = r2
        best_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators
        }
        best_model = model


# -----------------------------
# 3. نمایش بهترین نتیجه
# -----------------------------

print("=" * 70)
print("Best XGBoost Parameters")
print("=" * 70)

print(
    f"max_depth: "
    f"{best_params['max_depth']}"
)

print(
    f"learning_rate: "
    f"{best_params['learning_rate']}"
)

print(
    f"n_estimators: "
    f"{best_params['n_estimators']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2:.6f}"
)


# -----------------------------
# 4. ذخیره نتایج تمام ترکیب‌ها
# -----------------------------

results_sorted = sorted(
    results,
    key=lambda x: x["r2"],
    reverse=True
)

) as file:

    )

    )

    )

    )

    )

    for result in results_sorted:
        )


Best XGBoost Parameters
max_depth: 8
learning_rate: 0.1
n_estimators: 800
Best R² (log scale): 0.547289


In [162]:

# 1. محدوده پارامترها


param_grid_fine = {
    "max_depth": [7, 8, 9, 10],
    "learning_rate": [0.05, 0.08, 0.1],
    "n_estimators": [800, 1000, 1200]
}


results_fine = []

best_r2_fine = -np.inf
best_params_fine = None
best_model_fine = None


# -----------------------------
# 2. اجرای ترکیب پارامترها
# -----------------------------

for max_depth, learning_rate, n_estimators in product(
    param_grid_fine["max_depth"],
    param_grid_fine["learning_rate"],
    param_grid_fine["n_estimators"]
):

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_core_n_final,
        y_train_core_n_log
    )

    y_pred = model.predict(
        X_validation_core_n_final
    )

    r2 = r2_score(
        y_validation_core_n_log,
        y_pred
    )

    results_fine.append({
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "r2": r2
    })

    if r2 > best_r2_fine:
        best_r2_fine = r2

        best_params_fine = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators
        }

        best_model_fine = model


# -----------------------------
# 3. نمایش بهترین نتیجه
# -----------------------------

print("=" * 70)
print("Best Fine-Tuned XGBoost Parameters")
print("=" * 70)

print(
    f"max_depth: "
    f"{best_params_fine['max_depth']}"
)

print(
    f"learning_rate: "
    f"{best_params_fine['learning_rate']}"
)

print(
    f"n_estimators: "
    f"{best_params_fine['n_estimators']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_fine:.6f}"
)


# -----------------------------
# 4. ذخیره نتایج
# -----------------------------

results_fine_sorted = sorted(
    results_fine,
    key=lambda x: x["r2"],
    reverse=True
)

) as file:

    )

    )

    )

    )

    )

    for result in results_fine_sorted:

        )


Best Fine-Tuned XGBoost Parameters
max_depth: 10
learning_rate: 0.1
n_estimators: 1200
Best R² (log scale): 0.557051


In [163]:
# 1. محدوده پارامترها
param_grid_final = {
    "max_depth": [10, 12, 14],
    "n_estimators": [1200, 1500, 1800]
}

learning_rate_final = 0.1

results_final = []

best_r2_final = -np.inf
best_params_final = None
best_model_final = None


# -----------------------------
# 2. اجرای ترکیب پارامترها
# -----------------------------

for max_depth, n_estimators in product(
    param_grid_final["max_depth"],
    param_grid_final["n_estimators"]
):

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate_final,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_core_n_final,
        y_train_core_n_log
    )

    y_pred = model.predict(
        X_validation_core_n_final
    )

    r2 = r2_score(
        y_validation_core_n_log,
        y_pred
    )

    results_final.append({
        "max_depth": max_depth,
        "n_estimators": n_estimators,
        "learning_rate": learning_rate_final,
        "r2": r2
    })

    if r2 > best_r2_final:
        best_r2_final = r2

        best_params_final = {
            "max_depth": max_depth,
            "n_estimators": n_estimators,
            "learning_rate": learning_rate_final
        }

        best_model_final = model


# -----------------------------
# 3. نمایش بهترین نتیجه
# -----------------------------

print("=" * 70)
print("Best Final XGBoost Parameters")
print("=" * 70)

print(
    f"max_depth: "
    f"{best_params_final['max_depth']}"
)

print(
    f"learning_rate: "
    f"{best_params_final['learning_rate']}"
)

print(
    f"n_estimators: "
    f"{best_params_final['n_estimators']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_final:.6f}"
)


# -----------------------------
# 4. ذخیره نتایج
# -----------------------------

results_final_sorted = sorted(
    results_final,
    key=lambda x: x["r2"],
    reverse=True
)

) as file:

    )

    )

    )

    )

    )

    for result in results_final_sorted:

        )


Best Final XGBoost Parameters
max_depth: 10
learning_rate: 0.1
n_estimators: 1500
Best R² (log scale): 0.557524


In [165]:

y_test_core_n_log = np.log1p(y_test_core_n)

print(f"Test samples: {len(y_test_core_n_log)}")


Test samples: 28962


In [166]:
# Build final XGBoost model with the best validation parameters
final_xgb_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# Train on the training data
final_xgb_model.fit(
    X_train_core_n_final,
    y_train_core_n_log
)


# Predict on the test data
y_test_pred_log = final_xgb_model.predict(
    X_test_core_n_final
)


# Calculate R² on log scale
r2_test_log = r2_score(
    y_test_core_n_log,
    y_test_pred_log
)


print("=" * 70)
print("Final XGBoost Test Evaluation")
print("=" * 70)
print(f"R² (log scale): {r2_test_log:.6f}")


# Save result
) as file:
    )


Final XGBoost Test Evaluation
R² (log scale): 0.551850


## 21. انتخاب ویژگی‌های مؤثر

آزمایش اضافه کردن ویژگی‌ها به مدل و نگه‌داشتن فقط ویژگی‌هایی که روی Validation R² اثر مثبت دارند.


In [167]:
# Features
numeric_features_elevator = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_elevator = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator"
]

features_elevator = (
    numeric_features_elevator +
    categorical_features_elevator
)


# Prepare data
X_elevator = rent_target_data[features_elevator].copy()
X_elevator["neighborhood_slug"] = (
    X_elevator["neighborhood_slug"]
    .fillna("Unknown")
)


y_elevator = rent_target_data["rent_value"].copy()


# Same train/validation/test split
X_train_elevator, X_temp_elevator, y_train_elevator, y_temp_elevator = train_test_split(
    X_elevator,
    y_elevator,
    test_size=0.20,
    random_state=42
)

X_validation_elevator, X_test_elevator, y_validation_elevator, y_test_elevator = train_test_split(
    X_temp_elevator,
    y_temp_elevator,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_elevator = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_elevator = encoder_elevator.fit_transform(
    X_train_elevator[categorical_features_elevator]
)

X_validation_cat_elevator = encoder_elevator.transform(
    X_validation_elevator[categorical_features_elevator]
)


# Numeric features
X_train_num_elevator = X_train_elevator[
    numeric_features_elevator
].to_numpy()

X_validation_num_elevator = X_validation_elevator[
    numeric_features_elevator
].to_numpy()


# Combine numeric + categorical
from scipy.sparse import hstack

X_train_elevator_final = hstack([
    X_train_num_elevator,
    X_train_cat_elevator
])

X_validation_elevator_final = hstack([
    X_validation_num_elevator,
    X_validation_cat_elevator
])


# Log-transform target
y_train_elevator_log = np.log1p(
    y_train_elevator
)

y_validation_elevator_log = np.log1p(
    y_validation_elevator
)


# XGBoost
xgb_elevator_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


xgb_elevator_model.fit(
    X_train_elevator_final,
    y_train_elevator_log
)


# Validation prediction
y_validation_elevator_pred = (
    xgb_elevator_model.predict(
        X_validation_elevator_final
    )
)


# Validation R²
r2_elevator = r2_score(
    y_validation_elevator_log,
    y_validation_elevator_pred
)


print("=" * 70)
print("XGBoost - Adding has_elevator")
print("=" * 70)
print(f"R² (log scale): {r2_elevator:.6f}")


XGBoost - Adding has_elevator
R² (log scale): 0.573963


In [168]:
# Features
numeric_features_parking = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_parking = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking"
]

features_parking = (
    numeric_features_parking +
    categorical_features_parking
)


# Prepare data
X_parking = rent_target_data[features_parking].copy()

X_parking["neighborhood_slug"] = (
    X_parking["neighborhood_slug"]
    .fillna("Unknown")
)

y_parking = rent_target_data["rent_value"].copy()


# Same split
X_train_parking, X_temp_parking, y_train_parking, y_temp_parking = train_test_split(
    X_parking,
    y_parking,
    test_size=0.20,
    random_state=42
)

X_validation_parking, X_test_parking, y_validation_parking, y_test_parking = train_test_split(
    X_temp_parking,
    y_temp_parking,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_parking = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_parking = encoder_parking.fit_transform(
    X_train_parking[categorical_features_parking]
)

X_validation_cat_parking = encoder_parking.transform(
    X_validation_parking[categorical_features_parking]
)


# Numeric features
X_train_num_parking = X_train_parking[
    numeric_features_parking
].to_numpy()

X_validation_num_parking = X_validation_parking[
    numeric_features_parking
].to_numpy()


# Combine
X_train_parking_final = hstack([
    X_train_num_parking,
    X_train_cat_parking
])

X_validation_parking_final = hstack([
    X_validation_num_parking,
    X_validation_cat_parking
])


# Log target
y_train_parking_log = np.log1p(
    y_train_parking
)

y_validation_parking_log = np.log1p(
    y_validation_parking
)


# XGBoost
xgb_parking_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_parking_model.fit(
    X_train_parking_final,
    y_train_parking_log
)


# Validation prediction
y_validation_parking_pred = (
    xgb_parking_model.predict(
        X_validation_parking_final
    )
)


# R²
r2_parking = r2_score(
    y_validation_parking_log,
    y_validation_parking_pred
)


print("=" * 70)
print("XGBoost - Adding has_parking")
print("=" * 70)
print(f"R² (log scale): {r2_parking:.6f}")


XGBoost - Adding has_parking
R² (log scale): 0.579508


In [169]:
# Features
numeric_features_warehouse = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_warehouse = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse"
]

features_warehouse = (
    numeric_features_warehouse +
    categorical_features_warehouse
)


# Prepare data
X_warehouse = rent_target_data[features_warehouse].copy()

X_warehouse["neighborhood_slug"] = (
    X_warehouse["neighborhood_slug"]
    .fillna("Unknown")
)

y_warehouse = rent_target_data["rent_value"].copy()


# Same split
X_train_warehouse, X_temp_warehouse, y_train_warehouse, y_temp_warehouse = train_test_split(
    X_warehouse,
    y_warehouse,
    test_size=0.20,
    random_state=42
)

X_validation_warehouse, X_test_warehouse, y_validation_warehouse, y_test_warehouse = train_test_split(
    X_temp_warehouse,
    y_temp_warehouse,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_warehouse = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_warehouse = encoder_warehouse.fit_transform(
    X_train_warehouse[categorical_features_warehouse]
)

X_validation_cat_warehouse = encoder_warehouse.transform(
    X_validation_warehouse[categorical_features_warehouse]
)


# Numeric features
X_train_num_warehouse = X_train_warehouse[
    numeric_features_warehouse
].to_numpy()

X_validation_num_warehouse = X_validation_warehouse[
    numeric_features_warehouse
].to_numpy()


# Combine numeric + categorical
X_train_warehouse_final = hstack([
    X_train_num_warehouse,
    X_train_cat_warehouse
])

X_validation_warehouse_final = hstack([
    X_validation_num_warehouse,
    X_validation_cat_warehouse
])


# Log target
y_train_warehouse_log = np.log1p(
    y_train_warehouse
)

y_validation_warehouse_log = np.log1p(
    y_validation_warehouse
)


# XGBoost
xgb_warehouse_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_warehouse_model.fit(
    X_train_warehouse_final,
    y_train_warehouse_log
)


# Validation prediction
y_validation_warehouse_pred = (
    xgb_warehouse_model.predict(
        X_validation_warehouse_final
    )
)


# R²
r2_warehouse = r2_score(
    y_validation_warehouse_log,
    y_validation_warehouse_pred
)


print("=" * 70)
print("XGBoost - Adding has_warehouse")
print("=" * 70)
print(f"R² (log scale): {r2_warehouse:.6f}")


XGBoost - Adding has_warehouse
R² (log scale): 0.579638


In [170]:
# Features
numeric_features_heating = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_heating = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system"
]

features_heating = (
    numeric_features_heating +
    categorical_features_heating
)


# Prepare data
X_heating = rent_target_data[features_heating].copy()

X_heating["neighborhood_slug"] = (
    X_heating["neighborhood_slug"]
    .fillna("Unknown")
)

X_heating["has_heating_system"] = (
    X_heating["has_heating_system"]
    .fillna("Unknown")
)

y_heating = rent_target_data["rent_value"].copy()


# Same split
X_train_heating, X_temp_heating, y_train_heating, y_temp_heating = train_test_split(
    X_heating,
    y_heating,
    test_size=0.20,
    random_state=42
)

X_validation_heating, X_test_heating, y_validation_heating, y_test_heating = train_test_split(
    X_temp_heating,
    y_temp_heating,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_heating = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_heating = encoder_heating.fit_transform(
    X_train_heating[categorical_features_heating]
)

X_validation_cat_heating = encoder_heating.transform(
    X_validation_heating[categorical_features_heating]
)


# Numeric features
X_train_num_heating = X_train_heating[
    numeric_features_heating
].to_numpy()

X_validation_num_heating = X_validation_heating[
    numeric_features_heating
].to_numpy()


# Combine numeric + categorical
X_train_heating_final = hstack([
    X_train_num_heating,
    X_train_cat_heating
])

X_validation_heating_final = hstack([
    X_validation_num_heating,
    X_validation_cat_heating
])


# Log target
y_train_heating_log = np.log1p(
    y_train_heating
)

y_validation_heating_log = np.log1p(
    y_validation_heating
)


# XGBoost
xgb_heating_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_heating_model.fit(
    X_train_heating_final,
    y_train_heating_log
)


# Validation prediction
y_validation_heating_pred = (
    xgb_heating_model.predict(
        X_validation_heating_final
    )
)


# R²
r2_heating = r2_score(
    y_validation_heating_log,
    y_validation_heating_pred
)


print("=" * 70)
print("XGBoost - Adding has_heating_system")
print("=" * 70)
print(f"R² (log scale): {r2_heating:.6f}")


XGBoost - Adding has_heating_system
R² (log scale): 0.580988


In [172]:
# Features
numeric_features_cat3 = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_cat3 = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]

features_cat3 = (
    numeric_features_cat3 +
    categorical_features_cat3
)


# Prepare data
X_cat3 = rent_target_data[features_cat3].copy()

X_cat3["neighborhood_slug"] = (
    X_cat3["neighborhood_slug"]
    .fillna("Unknown")
)

X_cat3["has_heating_system"] = (
    X_cat3["has_heating_system"]
    .fillna("Unknown")
)

X_cat3["cat3_slug"] = (
    X_cat3["cat3_slug"]
    .fillna("Unknown")
)

y_cat3 = rent_target_data["rent_value"].copy()


# Same split
X_train_cat3, X_temp_cat3, y_train_cat3, y_temp_cat3 = train_test_split(
    X_cat3,
    y_cat3,
    test_size=0.20,
    random_state=42
)

X_validation_cat3, X_test_cat3, y_validation_cat3, y_test_cat3 = train_test_split(
    X_temp_cat3,
    y_temp_cat3,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_cat3 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_cat3 = encoder_cat3.fit_transform(
    X_train_cat3[categorical_features_cat3]
)

X_validation_cat_cat3 = encoder_cat3.transform(
    X_validation_cat3[categorical_features_cat3]
)


# Numeric features
X_train_num_cat3 = X_train_cat3[
    numeric_features_cat3
].to_numpy()

X_validation_num_cat3 = X_validation_cat3[
    numeric_features_cat3
].to_numpy()


# Combine numeric + categorical
X_train_cat3_final = hstack([
    X_train_num_cat3,
    X_train_cat_cat3
])

X_validation_cat3_final = hstack([
    X_validation_num_cat3,
    X_validation_cat_cat3
])


# Log target
y_train_cat3_log = np.log1p(
    y_train_cat3
)

y_validation_cat3_log = np.log1p(
    y_validation_cat3
)


# XGBoost
xgb_cat3_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_cat3_model.fit(
    X_train_cat3_final,
    y_train_cat3_log
)


# Validation prediction
y_validation_cat3_pred = (
    xgb_cat3_model.predict(
        X_validation_cat3_final
    )
)


# R²
r2_cat3 = r2_score(
    y_validation_cat3_log,
    y_validation_cat3_pred
)


print("=" * 70)
print("XGBoost - Adding cat3_slug")
print("=" * 70)
print(f"R² (log scale): {r2_cat3:.6f}")


XGBoost - Adding cat3_slug
R² (log scale): 0.584328


In [173]:
# Features
numeric_features_city_type = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric"
]

categorical_features_city_type = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "city_type"
]

features_city_type = (
    numeric_features_city_type +
    categorical_features_city_type
)


# Prepare data
X_city_type = rent_target_data[features_city_type].copy()

X_city_type["neighborhood_slug"] = (
    X_city_type["neighborhood_slug"]
    .fillna("Unknown")
)

X_city_type["has_heating_system"] = (
    X_city_type["has_heating_system"]
    .fillna("Unknown")
)

X_city_type["cat3_slug"] = (
    X_city_type["cat3_slug"]
    .fillna("Unknown")
)

X_city_type["city_type"] = (
    X_city_type["city_type"]
    .fillna("شهر کوچک")
)

y_city_type = rent_target_data["rent_value"].copy()


# Same split
X_train_city_type, X_temp_city_type, y_train_city_type, y_temp_city_type = train_test_split(
    X_city_type,
    y_city_type,
    test_size=0.20,
    random_state=42
)

X_validation_city_type, X_test_city_type, y_validation_city_type, y_test_city_type = train_test_split(
    X_temp_city_type,
    y_temp_city_type,
    test_size=0.50,
    random_state=42
)


# One-hot encoding
encoder_city_type = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_city_type = encoder_city_type.fit_transform(
    X_train_city_type[categorical_features_city_type]
)

X_validation_cat_city_type = encoder_city_type.transform(
    X_validation_city_type[categorical_features_city_type]
)


# Numeric features
X_train_num_city_type = X_train_city_type[
    numeric_features_city_type
].to_numpy()

X_validation_num_city_type = X_validation_city_type[
    numeric_features_city_type
].to_numpy()


# Combine numeric + categorical
X_train_city_type_final = hstack([
    X_train_num_city_type,
    X_train_cat_city_type
])

X_validation_city_type_final = hstack([
    X_validation_num_city_type,
    X_validation_cat_city_type
])


# Log target
y_train_city_type_log = np.log1p(
    y_train_city_type
)

y_validation_city_type_log = np.log1p(
    y_validation_city_type
)


# XGBoost
xgb_city_type_model = XGBRegressor(
    n_estimators=1500,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_city_type_model.fit(
    X_train_city_type_final,
    y_train_city_type_log
)


# Validation prediction
y_validation_city_type_pred = (
    xgb_city_type_model.predict(
        X_validation_city_type_final
    )
)


# R²
r2_city_type = r2_score(
    y_validation_city_type_log,
    y_validation_city_type_pred
)


print("=" * 70)
print("XGBoost - Adding city_type")
print("=" * 70)
print(f"R² (log scale): {r2_city_type:.6f}")


XGBoost - Adding city_type
R² (log scale): 0.584264


## 22. تنظیم نهایی پارامترهای XGBoost

Fine-tuning نهایی بر اساس Validation R².


In [174]:
min_child_weight_values = [1, 3, 5, 10]

results_min_child = []

best_r2_min_child = -np.inf
best_min_child_weight = None
best_model_min_child = None


for min_child_weight in min_child_weight_values:

    model = XGBRegressor(
        n_estimators=1500,
        max_depth=10,
        learning_rate=0.1,
        min_child_weight=min_child_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_min_child.append({
        "min_child_weight": min_child_weight,
        "r2": r2
    })

    if r2 > best_r2_min_child:
        best_r2_min_child = r2
        best_min_child_weight = min_child_weight
        best_model_min_child = model


print("=" * 70)
print("XGBoost - min_child_weight Tuning")
print("=" * 70)

print(
    f"Best min_child_weight: "
    f"{best_min_child_weight}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_min_child:.6f}"
)


print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_min_child,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"min_child_weight="
        f"{result['min_child_weight']}, "
        f"R²={result['r2']:.6f}"
    )


# Save results
) as file:

    )

    )

    )

    for result in sorted(
        results_min_child,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - min_child_weight Tuning
Best min_child_weight: 5
Best R² (log scale): 0.587830

All Results:
----------------------------------------------------------------------
min_child_weight=5, R²=0.587830
min_child_weight=10, R²=0.587515
min_child_weight=3, R²=0.585415
min_child_weight=1, R²=0.584328


In [175]:
gamma_values = [0, 0.01, 0.05, 0.1, 0.2]

results_gamma = []

best_r2_gamma = -np.inf
best_gamma = None
best_model_gamma = None


for gamma in gamma_values:

    model = XGBRegressor(
        n_estimators=1500,
        max_depth=10,
        learning_rate=0.1,
        min_child_weight=5,
        gamma=gamma,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_gamma.append({
        "gamma": gamma,
        "r2": r2
    })

    if r2 > best_r2_gamma:
        best_r2_gamma = r2
        best_gamma = gamma
        best_model_gamma = model


print("=" * 70)
print("XGBoost - gamma Tuning")
print("=" * 70)

print(f"Best gamma: {best_gamma}")
print(f"Best R² (log scale): {best_r2_gamma:.6f}")


print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_gamma,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"gamma={result['gamma']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )

    )

    for result in sorted(
        results_gamma,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - gamma Tuning
Best gamma: 0
Best R² (log scale): 0.587830

All Results:
----------------------------------------------------------------------
gamma=0, R²=0.587830
gamma=0.1, R²=0.587618
gamma=0.2, R²=0.587494
gamma=0.05, R²=0.587142
gamma=0.01, R²=0.586776


In [176]:
subsample_values = [0.6, 0.8, 1.0]

results_subsample = []

best_r2_subsample = -float("inf")
best_subsample = None
best_model_subsample = None


for subsample in subsample_values:

    model = XGBRegressor(
        n_estimators=1500,
        max_depth=10,
        learning_rate=0.1,
        min_child_weight=5,
        gamma=0,
        subsample=subsample,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_subsample.append({
        "subsample": subsample,
        "r2": r2
    })

    if r2 > best_r2_subsample:
        best_r2_subsample = r2
        best_subsample = subsample
        best_model_subsample = model


print("=" * 70)
print("XGBoost - subsample Tuning")
print("=" * 70)

print(f"Best subsample: {best_subsample}")
print(
    f"Best R² (log scale): "
    f"{best_r2_subsample:.6f}"
)


print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_subsample,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"subsample={result['subsample']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )

    )

    for result in sorted(
        results_subsample,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - subsample Tuning
Best subsample: 0.8
Best R² (log scale): 0.587830

All Results:
----------------------------------------------------------------------
subsample=0.8, R²=0.587830
subsample=0.6, R²=0.585572
subsample=1.0, R²=0.583561


In [177]:
colsample_values = [0.6, 0.8, 1.0]

results_colsample = []

best_r2_colsample = -float("inf")
best_colsample = None
best_model_colsample = None


for colsample_bytree in colsample_values:

    model = XGBRegressor(
        n_estimators=1500,
        max_depth=10,
        learning_rate=0.1,
        min_child_weight=5,
        gamma=0,
        subsample=0.8,
        colsample_bytree=colsample_bytree,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_colsample.append({
        "colsample_bytree": colsample_bytree,
        "r2": r2
    })

    if r2 > best_r2_colsample:
        best_r2_colsample = r2
        best_colsample = colsample_bytree
        best_model_colsample = model


print("=" * 70)
print("XGBoost - colsample_bytree Tuning")
print("=" * 70)

print(
    f"Best colsample_bytree: "
    f"{best_colsample}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_colsample:.6f}"
)


print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_colsample,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"colsample_bytree="
        f"{result['colsample_bytree']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )

    )

    )

    for result in sorted(
        results_colsample,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - colsample_bytree Tuning
Best colsample_bytree: 0.8
Best R² (log scale): 0.587830

All Results:
----------------------------------------------------------------------
colsample_bytree=0.8, R²=0.587830
colsample_bytree=1.0, R²=0.586343
colsample_bytree=0.6, R²=0.585782


In [178]:
max_depth_values = [8, 10, 12, 14]
min_child_weight_values = [3, 5, 7]

results_depth_child = []

best_r2_depth_child = -float("inf")
best_params_depth_child = None
best_model_depth_child = None


for max_depth, min_child_weight in product(
    max_depth_values,
    min_child_weight_values
):

    model = XGBRegressor(
        n_estimators=1500,
        max_depth=max_depth,
        learning_rate=0.1,
        min_child_weight=min_child_weight,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_depth_child.append({
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "r2": r2
    })

    if r2 > best_r2_depth_child:
        best_r2_depth_child = r2

        best_params_depth_child = {
            "max_depth": max_depth,
            "min_child_weight": min_child_weight
        }

        best_model_depth_child = model


print("=" * 70)
print("XGBoost - max_depth × min_child_weight Tuning")
print("=" * 70)

print(
    f"Best max_depth: "
    f"{best_params_depth_child['max_depth']}"
)

print(
    f"Best min_child_weight: "
    f"{best_params_depth_child['min_child_weight']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_depth_child:.6f}"
)


print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_depth_child,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"max_depth={result['max_depth']}, "
        f"min_child_weight={result['min_child_weight']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )

    )

    )

    for result in sorted(
        results_depth_child,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - max_depth × min_child_weight Tuning
Best max_depth: 10
Best min_child_weight: 5
Best R² (log scale): 0.587830

All Results:
----------------------------------------------------------------------
max_depth=10, min_child_weight=5, R²=0.587830
max_depth=10, min_child_weight=7, R²=0.587615
max_depth=8, min_child_weight=7, R²=0.586995
max_depth=12, min_child_weight=7, R²=0.586190
max_depth=12, min_child_weight=3, R²=0.585812
max_depth=10, min_child_weight=3, R²=0.585415
max_depth=8, min_child_weight=5, R²=0.584942
max_depth=8, min_child_weight=3, R²=0.584478
max_depth=12, min_child_weight=5, R²=0.584272
max_depth=14, min_child_weight=7, R²=0.582866
max_depth=14, min_child_weight=3, R²=0.580659
max_depth=14, min_child_weight=5, R²=0.579619


In [179]:
learning_rate_values = [0.03, 0.05, 0.08, 0.1]
n_estimators_values = [1000, 1500, 2000]

results_lr_estimators = []

best_r2_lr_estimators = -float("inf")
best_params_lr_estimators = None
best_model_lr_estimators = None

for learning_rate, n_estimators in product(
    learning_rate_values,
    n_estimators_values
):
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=10,
        learning_rate=learning_rate,
        min_child_weight=5,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_lr_estimators.append({
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "r2": r2
    })

    if r2 > best_r2_lr_estimators:
        best_r2_lr_estimators = r2
        best_params_lr_estimators = {
            "learning_rate": learning_rate,
            "n_estimators": n_estimators
        }
        best_model_lr_estimators = model


print("=" * 70)
print("XGBoost - learning_rate × n_estimators Tuning")
print("=" * 70)

print(
    f"Best learning_rate: "
    f"{best_params_lr_estimators['learning_rate']}"
)

print(
    f"Best n_estimators: "
    f"{best_params_lr_estimators['n_estimators']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_lr_estimators:.6f}"
)

print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_lr_estimators,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"learning_rate={result['learning_rate']}, "
        f"n_estimators={result['n_estimators']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:
    )
    )

    )

    for result in sorted(
        results_lr_estimators,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - learning_rate × n_estimators Tuning
Best learning_rate: 0.05
Best n_estimators: 2000
Best R² (log scale): 0.589134

All Results:
----------------------------------------------------------------------
learning_rate=0.05, n_estimators=2000, R²=0.589134
learning_rate=0.08, n_estimators=2000, R²=0.588236
learning_rate=0.08, n_estimators=1500, R²=0.587884
learning_rate=0.1, n_estimators=1500, R²=0.587830
learning_rate=0.1, n_estimators=1000, R²=0.587364
learning_rate=0.1, n_estimators=2000, R²=0.586567
learning_rate=0.05, n_estimators=1500, R²=0.586499
learning_rate=0.08, n_estimators=1000, R²=0.585161
learning_rate=0.03, n_estimators=2000, R²=0.583729
learning_rate=0.05, n_estimators=1000, R²=0.579857
learning_rate=0.03, n_estimators=1500, R²=0.578826
learning_rate=0.03, n_estimators=1000, R²=0.570721


In [180]:
learning_rate_values = [0.04, 0.05, 0.06]
n_estimators_values = [1800, 2000, 2200, 2500]

results_lr_estimators_fine = []

best_r2_lr_estimators_fine = -float("inf")
best_params_lr_estimators_fine = None
best_model_lr_estimators_fine = None

for learning_rate, n_estimators in product(
    learning_rate_values,
    n_estimators_values
):
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=10,
        learning_rate=learning_rate,
        min_child_weight=5,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_lr_estimators_fine.append({
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "r2": r2
    })

    if r2 > best_r2_lr_estimators_fine:
        best_r2_lr_estimators_fine = r2
        best_params_lr_estimators_fine = {
            "learning_rate": learning_rate,
            "n_estimators": n_estimators
        }
        best_model_lr_estimators_fine = model


print("=" * 70)
print("XGBoost - Fine Tuning: learning_rate × n_estimators")
print("=" * 70)

print(
    f"Best learning_rate: "
    f"{best_params_lr_estimators_fine['learning_rate']}"
)

print(
    f"Best n_estimators: "
    f"{best_params_lr_estimators_fine['n_estimators']}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_lr_estimators_fine:.6f}"
)

print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_lr_estimators_fine,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"learning_rate={result['learning_rate']}, "
        f"n_estimators={result['n_estimators']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )
    )

    )

    for result in sorted(
        results_lr_estimators_fine,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


XGBoost - Fine Tuning: learning_rate × n_estimators
Best learning_rate: 0.05
Best n_estimators: 2500
Best R² (log scale): 0.590598

All Results:
----------------------------------------------------------------------
learning_rate=0.05, n_estimators=2500, R²=0.590598
learning_rate=0.05, n_estimators=2200, R²=0.589984
learning_rate=0.05, n_estimators=2000, R²=0.589134
learning_rate=0.06, n_estimators=2500, R²=0.589066
learning_rate=0.04, n_estimators=2500, R²=0.588910
learning_rate=0.06, n_estimators=2200, R²=0.588661
learning_rate=0.05, n_estimators=1800, R²=0.588316
learning_rate=0.06, n_estimators=2000, R²=0.588281
learning_rate=0.06, n_estimators=1800, R²=0.587797
learning_rate=0.04, n_estimators=2200, R²=0.587756
learning_rate=0.04, n_estimators=2000, R²=0.586776
learning_rate=0.04, n_estimators=1800, R²=0.585276


In [102]:
n_estimators_values = [2500, 2800, 3000, 3500]

results_n_estimators = []

best_r2_n_estimators = -float("inf")
best_n_estimators = None
best_model_n_estimators = None

for n_estimators in n_estimators_values:

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=10,
        learning_rate=0.05,
        min_child_weight=5,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_cat3_final,
        y_train_cat3_log
    )

    y_validation_pred = model.predict(
        X_validation_cat3_final
    )

    r2 = r2_score(
        y_validation_cat3_log,
        y_validation_pred
    )

    results_n_estimators.append({
        "n_estimators": n_estimators,
        "r2": r2
    })

    if r2 > best_r2_n_estimators:
        best_r2_n_estimators = r2
        best_n_estimators = n_estimators
        best_model_n_estimators = model


print("=" * 70)
print("XGBoost - Fine Tuning: n_estimators")
print("=" * 70)

print(
    f"Best n_estimators: "
    f"{best_n_estimators}"
)

print(
    f"Best R² (log scale): "
    f"{best_r2_n_estimators:.6f}"
)

print("\nAll Results:")
print("-" * 70)

for result in sorted(
    results_n_estimators,
    key=lambda x: x["r2"],
    reverse=True
):
    print(
        f"n_estimators={result['n_estimators']}, "
        f"R²={result['r2']:.6f}"
    )


) as file:

    )
    )

    )

    for result in sorted(
        results_n_estimators,
        key=lambda x: x["r2"],
        reverse=True
    ):
        )


NameError: name 'X_train_cat3_final' is not defined

In [188]:

credit_value = rent_target_data["credit_value"]

output = []

output.append("=" * 70)
output.append("Credit Value Distribution Check")
output.append("=" * 70)

output.append(
    f"\nDtype: {credit_value.dtype}"
)

output.append(
    f"Missing: {credit_value.isna().sum()}"
)

output.append(
    f"Missing percentage: "
    f"{credit_value.isna().mean() * 100:.2f}%"
)

output.append(
    f"\nMinimum: {credit_value.min()}"
)

output.append(
    f"Maximum: {credit_value.max()}"
)

output.append("\nDescriptive statistics:")
output.append(
    credit_value.describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    ).to_string()
)

# Number of zero values
zero_count = (credit_value == 0).sum()

output.append(
    f"\nZero values: {zero_count}"
)

output.append(
    f"Zero percentage: "
    f"{zero_count / len(credit_value) * 100:.2f}%"
)

# Number of negative values
negative_count = (credit_value < 0).sum()

output.append(
    f"\nNegative values: {negative_count}"
)

# Log-transformed statistics
positive_credit = credit_value[
    credit_value >= 0
]

log_credit = np.log1p(
    positive_credit
)

output.append(
    "\nLog1p(Credit Value) statistics:"
)

output.append(
    log_credit.describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    ).to_string()
)

result = "\n".join(output)

print(result)

) as file:


Credit Value Distribution Check

Dtype: float64
Missing: 0
Missing percentage: 0.00%

Minimum: 0.0
Maximum: 99999999999.0

Descriptive statistics:
count    2.896170e+05
mean     3.931270e+08
std      1.105473e+09
min      0.000000e+00
1%       1.000000e+06
5%       1.000000e+07
10%      3.000000e+07
25%      1.000000e+08
50%      2.000000e+08
75%      4.000000e+08
90%      8.000000e+08
95%      1.200000e+09
99%      3.000000e+09
99.9%    1.111111e+10
max      1.000000e+11

Zero values: 2801
Zero percentage: 0.97%

Negative values: 0

Log1p(Credit Value) statistics:
count    289617.000000
mean         18.801680
std           2.319403
min           0.000000
1%           13.815512
5%           16.118096
10%          17.216708
25%          18.420681
50%          19.113828
75%          19.806975
90%          20.500122
95%          20.905587
99%          21.821878
99.9%        23.131211
max          25.328436


In [101]:
# Create credit value per square meter
rent_target_data["credit_value_per_size"] = (
    rent_target_data["credit_value"] /
    rent_target_data["building_size"].replace(0, np.nan)
)


# Check the result
print("=" * 70)
print("credit_value_per_size Check")
print("=" * 70)

print(
    "\nMissing:",
    rent_target_data["credit_value_per_size"].isna().sum()
)

print(
    "Infinite:",
    np.isinf(
        rent_target_data["credit_value_per_size"]
    ).sum()
)

print(
    "\nDescriptive statistics:"
)

print(
    rent_target_data["credit_value_per_size"].describe()
)


NameError: name 'rent_target_data' is not defined

In [100]:
# Add the new feature to the current 12-feature set
features_with_credit_per_size = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size"
]


X_new = rent_target_data[
    features_with_credit_per_size
].copy()


# Use the same train/validation indices as before
X_train_new = X_new.loc[X_train_final.index]
X_validation_new = X_new.loc[X_validation_final.index]


random_forest_model = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


random_forest_model.fit(
    X_train_new,
    y_train_cat3_log
)


y_validation_pred_new = (
    random_forest_model.predict(
        X_validation_new
    )
)


r2_new = r2_score(
    y_validation_cat3_log,
    y_validation_pred_new
)


baseline_r2 = 0.591228
r2_change = r2_new - baseline_r2


print("=" * 70)
print("credit_value_per_size Feature Evaluation")
print("=" * 70)

print(f"\nBaseline R2: {baseline_r2:.6f}")
print(f"New R2:      {r2_new:.6f}")
print(f"Change:      {r2_change:+.6f}")


NameError: name 'rent_target_data' is not defined

## 23. مهندسی ویژگی نهایی

آزمایش ویژگی‌های مشتق‌شده مانند نسبت credit به متراژ و آمارهای Neighborhood/City. آمارهای گروهی به‌صورت Train-only ساخته می‌شوند تا Data Leakage ایجاد نشود.


In [99]:
# Add the new feature to the current 12-feature set
features_with_credit_per_size = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size"
]


X_new = rent_target_data[
    features_with_credit_per_size
].copy()


# Use the same train/validation split
X_train_new = X_new.loc[X_train.index]
X_validation_new = X_new.loc[X_validation.index]


# Encode categorical features
categorical_features_new = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_heating_system",
    "cat3_slug"
]

numeric_features_new = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size"
]


from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack


encoder_new = OneHotEncoder(
    handle_unknown="ignore"
)

X_train_cat_new = encoder_new.fit_transform(
    X_train_new[categorical_features_new]
)

X_validation_cat_new = encoder_new.transform(
    X_validation_new[categorical_features_new]
)


X_train_num_new = X_train_new[
    numeric_features_new
].values

X_validation_num_new = X_validation_new[
    numeric_features_new
].values


X_train_new_final = hstack([
    X_train_num_new,
    X_train_cat_new
])

X_validation_new_final = hstack([
    X_validation_num_new,
    X_validation_cat_new
])


# XGBoost with current best parameters
xgb_model_new = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


xgb_model_new.fit(
    X_train_new_final,
    y_train_cat3_log
)


y_validation_pred_new = (
    xgb_model_new.predict(
        X_validation_new_final
    )
)


r2_new = r2_score(
    y_validation_cat3_log,
    y_validation_pred_new
)


baseline_r2 = 0.591228
r2_change = r2_new - baseline_r2


print("=" * 70)
print("credit_value_per_size Feature Evaluation")
print("=" * 70)

print(f"\nBaseline R2: {baseline_r2:.6f}")
print(f"New R2:      {r2_new:.6f}")
print(f"Change:      {r2_change:+.6f}")


NameError: name 'rent_target_data' is not defined

In [98]:
rent_target_data["credit_value_per_room"] = (
    rent_target_data["credit_value"] /
    rent_target_data["rooms_count_numeric"].replace(0, np.nan)
)


# Check the result
print("=" * 70)
print("credit_value_per_room Check")
print("=" * 70)

print(
    "\nMissing:",
    rent_target_data["credit_value_per_room"].isna().sum()
)

print(
    "Infinite:",
    np.isinf(
        rent_target_data["credit_value_per_room"]
    ).sum()
)

print("\nDescriptive statistics:")

print(
    rent_target_data["credit_value_per_room"].describe()
)


NameError: name 'rent_target_data' is not defined

In [97]:
# Analyze the relationship between building age and rent value

age_rent_analysis = (
    rent_target_data
    .groupby("building_age")["rent_value"]
    .agg(
        count="count",
        median="median",
        mean="mean"
    )
    .reset_index()
    .sort_values("building_age")
)


# Add log of rent for a more stable comparison
rent_target_data["rent_value_log_temp"] = np.log1p(
    rent_target_data["rent_value"]
)

age_rent_log_analysis = (
    rent_target_data
    .groupby("building_age")["rent_value_log_temp"]
    .agg(
        median_log_rent="median",
        mean_log_rent="mean"
    )
    .reset_index()
    .sort_values("building_age")
)


# Merge the results
age_rent_analysis = age_rent_analysis.merge(
    age_rent_log_analysis,
    on="building_age",
    how="left"
)


# Save results
) as file:

    )


# Show results
print("=" * 70)
print("Building Age vs Rent Value")
print("=" * 70)

print(age_rent_analysis.to_string(index=False))


NameError: name 'rent_target_data' is not defined

In [96]:
neighborhood_frequency = (
    rent_target_data["neighborhood_slug"]
    .value_counts()
)


# Create frequency feature
rent_target_data["neighborhood_frequency"] = (
    rent_target_data["neighborhood_slug"]
    .map(neighborhood_frequency)
)


# Check the result
print("=" * 70)
print("neighborhood_frequency Check")
print("=" * 70)

print(
    "\nMissing:",
    rent_target_data["neighborhood_frequency"].isna().sum()
)

print(
    "\nDescriptive statistics:"
)

print(
    rent_target_data["neighborhood_frequency"].describe()
)

print(
    "\nFirst 10 values:"
)

print(
    rent_target_data[
        [
            "neighborhood_slug",
            "neighborhood_frequency"
        ]
    ].head(10)
)


NameError: name 'rent_target_data' is not defined

In [95]:
# Get neighborhood_slug from train and validation rows
train_neighborhood = rent_target_data.loc[
    X_train.index,
    "neighborhood_slug"
]

validation_neighborhood = rent_target_data.loc[
    X_validation.index,
    "neighborhood_slug"
]


# Calculate frequency using TRAINING data only
neighborhood_frequency_train = (
    train_neighborhood.value_counts()
)


# Create train dataframe
X_train_new = rent_target_data.loc[
    X_train.index
].copy()

X_train_new["neighborhood_frequency"] = (
    train_neighborhood.map(
        neighborhood_frequency_train
    )
)


# Create validation dataframe
X_validation_new = rent_target_data.loc[
    X_validation.index
].copy()

X_validation_new["neighborhood_frequency"] = (
    validation_neighborhood.map(
        neighborhood_frequency_train
    )
)


# Check the result
print("=" * 70)
print("Neighborhood Frequency - Leakage Safe Check")
print("=" * 70)

print(
    "\nTrain missing:",
    X_train_new["neighborhood_frequency"].isna().sum()
)

print(
    "Validation missing:",
    X_validation_new["neighborhood_frequency"].isna().sum()
)


print("\nTrain statistics:")
print(
    X_train_new["neighborhood_frequency"].describe()
)


print("\nValidation statistics:")
print(
    X_validation_new["neighborhood_frequency"].describe()
)


NameError: name 'rent_target_data' is not defined

In [94]:
# Current features + neighborhood frequency
features_with_neighborhood_frequency = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size",
    "neighborhood_frequency"
]


X_new = rent_target_data[
    features_with_neighborhood_frequency
].copy()


# Same train / validation split
X_train_new = X_new.loc[X_train.index]
X_validation_new = X_new.loc[X_validation.index]


# Feature groups
categorical_features_new = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_heating_system",
    "cat3_slug"
]

numeric_features_new = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency"
]


# One-hot encoding
encoder_new = OneHotEncoder(
    handle_unknown="ignore"
)

X_train_cat_new = encoder_new.fit_transform(
    X_train_new[categorical_features_new]
)

X_validation_cat_new = encoder_new.transform(
    X_validation_new[categorical_features_new]
)


# Numeric features
X_train_num_new = X_train_new[
    numeric_features_new
].values

X_validation_num_new = X_validation_new[
    numeric_features_new
].values


# Combine numeric + categorical
X_train_new_final = hstack([
    X_train_num_new,
    X_train_cat_new
])

X_validation_new_final = hstack([
    X_validation_num_new,
    X_validation_cat_new
])


# Current best XGBoost parameters
xgb_model_new = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


xgb_model_new.fit(
    X_train_new_final,
    y_train_cat3_log
)


# Validation prediction
y_validation_pred_new = (
    xgb_model_new.predict(
        X_validation_new_final
    )
)


# Validation R2
r2_new = r2_score(
    y_validation_cat3_log,
    y_validation_pred_new
)


baseline_r2 = 0.594831
r2_change = r2_new - baseline_r2


print("=" * 70)
print("neighborhood_frequency Feature Evaluation")
print("=" * 70)

print(f"\nBaseline R2: {baseline_r2:.6f}")
print(f"New R2:      {r2_new:.6f}")
print(f"Change:      {r2_change:+.6f}")


NameError: name 'rent_target_data' is not defined

In [93]:
# Get city_slug from train and validation rows
train_city = rent_target_data.loc[
    X_train.index,
    "city_slug"
]

validation_city = rent_target_data.loc[
    X_validation.index,
    "city_slug"
]


# Calculate city frequency using TRAINING data only
city_frequency_train = (
    train_city.value_counts()
)


# Create train dataframe
X_train_city_freq = rent_target_data.loc[
    X_train.index
].copy()

X_train_city_freq["city_frequency"] = (
    train_city.map(
        city_frequency_train
    )
)


# Create validation dataframe
X_validation_city_freq = rent_target_data.loc[
    X_validation.index
].copy()

X_validation_city_freq["city_frequency"] = (
    validation_city.map(
        city_frequency_train
    )
)


# Check the result
print("=" * 70)
print("City Frequency - Leakage Safe Check")
print("=" * 70)

print(
    "\nTrain missing:",
    X_train_city_freq["city_frequency"].isna().sum()
)

print(
    "Validation missing:",
    X_validation_city_freq["city_frequency"].isna().sum()
)


print("\nTrain statistics:")
print(
    X_train_city_freq["city_frequency"].describe()
)


print("\nValidation statistics:")
print(
    X_validation_city_freq["city_frequency"].describe()
)


NameError: name 'rent_target_data' is not defined

In [92]:
# Get city and neighborhood from train and validation rows
train_city = rent_target_data.loc[
    X_train.index,
    "city_slug"
]

train_neighborhood = rent_target_data.loc[
    X_train.index,
    "neighborhood_slug"
]

validation_city = rent_target_data.loc[
    X_validation.index,
    "city_slug"
]

validation_neighborhood = rent_target_data.loc[
    X_validation.index,
    "neighborhood_slug"
]


# Combine city and neighborhood in the training data
train_location = pd.DataFrame({
    "city_slug": train_city,
    "neighborhood_slug": train_neighborhood
})


# Count all listings in each city
city_frequency_train = (
    train_location["city_slug"]
    .value_counts()
)


# Count listings in each neighborhood within each city
neighborhood_frequency_in_city_train = (
    train_location
    .groupby(
        ["city_slug", "neighborhood_slug"],
        dropna=False
    )
    .size()
)


# Calculate neighborhood share within its city
neighborhood_share_in_city_train = (
    neighborhood_frequency_in_city_train
    / train_location["city_slug"].value_counts()
)


# Create keys for train and validation
train_keys = list(
    zip(
        train_city,
        train_neighborhood
    )
)

validation_keys = list(
    zip(
        validation_city,
        validation_neighborhood
    )
)


# Map the training statistics to train and validation
X_train_share = pd.Series(
    train_keys,
    index=X_train.index
).map(
    neighborhood_share_in_city_train
)

X_validation_share = pd.Series(
    validation_keys,
    index=X_validation.index
).map(
    neighborhood_share_in_city_train
)


# Save the feature into rent_target_data
rent_target_data["neighborhood_share_in_city"] = np.nan

rent_target_data.loc[
    X_train.index,
    "neighborhood_share_in_city"
] = X_train_share

rent_target_data.loc[
    X_validation.index,
    "neighborhood_share_in_city"
] = X_validation_share


# Check the result
print("=" * 70)
print("Neighborhood Share in City - Leakage Safe Check")
print("=" * 70)

print(
    "\nTrain missing:",
    X_train_share.isna().sum()
)

print(
    "Validation missing:",
    X_validation_share.isna().sum()
)

print("\nTrain statistics:")
print(
    X_train_share.describe()
)

print("\nValidation statistics:")
print(
    X_validation_share.describe()
)


NameError: name 'rent_target_data' is not defined

In [91]:
# Current feature set + new engineered feature
features_with_neighborhood_share = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city"
]

numeric_features_share = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city"
]

categorical_features_share = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]


# Prepare train and validation data
X_train_share = rent_target_data.loc[
    X_train.index,
    features_with_neighborhood_share
].copy()

X_validation_share = rent_target_data.loc[
    X_validation.index,
    features_with_neighborhood_share
].copy()


# Fill the 4 unseen validation combinations
# with 0 because no train observation exists for them
X_train_share["neighborhood_share_in_city"] = (
    X_train_share["neighborhood_share_in_city"]
    .fillna(0)
)

X_validation_share["neighborhood_share_in_city"] = (
    X_validation_share["neighborhood_share_in_city"]
    .fillna(0)
)


# Encode categorical features using train only
encoder_share = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_share.fit(
    X_train_share[categorical_features_share]
)

X_train_cat_share = encoder_share.transform(
    X_train_share[categorical_features_share]
)

X_validation_cat_share = encoder_share.transform(
    X_validation_share[categorical_features_share]
)


# Numeric matrices
X_train_num_share = csr_matrix(
    X_train_share[numeric_features_share].values
)

X_validation_num_share = csr_matrix(
    X_validation_share[numeric_features_share].values
)


# Final matrices
X_train_final_share = hstack([
    X_train_num_share,
    X_train_cat_share
]).tocsr()

X_validation_final_share = hstack([
    X_validation_num_share,
    X_validation_cat_share
]).tocsr()


# XGBoost with current best parameters
xgb_neighborhood_share = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# Train on log-transformed target
xgb_neighborhood_share.fit(
    X_train_final_share,
    y_train_cat3_log
)


# Validation prediction
y_validation_pred_share_log = (
    xgb_neighborhood_share.predict(
        X_validation_final_share
    )
)


# Validation R2 on log target
r2_neighborhood_share = r2_score(
    y_validation_cat3_log,
    y_validation_pred_share_log
)


print("=" * 70)
print("XGBoost - Neighborhood Share in City")
print("=" * 70)

print(
    f"\nBaseline R2: 0.596777"
)

print(
    f"New R2: {r2_neighborhood_share:.6f}"
)

print(
    f"Change: "
    f"{r2_neighborhood_share - 0.596777:+.6f}"
)


NameError: name 'rent_target_data' is not defined

In [90]:
# Get train and validation data
train_building_size = rent_target_data.loc[
    X_train.index,
    "building_size"
]

train_neighborhood = rent_target_data.loc[
    X_train.index,
    "neighborhood_slug"
]

validation_building_size = rent_target_data.loc[
    X_validation.index,
    "building_size"
]

validation_neighborhood = rent_target_data.loc[
    X_validation.index,
    "neighborhood_slug"
]


# Create training dataframe
train_neighborhood_size = pd.DataFrame({
    "neighborhood_slug": train_neighborhood,
    "building_size": train_building_size
})


# Calculate median building size for each neighborhood
neighborhood_size_median_train = (
    train_neighborhood_size
    .groupby(
        "neighborhood_slug",
        dropna=False
    )["building_size"]
    .median()
)


# Map train-derived median to train and validation
train_neighborhood_median = train_neighborhood.map(
    neighborhood_size_median_train
)

validation_neighborhood_median = validation_neighborhood.map(
    neighborhood_size_median_train
)


# Calculate relative building size
building_size_neighborhood_ratio_train = (
    train_building_size /
    train_neighborhood_median.replace(0, np.nan)
)

building_size_neighborhood_ratio_validation = (
    validation_building_size /
    validation_neighborhood_median.replace(0, np.nan)
)


# Save feature
rent_target_data["building_size_neighborhood_ratio"] = np.nan

rent_target_data.loc[
    X_train.index,
    "building_size_neighborhood_ratio"
] = (
    building_size_neighborhood_ratio_train
)

rent_target_data.loc[
    X_validation.index,
    "building_size_neighborhood_ratio"
] = (
    building_size_neighborhood_ratio_validation
)


# Check result
print("=" * 70)
print("Building Size / Neighborhood Median - Leakage Safe Check")
print("=" * 70)

print(
    "\nTrain missing:",
    building_size_neighborhood_ratio_train.isna().sum()
)

print(
    "Validation missing:",
    building_size_neighborhood_ratio_validation.isna().sum()
)

print("\nTrain statistics:")
print(
    building_size_neighborhood_ratio_train.describe()
)

print("\nValidation statistics:")
print(
    building_size_neighborhood_ratio_validation.describe()
)


ratio_train = (
    rent_target_data.loc[
        X_train.index,
        "building_size_neighborhood_ratio"
    ]
)

ratio_validation = (
    rent_target_data.loc[
        X_validation.index,
        "building_size_neighborhood_ratio"
    ]
)

print("=" * 70)
print("Building Size / Neighborhood Median - Extreme Values Check")
print("=" * 70)

for threshold in [2, 3, 5, 10, 50, 100, 500, 1000]:

    train_count = (
        ratio_train > threshold
    ).sum()

    validation_count = (
        ratio_validation > threshold
    ).sum()

    print(
        f"\nRatio > {threshold}:"
    )

    print(
        f"Train: {train_count:,} "
        f"({train_count / len(ratio_train) * 100:.4f}%)"
    )

    print(
        f"Validation: {validation_count:,} "
        f"({validation_count / len(ratio_validation) * 100:.4f}%)"
    )


print("\nTop 20 Train values:")
print(
    ratio_train
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)


NameError: name 'rent_target_data' is not defined

In [89]:
thresholds = [
    2,
    3,
    5,
    10,
    50,
    100,
    500,
    1000
]

output = []

output.append("=" * 70)
output.append(
    "Building Size / Neighborhood Median - Extreme Values"
)
output.append("=" * 70)

for threshold in thresholds:

    train_count = (
        ratio_train > threshold
    ).sum()

    validation_count = (
        ratio_validation > threshold
    ).sum()

    train_percentage = (
        train_count /
        len(ratio_train) *
        100
    )

    validation_percentage = (
        validation_count /
        len(ratio_validation) *
        100
    )

    output.append(
        f"\nRatio > {threshold}"
    )

    output.append(
        f"Train: {train_count:,} "
        f"({train_percentage:.4f}%)"
    )

    output.append(
        f"Validation: {validation_count:,} "
        f"({validation_percentage:.4f}%)"
    )


output.append("\n" + "=" * 70)
output.append("Selected percentiles")
output.append("=" * 70)

percentiles = [
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
]

train_percentiles = ratio_train.quantile(
    percentiles
)

validation_percentiles = ratio_validation.quantile(
    percentiles
)

for percentile in percentiles:

    output.append(
        f"\n{percentile * 100:.1f}th percentile:"
    )

    output.append(
        f"Train: {train_percentiles[percentile]:,.4f}"
    )

    output.append(
        f"Validation: "
        f"{validation_percentiles[percentile]:,.4f}"
    )


result = "\n".join(output)

print(result)

) as file:



NameError: name 'ratio_train' is not defined

In [88]:
features_with_neighborhood_size_ratio = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio"
]


numeric_features_ratio = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio"
]


categorical_features_ratio = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]


# Train / validation data
X_train_ratio = rent_target_data.loc[
    X_train.index,
    features_with_neighborhood_size_ratio
].copy()

X_validation_ratio = rent_target_data.loc[
    X_validation.index,
    features_with_neighborhood_size_ratio
].copy()


# Handle the two unseen validation values
X_train_ratio["building_size_neighborhood_ratio"] = (
    X_train_ratio["building_size_neighborhood_ratio"]
    .fillna(0)
)

X_validation_ratio["building_size_neighborhood_ratio"] = (
    X_validation_ratio["building_size_neighborhood_ratio"]
    .fillna(0)
)


# One-hot encoding
encoder_ratio = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_ratio.fit(
    X_train_ratio[categorical_features_ratio]
)

X_train_cat_ratio = encoder_ratio.transform(
    X_train_ratio[categorical_features_ratio]
)

X_validation_cat_ratio = encoder_ratio.transform(
    X_validation_ratio[categorical_features_ratio]
)


# Numeric matrices
X_train_num_ratio = csr_matrix(
    X_train_ratio[numeric_features_ratio].values
)

X_validation_num_ratio = csr_matrix(
    X_validation_ratio[numeric_features_ratio].values
)


# Final matrices
X_train_final_ratio = hstack([
    X_train_num_ratio,
    X_train_cat_ratio
]).tocsr()

X_validation_final_ratio = hstack([
    X_validation_num_ratio,
    X_validation_cat_ratio
]).tocsr()


# Current best XGBoost parameters
xgb_neighborhood_size_ratio = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# Train
xgb_neighborhood_size_ratio.fit(
    X_train_final_ratio,
    y_train_cat3_log
)


# Validation prediction
y_validation_pred_ratio_log = (
    xgb_neighborhood_size_ratio.predict(
        X_validation_final_ratio
    )
)


# Validation R2
r2_neighborhood_size_ratio = r2_score(
    y_validation_cat3_log,
    y_validation_pred_ratio_log
)


print("=" * 70)
print(
    "XGBoost - Building Size / Neighborhood Median"
)
print("=" * 70)

print(
    "\nBaseline R2: 0.605778"
)

print(
    f"New R2: {r2_neighborhood_size_ratio:.6f}"
)

print(
    f"Change: "
    f"{r2_neighborhood_size_ratio - 0.605778:+.6f}"
)


NameError: name 'rent_target_data' is not defined

In [87]:
# Get train and validation data
train_credit_value = rent_target_data.loc[
    X_train.index,
    "credit_value"
]

train_neighborhood = rent_target_data.loc[
    X_train.index,
    "neighborhood_slug"
]

validation_credit_value = rent_target_data.loc[
    X_validation.index,
    "credit_value"
]

validation_neighborhood = rent_target_data.loc[
    X_validation.index,
    "neighborhood_slug"
]


# Create training dataframe
train_neighborhood_credit = pd.DataFrame({
    "neighborhood_slug": train_neighborhood,
    "credit_value": train_credit_value
})


# Calculate median credit value for each neighborhood
neighborhood_credit_median_train = (
    train_neighborhood_credit
    .groupby(
        "neighborhood_slug",
        dropna=False
    )["credit_value"]
    .median()
)


# Map train-derived median to train and validation
train_neighborhood_credit_median = (
    train_neighborhood.map(
        neighborhood_credit_median_train
    )
)

validation_neighborhood_credit_median = (
    validation_neighborhood.map(
        neighborhood_credit_median_train
    )
)


# Calculate relative credit value
credit_value_neighborhood_median_ratio_train = (
    train_credit_value /
    train_neighborhood_credit_median.replace(0, np.nan)
)

credit_value_neighborhood_median_ratio_validation = (
    validation_credit_value /
    validation_neighborhood_credit_median.replace(0, np.nan)
)


# Save feature
rent_target_data[
    "credit_value_neighborhood_median_ratio"
] = np.nan

rent_target_data.loc[
    X_train.index,
    "credit_value_neighborhood_median_ratio"
] = (
    credit_value_neighborhood_median_ratio_train
)

rent_target_data.loc[
    X_validation.index,
    "credit_value_neighborhood_median_ratio"
] = (
    credit_value_neighborhood_median_ratio_validation
)


# Check result
print("=" * 70)
print(
    "Credit Value / Neighborhood Median - Leakage Safe Check"
)
print("=" * 70)

print(
    "\nTrain missing:",
    credit_value_neighborhood_median_ratio_train.isna().sum()
)

print(
    "Validation missing:",
    credit_value_neighborhood_median_ratio_validation.isna().sum()
)

print("\nTrain statistics:")
print(
    credit_value_neighborhood_median_ratio_train.describe()
)

print("\nValidation statistics:")
print(
    credit_value_neighborhood_median_ratio_validation.describe()
)


NameError: name 'rent_target_data' is not defined

In [86]:
features_with_credit_neighborhood_ratio = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio",
    "credit_value_neighborhood_median_ratio"
]


numeric_features_credit_ratio = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio",
    "credit_value_neighborhood_median_ratio"
]


categorical_features_credit_ratio = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]


# Train / validation data
X_train_credit_ratio = rent_target_data.loc[
    X_train.index,
    features_with_credit_neighborhood_ratio
].copy()

X_validation_credit_ratio = rent_target_data.loc[
    X_validation.index,
    features_with_credit_neighborhood_ratio
].copy()


# Fill the two missing values in each split
X_train_credit_ratio[
    "credit_value_neighborhood_median_ratio"
] = (
    X_train_credit_ratio[
        "credit_value_neighborhood_median_ratio"
    ]
    .fillna(0)
)

X_validation_credit_ratio[
    "credit_value_neighborhood_median_ratio"
] = (
    X_validation_credit_ratio[
        "credit_value_neighborhood_median_ratio"
    ]
    .fillna(0)
)


# One-hot encoding
encoder_credit_ratio = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_credit_ratio.fit(
    X_train_credit_ratio[
        categorical_features_credit_ratio
    ]
)

X_train_cat_credit_ratio = (
    encoder_credit_ratio.transform(
        X_train_credit_ratio[
            categorical_features_credit_ratio
        ]
    )
)

X_validation_cat_credit_ratio = (
    encoder_credit_ratio.transform(
        X_validation_credit_ratio[
            categorical_features_credit_ratio
        ]
    )
)


# Numeric matrices
X_train_num_credit_ratio = csr_matrix(
    X_train_credit_ratio[
        numeric_features_credit_ratio
    ].values
)

X_validation_num_credit_ratio = csr_matrix(
    X_validation_credit_ratio[
        numeric_features_credit_ratio
    ].values
)


# Final matrices
X_train_final_credit_ratio = hstack([
    X_train_num_credit_ratio,
    X_train_cat_credit_ratio
]).tocsr()

X_validation_final_credit_ratio = hstack([
    X_validation_num_credit_ratio,
    X_validation_cat_credit_ratio
]).tocsr()


# Current best XGBoost parameters
xgb_credit_neighborhood_ratio = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# Train
xgb_credit_neighborhood_ratio.fit(
    X_train_final_credit_ratio,
    y_train_cat3_log
)


# Validation prediction
y_validation_pred_credit_ratio_log = (
    xgb_credit_neighborhood_ratio.predict(
        X_validation_final_credit_ratio
    )
)


# Validation R2
r2_credit_neighborhood_ratio = r2_score(
    y_validation_cat3_log,
    y_validation_pred_credit_ratio_log
)


print("=" * 70)
print(
    "XGBoost - Credit Value / Neighborhood Median"
)
print("=" * 70)

print(
    "\nBaseline R2: 0.606679"
)

print(
    f"New R2: {r2_credit_neighborhood_ratio:.6f}"
)

print(
    f"Change: "
    f"{r2_credit_neighborhood_ratio - 0.606679:+.6f}"
)


NameError: name 'rent_target_data' is not defined

In [85]:
# Get train and validation data
train_credit_value = rent_target_data.loc[
    X_train.index,
    "credit_value"
]

train_city = rent_target_data.loc[
    X_train.index,
    "city_slug"
]

validation_credit_value = rent_target_data.loc[
    X_validation.index,
    "credit_value"
]

validation_city = rent_target_data.loc[
    X_validation.index,
    "city_slug"
]


# Create training dataframe
train_city_credit = pd.DataFrame({
    "city_slug": train_city,
    "credit_value": train_credit_value
})


# Calculate median credit value for each city
city_credit_median_train = (
    train_city_credit
    .groupby(
        "city_slug",
        dropna=False
    )["credit_value"]
    .median()
)


# Map train-derived median to train and validation
train_city_credit_median = train_city.map(
    city_credit_median_train
)

validation_city_credit_median = validation_city.map(
    city_credit_median_train
)


# Calculate relative credit value
credit_value_city_median_ratio_train = (
    train_credit_value /
    train_city_credit_median.replace(0, np.nan)
)

credit_value_city_median_ratio_validation = (
    validation_credit_value /
    validation_city_credit_median.replace(0, np.nan)
)


# Save feature
rent_target_data[
    "credit_value_city_median_ratio"
] = np.nan

rent_target_data.loc[
    X_train.index,
    "credit_value_city_median_ratio"
] = (
    credit_value_city_median_ratio_train
)

rent_target_data.loc[
    X_validation.index,
    "credit_value_city_median_ratio"
] = (
    credit_value_city_median_ratio_validation
)


# Check result
print("=" * 70)
print(
    "Credit Value / City Median - Leakage Safe Check"
)
print("=" * 70)

print(
    "\nTrain missing:",
    credit_value_city_median_ratio_train.isna().sum()
)

print(
    "Validation missing:",
    credit_value_city_median_ratio_validation.isna().sum()
)

print("\nTrain statistics:")
print(
    credit_value_city_median_ratio_train.describe()
)

print("\nValidation statistics:")
print(
    credit_value_city_median_ratio_validation.describe()
)


NameError: name 'rent_target_data' is not defined

In [84]:
features_with_city_credit_ratio = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio",
    "credit_value_neighborhood_median_ratio",
    "credit_value_city_median_ratio"
]


numeric_features_city_credit = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio",
    "credit_value_neighborhood_median_ratio",
    "credit_value_city_median_ratio"
]


categorical_features_city_credit = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]


# Train / validation data
X_train_city_credit = rent_target_data.loc[
    X_train.index,
    features_with_city_credit_ratio
].copy()

X_validation_city_credit = rent_target_data.loc[
    X_validation.index,
    features_with_city_credit_ratio
].copy()


# Fill the two unseen validation values
X_train_city_credit[
    "credit_value_city_median_ratio"
] = (
    X_train_city_credit[
        "credit_value_city_median_ratio"
    ]
    .fillna(0)
)

X_validation_city_credit[
    "credit_value_city_median_ratio"
] = (
    X_validation_city_credit[
        "credit_value_city_median_ratio"
    ]
    .fillna(0)
)


# One-hot encoding
encoder_city_credit = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

encoder_city_credit.fit(
    X_train_city_credit[
        categorical_features_city_credit
    ]
)

X_train_cat_city_credit = (
    encoder_city_credit.transform(
        X_train_city_credit[
            categorical_features_city_credit
        ]
    )
)

X_validation_cat_city_credit = (
    encoder_city_credit.transform(
        X_validation_city_credit[
            categorical_features_city_credit
        ]
    )
)


# Numeric matrices
X_train_num_city_credit = csr_matrix(
    X_train_city_credit[
        numeric_features_city_credit
    ].values
)

X_validation_num_city_credit = csr_matrix(
    X_validation_city_credit[
        numeric_features_city_credit
    ].values
)


# Final matrices
X_train_final_city_credit = hstack([
    X_train_num_city_credit,
    X_train_cat_city_credit
]).tocsr()

X_validation_final_city_credit = hstack([
    X_validation_num_city_credit,
    X_validation_cat_city_credit
]).tocsr()


# Current best XGBoost parameters
xgb_city_credit_ratio = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# Train
xgb_city_credit_ratio.fit(
    X_train_final_city_credit,
    y_train_cat3_log
)


# Validation prediction
y_validation_pred_city_credit_log = (
    xgb_city_credit_ratio.predict(
        X_validation_final_city_credit
    )
)


# Validation R2
r2_city_credit_ratio = r2_score(
    y_validation_cat3_log,
    y_validation_pred_city_credit_log
)


print("=" * 70)
print(
    "XGBoost - Credit Value / City Median"
)
print("=" * 70)

print(
    "\nBaseline R2: 0.608819"
)

print(
    f"New R2: {r2_city_credit_ratio:.6f}"
)

print(
    f"Change: "
    f"{r2_city_credit_ratio - 0.608819:+.6f}"
)


NameError: name 'rent_target_data' is not defined

In [83]:

# ============================================================
# 1. Create test engineered features using TRAIN statistics
# ============================================================

test_data = rent_target_data.loc[
    X_test.index
].copy()


# ------------------------------------------------------------
# Neighborhood frequency
# ------------------------------------------------------------

test_data["neighborhood_frequency"] = (
    test_data["neighborhood_slug"]
    .map(neighborhood_frequency_train)
)


# ------------------------------------------------------------
# Neighborhood share in city
# ------------------------------------------------------------

test_city = test_data["city_slug"]
test_neighborhood = test_data["neighborhood_slug"]

test_keys = list(
    zip(
        test_city,
        test_neighborhood
    )
)

test_data["neighborhood_share_in_city"] = (
    pd.Series(
        test_keys,
        index=test_data.index
    )
    .map(
        neighborhood_share_in_city_train
    )
)


# ------------------------------------------------------------
# Building size / neighborhood median
# ------------------------------------------------------------

test_neighborhood_median = (
    test_neighborhood.map(
        neighborhood_size_median_train
    )
)

test_data["building_size_neighborhood_ratio"] = (
    test_data["building_size"] /
    test_neighborhood_median.replace(0, np.nan)
)


# ------------------------------------------------------------
# Credit value / neighborhood median
# ------------------------------------------------------------

test_neighborhood_credit_median = (
    test_neighborhood.map(
        neighborhood_credit_median_train
    )
)

test_data["credit_value_neighborhood_median_ratio"] = (
    test_data["credit_value"] /
    test_neighborhood_credit_median.replace(0, np.nan)
)


# ------------------------------------------------------------
# Credit value / city median
# ------------------------------------------------------------

test_city_credit_median = (
    test_city.map(
        city_credit_median_train
    )
)

test_data["credit_value_city_median_ratio"] = (
    test_data["credit_value"] /
    test_city_credit_median.replace(0, np.nan)
)


# ------------------------------------------------------------
# Credit value / building size
# ------------------------------------------------------------

test_data["credit_value_per_size"] = (
    test_data["credit_value"] /
    test_data["building_size"].replace(0, np.nan)
)


# ============================================================
# 2. Define final feature set
# ============================================================

final_numeric_features = [
    "credit_value",
    "building_size",
    "building_age",
    "rooms_count_numeric",
    "credit_value_per_size",
    "neighborhood_frequency",
    "neighborhood_share_in_city",
    "building_size_neighborhood_ratio",
    "credit_value_neighborhood_median_ratio",
    "credit_value_city_median_ratio"
]

final_categorical_features = [
    "city_slug",
    "cat2_slug",
    "neighborhood_slug",
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "has_heating_system",
    "cat3_slug"
]


# ============================================================
# 3. Prepare train and test
# ============================================================

X_train_test_final = rent_target_data.loc[
    X_train.index,
    final_numeric_features + final_categorical_features
].copy()

X_test_final = test_data[
    final_numeric_features + final_categorical_features
].copy()


# ============================================================
# 4. Handle missing engineered values
# ============================================================

for col in final_numeric_features:

    X_train_test_final[col] = (
        X_train_test_final[col]
        .fillna(0)
    )

    X_test_final[col] = (
        X_test_final[col]
        .fillna(0)
    )


# ============================================================
# 5. One-hot encoding
#    Fit ONLY on train
# ============================================================

final_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

final_encoder.fit(
    X_train_test_final[
        final_categorical_features
    ]
)

X_train_cat_test = final_encoder.transform(
    X_train_test_final[
        final_categorical_features
    ]
)

X_test_cat_final = final_encoder.transform(
    X_test_final[
        final_categorical_features
    ]
)


# ============================================================
# 6. Numeric matrices
# ============================================================

X_train_num_test = csr_matrix(
    X_train_test_final[
        final_numeric_features
    ].values
)

X_test_num_final = csr_matrix(
    X_test_final[
        final_numeric_features
    ].values
)


# ============================================================
# 7. Final matrices
# ============================================================

X_train_final_test = hstack([
    X_train_num_test,
    X_train_cat_test
]).tocsr()

X_test_final = hstack([
    X_test_num_final,
    X_test_cat_final
]).tocsr()


# ============================================================
# 8. Log target
# ============================================================

y_train_final_log = np.log1p(
    y_train_n
)

y_test_final_log = np.log1p(
    y_test_n
)


# ============================================================
# 9. Final XGBoost model
# ============================================================

final_xgb_model = XGBRegressor(
    n_estimators=3000,
    max_depth=10,
    min_child_weight=5,
    learning_rate=0.05,
    gamma=0,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


final_xgb_model.fit(
    X_train_final_test,
    y_train_final_log
)


# ============================================================
# 10. Test prediction
# ============================================================

y_test_pred_log = (
    final_xgb_model.predict(
        X_test_final
    )
)


# ============================================================
# 11. Test metrics
# ============================================================

test_r2_log = r2_score(
    y_test_final_log,
    y_test_pred_log
)

test_mae_log = mean_absolute_error(
    y_test_final_log,
    y_test_pred_log
)

test_mse_log = mean_squared_error(
    y_test_final_log,
    y_test_pred_log
)

test_rmse_log = (
    test_mse_log ** 0.5
)


# ============================================================
# 12. Save final test results
# ============================================================

output = []

output.append(
    "=" * 70
)

output.append(
    "FINAL XGBOOST - TEST EVALUATION"
)

output.append(
    "=" * 70
)

output.append(
    f"\nTrain shape: "
    f"{X_train_final_test.shape}"
)

output.append(
    f"Test shape: "
    f"{X_test_final.shape}"
)

output.append(
    "\nFinal features:"
)

for feature in (
    final_numeric_features +
    final_categorical_features
):

    output.append(
        f"- {feature}"
    )

output.append(
    "\nFinal parameters:"
)

output.append(
    "n_estimators: 3000"
)

output.append(
    "max_depth: 10"
)

output.append(
    "min_child_weight: 5"
)

output.append(
    "learning_rate: 0.05"
)

output.append(
    "gamma: 0"
)

output.append(
    "subsample: 0.8"
)

output.append(
    "colsample_bytree: 0.8"
)

output.append(
    "\nTest metrics on log-transformed rent_value:"
)

output.append(
    f"R2: {test_r2_log:.6f}"
)

output.append(
    f"MAE: {test_mae_log:,.6f}"
)

output.append(
    f"MSE: {test_mse_log:,.6f}"
)

output.append(
    f"RMSE: {test_rmse_log:,.6f}"
)

result = "\n".join(output)

print(result)

) as file:



NameError: name 'rent_target_data' is not defined

In [82]:
# 1. Convert predictions back to original rent scale
# ============================================================

y_test_actual = y_test_n.copy()

y_test_pred_original = np.expm1(
    y_test_pred_log
)


# ============================================================
# 2. Create error dataframe
# ============================================================

test_error_analysis = pd.DataFrame({
    "actual_rent": y_test_actual.values,
    "predicted_rent": y_test_pred_original
})


test_error_analysis["absolute_error"] = (
    np.abs(
        test_error_analysis["actual_rent"]
        - test_error_analysis["predicted_rent"]
    )
)

test_error_analysis["percentage_error"] = (
    test_error_analysis["absolute_error"]
    /
    test_error_analysis["actual_rent"]
    * 100
)


# ============================================================
# 3. Define rent ranges
# ============================================================

rent_bins = [
    0,
    5_000_000,
    10_000_000,
    20_000_000,
    50_000_000,
    100_000_000,
    np.inf
]

rent_labels = [
    "0-5M",
    "5-10M",
    "10-20M",
    "20-50M",
    "50-100M",
    "100M+"
]


test_error_analysis["rent_range"] = pd.cut(
    test_error_analysis["actual_rent"],
    bins=rent_bins,
    labels=rent_labels,
    right=False
)


# ============================================================
# 4. Calculate error metrics for each rent range
# ============================================================

error_by_range = (
    test_error_analysis
    .groupby(
        "rent_range",
        observed=False
    )
    .agg(
        samples=("actual_rent", "size"),
        mean_actual_rent=("actual_rent", "mean"),
        mean_predicted_rent=("predicted_rent", "mean"),
        MAE=("absolute_error", "mean"),
        MAPE=("percentage_error", "mean")
    )
    .reset_index()
)


# ============================================================
# 5. Save results
# ============================================================

output = []

output.append("=" * 90)
output.append(
    "FINAL XGBOOST - ERROR ANALYSIS BY RENT RANGE"
)
output.append("=" * 90)

output.append(
    "\n"
    + error_by_range.to_string(
        index=False
    )
)

result = "\n".join(output)

print(result)

) as file:



NameError: name 'y_test_n' is not defined

## 24. ارزیابی نهایی روی Test

ساخت ویژگی‌های Test با استفاده از آمارهای استخراج‌شده فقط از Train و اجرای مدل نهایی.


In [81]:

high_rent_analysis = test_error_analysis[
    test_error_analysis["actual_rent"] >= 100_000_000
].copy()

high_rent_analysis["prediction_ratio"] = (
    high_rent_analysis["predicted_rent"] /
    high_rent_analysis["actual_rent"]
)

high_rent_analysis["underprediction"] = (
    high_rent_analysis["actual_rent"] -
    high_rent_analysis["predicted_rent"]
)

output = []

output.append("=" * 80)
output.append("HIGH RENT ERROR ANALYSIS - 100M+")
output.append("=" * 80)

output.append(
    f"\nNumber of samples: {len(high_rent_analysis):,}"
)

output.append("\n\nActual rent percentiles:")

output.append(
    high_rent_analysis["actual_rent"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_string()
)

output.append("\n\nPredicted rent percentiles:")

output.append(
    high_rent_analysis["predicted_rent"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_string()
)

output.append("\n\nPrediction ratio (Predicted / Actual):")

output.append(
    high_rent_analysis["prediction_ratio"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_string()
)

output.append("\n\nTop 20 largest underpredictions:")

top_underpredictions = (
    high_rent_analysis[
        [
            "actual_rent",
            "predicted_rent",
            "underprediction",
            "prediction_ratio"
        ]
    ]
    .sort_values(
        "underprediction",
        ascending=False
    )
    .head(20)
)

output.append(
    top_underpredictions.to_string(
        index=False
    )
)

result = "\n".join(output)

print(result)

) as file:


NameError: name 'test_error_analysis' is not defined

## 25. تحلیل خطای مدل نهایی

بررسی خطا در بازه‌های مختلف اجاره و بررسی رفتار مدل روی اجاره‌های بسیار بالا.


In [80]:
top_20_high_rent = (
    rent_target_data[
        [
            "rent_value",
            "credit_value",
            "building_size",
            "building_age",
            "rooms_count_numeric",
            "city_slug",
            "cat2_slug",
            "cat3_slug",
            "neighborhood_slug"
        ]
    ]
    .sort_values(
        "rent_value",
        ascending=False
    )
    .head(20)
)

output = []

output.append("=" * 120)
output.append("TOP 20 HIGHEST RENT VALUES")
output.append("=" * 120)

output.append(
    top_20_high_rent.to_string(
        index=False
    )
)

result = "\n".join(output)

print(result)

) as file:


NameError: name 'rent_target_data' is not defined

In [79]:

suspicious_values = [
    11_111_111_111,
    12_000_000_000,
    12_500_000_000,
    12_900_000_000,
    13_000_000_000
]

output = []

output.append("=" * 90)
output.append("SUSPICIOUS RENT VALUES FREQUENCY")
output.append("=" * 90)

total_records = len(rent_target_data)

output.append(
    f"\nTotal positive-rent records: {total_records:,}"
)

output.append("\n\nSpecific suspicious values:")

for value in suspicious_values:

    count = (
        rent_target_data["rent_value"]
        == value
    ).sum()

    percentage = (
        count / total_records * 100
    )

    output.append(
        f"\nValue: {value:,.0f}"
    )

    output.append(
        f"Count: {count:,}"
    )

    output.append(
        f"Percentage: {percentage:.4f}%"
    )


output.append("\n\nMost frequent values among rent_value:")

value_frequency = (
    rent_target_data["rent_value"]
    .value_counts()
    .head(30)
)

output.append(
    value_frequency.to_string()
)

result = "\n".join(output)

print(result)

) as file:


NameError: name 'rent_target_data' is not defined

## 26. مقایسه نهایی مدل‌ها

مقایسه Validation R² مدل‌ها در مراحل مختلف توسعه پروژه. ارزیابی Test فقط برای مدل نهایی انجام شده و برای انتخاب مدل استفاده نشده است.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

model_comparison = pd.DataFrame({
    "model": [
        "Decision Tree",
        "Random Forest",
        "XGBoost Baseline",
        "XGBoost Tuned",
        "XGBoost + Feature Engineering"
    ],
    "validation_r2": [
        0.423577,
        0.246030,
        0.498487,
        0.557524,
        0.611831
    ]
})

# مرتب‌سازی از کمترین به بیشترین R²
model_comparison = (
    model_comparison
    .sort_values(
        "validation_r2",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

plt.bar(
    model_comparison["model"],
    model_comparison["validation_r2"]
)

plt.title(
    "Comparison of Models Based on Validation R²"
)

plt.xlabel("Model")
plt.ylabel("Validation R²")

plt.xticks(
    rotation=20,
    ha="right"
)

plt.ylim(0, 0.7)

plt.tight_layout()

plt.show()


هدف این پروژه، ساخت یک مدل یادگیری ماشین برای پیش‌بینی مبلغ اجاره‌بهای املاک بر اساس ویژگی‌های مربوط به ملک، موقعیت جغرافیایی و برخی اطلاعات مالی و دسته‌ای موجود در داده‌ها است.
متغیر هدف در این پروژه rent_value در نظر گرفته شد. از آنجا که بخشی از رکوردها دارای مقدار صفر برای این متغیر بودند و مقدار صفر در این داده‌ها نشان‌دهنده‌ی «توافقی بودن مبلغ اجاره» بود، این رکوردها برای آموزش مدل مناسب تشخیص داده نشدند و از مجموعه‌ی هدف حذف شدند.
پس از حذف رکوردهای دارای اجاره صفر، تعداد داده‌های مورد استفاده برای مدل‌سازی برابر با 289,617 رکورد شد.
 
2. آماده‌سازی داده‌ها
در مرحله‌ی آماده‌سازی داده‌ها، ابتدا داده‌های مربوط به اجاره‌های مسکونی و تجاری از مجموعه‌ی اصلی استخراج شدند.
برخی از ویژگی‌ها نیاز به تبدیل یا پاک‌سازی داشتند. برای مثال، ویژگی مربوط به سال ساخت به یک ویژگی عددی به نام building_age تبدیل شد تا سن ساختمان به‌صورت عددی در اختیار مدل قرار گیرد.
همچنین ویژگی rooms_count که به‌صورت دسته‌ای و متنی ذخیره شده بود، به متغیر عددی rooms_count_numeric تبدیل شد.
برای برخی ویژگی‌های دودویی، مقادیر گمشده با مقدار False جایگزین شدند. در ویژگی‌های دسته‌ای نیز مقدار Unknown برای داده‌های گمشده استفاده شد.
در مورد building_size نیز مقادیر غیرمنطقی بر اساس نوع ملک بررسی شدند و تنها برخی مقادیر کاملاً نامتناسب با دسته‌ی مربوطه حذف شدند. مقادیر بزرگ اما قابل‌قبول، به‌دلیل احتمال نماینده بودن املاک بزرگ، صرفاً بر اساس روش IQR حذف نشدند.
 
3. بررسی متغیر هدف
متغیر rent_value دارای توزیع به‌شدت راست‌چوله بود. بخش عمده‌ای از اجاره‌ها در محدوده‌های پایین‌تر قرار داشتند، در حالی که تعداد محدودی از املاک دارای مبالغ بسیار بالا بودند.
برای کاهش اثر این چولگی و ایجاد شرایط مناسب‌تر برای مدل‌سازی، متغیر هدف به شکل زیر تبدیل شد:
log1p(rent_value)
بنابراین مدل‌ها در مرحله‌ی اصلی آموزش، مقدار لگاریتمی اجاره را پیش‌بینی کردند.
این تبدیل باعث شد اختلاف بسیار زیاد بین اجاره‌های معمولی و مقادیر بسیار بزرگ، تأثیر کمتری بر فرایند یادگیری داشته باشد.
 
4. انتخاب ویژگی‌ها
در مراحل اولیه، مجموعه‌ای از ویژگی‌های عددی و دسته‌ای مورد بررسی قرار گرفتند.
از جمله ویژگی‌های اصلی مورد استفاده می‌توان به موارد زیر اشاره کرد:
• 
credit_value
• 
building_size
• 
building_age
• 
rooms_count_numeric
• 
city_slug
• 
cat2_slug
• 
neighborhood_slug
• 
has_elevator
• 
has_parking
• 
has_warehouse
• 
has_heating_system
• 
cat3_slug
ویژگی‌هایی مانند has_cooling_system، city_type و luxury_score نیز بررسی شدند، اما در آزمایش‌های انجام‌شده بهبود معناداری در عملکرد مدل ایجاد نکردند و در مجموعه نهایی قرار نگرفتند.
 
5. استفاده از ویژگی‌های مهندسی‌شده
یکی از مهم‌ترین بخش‌های پروژه، مهندسی ویژگی بود.
در این مرحله ویژگی‌هایی ایجاد شدند که بتوانند رابطه‌ی بین متغیرهای عددی و موقعیت مکانی را بهتر به مدل منتقل کنند.
ویژگی‌های نهایی مهندسی‌شده عبارت‌اند از:
• 
credit_value_per_size
• 
neighborhood_frequency
• 
neighborhood_share_in_city
• 
building_size_neighborhood_ratio
• 
credit_value_neighborhood_median_ratio
• 
credit_value_city_median_ratio
برای مثال، credit_value_per_size نسبت ارزش ودیعه/اعتبار به متراژ ملک را نشان می‌دهد و می‌تواند اطلاعاتی درباره‌ی سطح ارزش مالی ملک نسبت به اندازه‌ی آن ارائه کند.
همچنین ویژگی‌هایی مانند نسبت متراژ ملک به میانه‌ی متراژ در محله، و نسبت credit_value به میانه‌ی آن در محله و شهر، به مدل کمک کردند تا اطلاعات نسبی مربوط به موقعیت ملک را بهتر درک کند.
در ویژگی‌های مبتنی بر فراوانی و میانه، محاسبات مربوط به داده‌های Train انجام شد و سپس همان نگاشت‌ها روی Validation و Test اعمال شدند. این روش برای جلوگیری از Data Leakage مورد استفاده قرار گرفت.
 
6. تقسیم داده‌ها
داده‌ها با random_state=42 به سه بخش تقسیم شدند:
• 
80 درصد برای Train
• 
10 درصد برای Validation
• 
10 درصد برای Test
در نتیجه:
• 
Train: 231,693 رکورد
• 
Validation: 28,962 رکورد
• 
Test: 28,962 رکورد
مجموعه Test تا مرحله‌ی نهایی کنار گذاشته شد و برای انتخاب ویژگی‌ها و تنظیم پارامترهای مدل مورد استفاده قرار نگرفت.
 
7. تبدیل ویژگی‌های دسته‌ای
ویژگی‌های دسته‌ای برای استفاده در مدل‌های یادگیری ماشین با روش One-Hot Encoding به ویژگی‌های عددی تبدیل شدند.
برای جلوگیری از ایجاد دسته‌هایی که فقط در Validation یا Test دیده می‌شوند، Encoder روی داده‌های Train آموزش داده شد و سپس روی Validation و Test اعمال شد.
همچنین گزینه‌ی handle_unknown="ignore" استفاده شد تا دسته‌هایی که در Train وجود نداشتند، باعث خطا در زمان پیش‌بینی نشوند.
 
8. مقایسه مدل‌ها
در طول پروژه چند مدل مختلف بررسی شدند.
نتایج Validation R² مدل‌های اصلی به شکل زیر بود:
مدل
Validation R²
Random Forest
0.246030
Decision Tree
0.423577
XGBoost Baseline
0.498487
XGBoost Tuned
0.557524
XGBoost + Feature Engineering
0.611831 
بر اساس این نتایج، مدل XGBoost پس از تنظیم پارامترها عملکرد بهتری نسبت به نسخه‌ی پایه نشان داد و اضافه کردن ویژگی‌های مهندسی‌شده باعث افزایش بیشتر عملکرد شد.
لازم به ذکر است که این جدول بیشتر برای نمایش روند توسعه مدل‌ها استفاده می‌شود و مقایسه‌ی کاملاً هم‌شرایط بین تمام مدل‌ها نیست، زیرا برخی مدل‌ها در مراحل مختلف پروژه با مجموعه ویژگی‌های متفاوت آزمایش شده‌اند.
 
9. تنظیم پارامترهای XGBoost
پس از انتخاب XGBoost به‌عنوان مدل اصلی، پارامترهای آن به‌صورت مرحله‌ای بررسی شدند.
پارامترهای مهم مورد بررسی شامل موارد زیر بودند:
• 
n_estimators
• 
max_depth
• 
learning_rate
• 
min_child_weight
• 
gamma
• 
subsample
• 
colsample_bytree
در نهایت بهترین ترکیب به دست آمده برای مدل نهایی عبارت بود از:
n_estimators = 3000
max_depth = 10
min_child_weight = 5
learning_rate = 0.05
gamma = 0
subsample = 0.8
colsample_bytree = 0.8
با این تنظیمات، مقدار Validation R² از حدود 0.4985 در مدل پایه به 0.5575 و پس از مهندسی ویژگی به 0.6118 رسید.
 
10. نتیجه نهایی روی داده Test
پس از نهایی شدن ویژگی‌ها و پارامترهای مدل، مدل نهایی تنها با استفاده از داده‌های Train آموزش داده شد و سپس روی داده‌های Test که قبلاً کنار گذاشته شده بود ارزیابی شد.
نتیجه نهایی:
Test R² = 0.601877
همچنین:
MAE  = 0.684981
MSE  = 1.239641
RMSE = 1.113392
این معیارهای خطا در این مرحله بر اساس مقدار log1p(rent_value) محاسبه شده‌اند و بنابراین نباید مستقیماً به‌عنوان خطای اجاره‌بها در واحد اصلی پول تفسیر شوند.
مقدار R² روی Test برابر حدود 0.602 است؛ یعنی مدل توانسته است حدود 60 درصد از واریانس متغیر هدف در مقیاس لگاریتمی را روی داده‌های دیده‌نشده توضیح دهد.
همچنین اختلاف Validation و Test برابر است با:
0.611831 - 0.601877 = 0.009954
این اختلاف نسبتاً محدود است و نشان می‌دهد عملکرد مدل روی داده Test به‌طور قابل‌توجهی از Validation فاصله نگرفته است.
 
11. تحلیل خطا
بررسی خطاها نشان داد که عملکرد مدل در تمام محدوده‌های اجاره یکسان نیست.
در املاک با اجاره‌های پایین‌تر، خطای مطلق مدل نسبتاً محدودتر است، در حالی که در گروه املاک با اجاره‌ی بیش از 100 میلیون، خطای مطلق بسیار بزرگ‌تر می‌شود.
دلیل اصلی این مسئله، وجود تعداد محدودی از مقادیر بسیار بزرگ در متغیر هدف است.
برای نمونه، در گروه اجاره‌های بالاتر از 100 میلیون، میانگین مقدار واقعی اجاره بسیار بیشتر از میانگین مقدار پیش‌بینی‌شده توسط مدل بود. بنابراین مدل در مقادیر بسیار بزرگ تمایل به کم‌برآورد کردن اجاره دارد.
این رفتار با توجه به توزیع بسیار چوله‌ی متغیر هدف قابل انتظار است.
از طرف دیگر، استفاده از MAPE برای اجاره‌های بسیار پایین معیار مناسبی نیست، زیرا وقتی مقدار واقعی به صفر نزدیک باشد، درصد خطا می‌تواند بسیار بزرگ شود.
 
12. اهمیت ویژگی‌ها
بررسی اهمیت ویژگی‌ها در مدل نشان داد که برخی متغیرها نقش بیشتری در پیش‌بینی اجاره دارند.
در آزمایش‌های انجام‌شده، credit_value و building_size از مهم‌ترین متغیرهای اصلی بودند.
همچنین ویژگی‌های مربوط به موقعیت مکانی، به‌خصوص city_slug و neighborhood_slug، نقش قابل‌توجهی در عملکرد مدل داشتند.
در بخش مهندسی ویژگی نیز ویژگی‌هایی که رابطه‌ی نسبی ملک با شرایط معمول محله یا شهر را بیان می‌کردند، باعث افزایش عملکرد مدل شدند.
مهم‌ترین افزایش حاصل از مهندسی ویژگی مربوط به neighborhood_share_in_city بود که به‌تنهایی حدود 0.009 به Validation R² اضافه کرد.
در مجموع، Validation R² از:
0.591228
به:
0.611831
رسید؛ یعنی مجموع ویژگی‌های مهندسی‌شده‌ی پذیرفته‌شده حدود 0.0206 افزایش در R² ایجاد کردند.
 

## 27. خلاصه نتایج نهایی
- بهترین Validation R² مدل نهایی XGBoost: **0.611831**
- R² نهایی روی Test: **0.601877**
- اختلاف Validation و Test: **0.009954**
- پارامترهای مدل نهایی: `n_estimators=3000`, `max_depth=10`, `min_child_weight=5`, `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.8`
- Target با `log1p(rent_value)` مدل‌سازی شده است.

> نکته: R²های مراحل مختلف الزاماً یک مقایسه کاملاً هم‌شرایط نیستند، چون برخی آزمایش‌های میانی با مجموعه ویژگی متفاوت انجام شده‌اند. نمودار مقایسه برای نمایش روند توسعه مدل‌هاست، نه یک Benchmark کاملاً کنترل‌شده.
